# MuJoCo prep

## Importers

In [ ]:
# ---- Core stdlib ----
import os, re, json, math, sys, time, logging
from collections import defaultdict, deque
from pathlib import Path

# ---- Third-party ----
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
import warnings

# NEURON
from neuron import h  # keep this single, canonical import

# neuprint (API) — single alias
import neuprint as npt
from neuprint import (
    Client as NPClient,
    NeuronCriteria,            # you can use the class directly
    fetch_neurons,
    fetch_simple_connections,
    fetch_synapses,
    fetch_synapse_connections,
)
from neuprint.skeleton import (
    fetch_skeleton, heal_skeleton, upsample_skeleton,
    reorient_skeleton, attach_synapses_to_skeleton, skeleton_segments
)

# navis + its neuprint interface — different alias so it never shadows neuprint
import navis
import navis.interfaces.neuprint as navnp  # use navnp.* when you really need the navis bridge

# Optional UI utils
from IPython.display import display, clear_output
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib
matplotlib.use("Agg")   # do this once, near the top of the notebook

# ---- Configure neuprint client (hardcoded token, no alias collisions) ----
NEUPRINT_HOST = "https://neuprint.janelia.org"
NEUPRINT_DATASET = "manc:v1.2.1"
NEUPRINT_TOKEN = (
    "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9."
    "eyJlbWFpbCI6ImpwYWJsbzE5OTUwOUBnbWFpbC5jb20iLCJsZXZlbCI6Im5vYXV0aCIs"
    "ImltYWdlLXVybCI6Imh0dHBzOi8vbGgzLmdvb2dsZXVzZXJjb250ZW50LmNvbS9hL0FD"
    "ZzhvY0pCM3JUMm9HV19va1hBV2VMcVZoSDhoUVRXM1FoNUNnXzVFRW00Tm9VaWE0bS01"
    "QT1zOTYtYz9zej01MD9zej01MCIsImV4cCI6MTkyMjMxMjQ0Nn0."
    "Hmou-N87hM-qp1rVZinuQTKEwDymqioIM5LzX3s5bPk"
)

# Primary neuprint client (official API)
client = NPClient(NEUPRINT_HOST, dataset=NEUPRINT_DATASET, token=NEUPRINT_TOKEN)
npt.set_default_client(client)  # <- keep this on the neuprint module alias

# Navis→neuprint bridge client (separate module/alias)
navis_client = navnp.Client(NEUPRINT_HOST, dataset=NEUPRINT_DATASET, token=NEUPRINT_TOKEN)

# Convenience so your downstream code that expects `neu` to be the navis bridge keeps working
neu = navnp


import re


## Identify actuators

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import re

def list_mjcf_actuators(xml_path):
    """
    Parse MJCF and return a DataFrame with actuator and joint metadata:
    [actuator_name, type, joint, gear, ctrlrange, side, limb, axis, direction_hint]
    """
    xml_path = Path(xml_path)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = ""  # MuJoCo doesn't require XML namespaces

    # Collect joints for axis/direction hints if present
    joint_meta = {}
    for j in root.findall(".//joint"):
        name = j.attrib.get("name")
        axis = j.attrib.get("axis", "")
        # normalize axis label
        axis_hint = {"1 0 0":"x","0 1 0":"y","0 0 1":"z"}.get(axis.strip(), axis.strip())
        joint_meta[name] = {"axis_hint": axis_hint}

    rows = []
    # MuJoCo actuators are usually <motor>, <position>, <velocity>, <general>
    for tag in ("motor","position","velocity","general"):
        for a in root.findall(f".//{tag}"):
            aname = a.attrib.get("name", "")
            joint = a.attrib.get("joint", "")
            gear  = a.attrib.get("gear", "")
            ctrl  = a.attrib.get("ctrlrange", "")
            # crude, but practical: infer side/limb/joint from naming
            n = aname.lower()
            side = ("L" if re.search(r"\b(l|left)\b", n) else
                    "R" if re.search(r"\b(r|right)\b", n) else None)
            # common limb tokens seen in fly models
            limb = ( "fore"   if "fore"   in n or "front"  in n else
                     "mid"    if "mid"    in n else
                     "hind"   if "hind"   in n or "rear"   in n else None)
            # joint segment tokens (coxa/femur/tibia/tarsus) or body joints (head/wing/etc.)
            seg = None
            for t in ("coxa","femur","tibia","tarsus","trochanter","head","wing","abdomen","antenna"):
                if t in n:
                    seg = t; break

            axis_hint = joint_meta.get(joint,{}).get("axis_hint","")
            # sign hint (purely name-based; will be calibrated later)
            direction_hint = ("flex" if any(k in n for k in ["flex","retract","adduct","depress"])
                              else "extend" if any(k in n for k in ["extend","protract","abduct","elevate"])
                              else None)

            rows.append({
                "actuator_name": a.attrib.get("name",""),
                "type": tag,
                "joint": joint,
                "gear": gear,
                "ctrlrange": ctrl,
                "side": side,
                "limb_group": limb,
                "segment": seg,
                "axis_hint": axis_hint,
                "direction_hint": direction_hint
            })
    df = pd.DataFrame(rows).sort_values(["side","limb_group","segment","actuator_name"], na_position="last")
    return df

# EXAMPLE: point this to your model (adjust the path to the MJCF used by flybody)
df_act = list_mjcf_actuators("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/build_fruitfly/fruitfly.xml")
print(df_act.to_string(index=False))


## Clean up MN naming for folders

In [ ]:
# === MN candidates → annotate (instance/type/origin/target) and rebucket by Instance ===
import os, json, re, shutil, math
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from neuprint import fetch_custom
except Exception as e:
    fetch_custom = None

MN_CAND_ROOT = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates")

def _sanitize_folder_name(s: str) -> str:
    """Filesystem-safe folder name (letters, digits, _, -, .)."""
    return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s)).strip('-_.') or "unnamed"

def _discover_body_id_dirs(root: Path) -> list[int]:
    root = Path(root)
    if not root.exists():
        print(f"[mn-bucket] root missing: {root}")
        return []
    ids = []
    for p in sorted(root.iterdir()):
        if p.is_dir() and p.name.isdigit():
            ids.append(int(p.name))
    return ids

def _chunked(iterable, n):
    it = list(iterable)
    for i in range(0, len(it), n):
        yield it[i:i+n]

def _fetch_labels_for_ids(client, ids: list[int]) -> pd.DataFrame:
    """
    Batch fetch of (instance, type, origin/Origin, target/Target) for bodyIds.
    Returns DataFrame with columns: bodyId, instance, type, origin, target
    """
    if fetch_custom is None:
        raise RuntimeError("neuprint Python client is not available in this session.")
    if not ids:
        return pd.DataFrame(columns=["bodyId","instance","type","origin","target"])

    rows = []
    # Keep IN-list sizes reasonable (neuPrint is fine with a few hundred–1k per query)
    for chunk in _chunked(sorted(set(int(i) for i in ids)), 800):
        idlist = ",".join(str(int(i)) for i in chunk)
        cypher = f"""
        MATCH (n:Neuron)
        WHERE n.bodyId IN [{idlist}]
        RETURN n.bodyId AS bodyId,
               n.instance AS instance,
               n.type AS type,
               n.Origin AS Origin,  n.origin AS origin,
               n.Target AS Target,  n.target AS target
        """
        res = fetch_custom(cypher, client=client)
        df = res[0] if isinstance(res, tuple) else res
        if df is None or df.empty:
            continue
        rows.append(df)

    if not rows:
        return pd.DataFrame(columns=["bodyId","instance","type","origin","target"])

    df = pd.concat(rows, ignore_index=True)
    # coalesce Origin/origin → origin ; Target/target → target
    if "origin" not in df.columns: df["origin"] = np.nan
    if "Origin" in df.columns:
        df["origin"] = df["origin"].fillna(df["Origin"])
    if "target" not in df.columns: df["target"] = np.nan
    if "Target" in df.columns:
        df["target"] = df["target"].fillna(df["Target"])

    # standardize empties
    for c in ("instance","type","origin","target"):
        if c in df.columns:
            df[c] = df[c].replace({"": np.nan, "None": np.nan, "none": np.nan})

    keep_cols = ["bodyId","instance","type","origin","target"]
    for c in keep_cols:
        if c not in df.columns:
            df[c] = np.nan

    # drop dups (one row per bodyId)
    df = (df[keep_cols].drop_duplicates(subset=["bodyId"])
          .sort_values("bodyId")
          .reset_index(drop=True))
    return df

def annotate_and_rebucket_mn_candidates(client,
                                        root_dir: str | Path = MN_CAND_ROOT,
                                        dry_run=False,
                                        keep_old_structure_symlink=False,
                                        write_index_csv=True):
    """
    1) For each <root>/<ID>, query neuPrint to get instance/type/origin/target.
       Save to <root>/<ID>/metadata.json
    2) Move <root>/<ID> -> <root>/<Instance>/<ID>
       (Instance folder uses sanitized name; if missing, falls back to type, else 'UNLABELED').

    Args:
      client: neuprint.Client already configured to your dataset.
      root_dir: path to MN/MN_candidates
      dry_run: if True, only print planned changes.
      keep_old_structure_symlink: if True, leave a symlink at <root>/<ID> pointing to new location.
      write_index_csv: if True, also writes <root>/_mn_instance_index.csv
    """
    root = Path(root_dir).expanduser().resolve()
    ids = _discover_body_id_dirs(root)
    if not ids:
        print(f"[mn-bucket] No ID folders under: {root}")
        return pd.DataFrame()

    print(f"[mn-bucket] Found {len(ids)} ID folders.")

    # Batch pull labels
    meta = _fetch_labels_for_ids(client, ids)

    # per-ID save & rebucket plan
    plan = []
    for bid in ids:
        row = meta.loc[meta["bodyId"] == bid].head(1)
        inst = str(row["instance"].iloc[0]).strip() if not row.empty and pd.notna(row["instance"].iloc[0]) else ""
        typ  = str(row["type"].iloc[0]).strip() if not row.empty and pd.notna(row["type"].iloc[0]) else ""
        org  = str(row["origin"].iloc[0]).strip() if not row.empty and pd.notna(row["origin"].iloc[0]) else ""
        tgt  = str(row["target"].iloc[0]).strip() if not row.empty and pd.notna(row["target"].iloc[0]) else ""

        # destination folder name preference: instance → type → UNLABELED
        label_for_dir = inst or typ or "UNLABELED"
        dst_dir_name  = _sanitize_folder_name(label_for_dir)

        src = root / str(bid)
        dst = root / dst_dir_name / str(bid)

        plan.append({
            "bodyId": bid,
            "instance": inst,
            "type": typ,
            "origin": org,
            "target": tgt,
            "src": str(src),
            "dst": str(dst),
            "dst_type_dir": str(root / dst_dir_name)
        })

    # write metadata.json in each ID folder
    for item in plan:
        meta_path = Path(item["src"]) / "metadata.json"
        payload = {
            "bodyId": item["bodyId"],
            "instance": item["instance"],
            "type": item["type"],
            "origin": item["origin"],
            "target": item["target"]
        }
        if dry_run:
            print(f"[dry] write {meta_path}  ←  {payload}")
        else:
            try:
                with open(meta_path, "w") as f:
                    json.dump(payload, f, indent=2)
            except Exception as e:
                print(f"[warn] could not write {meta_path}: {e}")

    # show planned moves
    print("\n[mn-bucket] Planned moves:")
    for item in plan:
        print(f"  {item['src']}  →  {item['dst']}")

    if dry_run:
        print("[mn-bucket] Dry run only — no folders moved.")
    else:
        # perform moves
        for item in plan:
            src = Path(item["src"])
            dst = Path(item["dst"])
            try:
                dst.parent.mkdir(parents=True, exist_ok=True)
                # If a destination exists already, try to merge (move contents) then remove source
                if dst.exists():
                    # copy files inside src into dst (without clobbering existing files)
                    for child in src.iterdir():
                        target = dst / child.name
                        if target.exists():
                            continue
                        if child.is_dir():
                            shutil.move(str(child), str(target))
                        else:
                            shutil.move(str(child), str(dst))
                    # remove empty src
                    try:
                        src.rmdir()
                    except OSError:
                        pass
                else:
                    shutil.move(str(src), str(dst))

                # optional back-compat symlink at old location
                if keep_old_structure_symlink:
                    try:
                        link = src
                        if not link.exists():
                            link.symlink_to(dst)
                    except Exception as e:
                        print(f"[warn] symlink failed {src} → {dst}: {e}")

            except Exception as e:
                print(f"[error] move failed {src} → {dst}: {e}")

    # index CSV
    out_df = pd.DataFrame(plan).sort_values(["dst_type_dir","bodyId"])
    if write_index_csv:
        idx_path = root / "_mn_instance_index.csv"
        try:
            out_df.to_csv(idx_path, index=False)
            print(f"\n[mn-bucket] Wrote index → {idx_path}")
        except Exception as e:
            print(f"[warn] could not write index CSV: {e}")

    print("[mn-bucket] Done.")
    return out_df


In [ ]:
# # After you’ve set up `client = neuprint.Client(..., dataset=..., token=...)`
# annotate_and_rebucket_mn_candidates(client,
#     root_dir="/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates",
#     dry_run=False,                      # set True first to preview
#     keep_old_structure_symlink=False,   # set True if you want legacy links left behind
#     write_index_csv=True
# )


### Other Version OLD

In [ ]:
# import os, re, csv, shutil, json
# from pathlib import Path
# import pandas as pd

# # ---------- NeuPrint helpers ----------
# def get_full_type_from_neuprint(body_id, client):
#     """
#     Returns the *full* MN label from NeuPrint, prioritizing `instance`
#     (e.g., "MNwm34_PDMNp_L"). Falls back to `type`, `systematicType`, etc.
#     """
#     from neuprint import NeuronCriteria, fetch_neurons
#     df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
#     if df is None or df.empty:
#         return None

#     # Prefer instance first (human-readable + encodes side/nerve)
#     priority = ("instance", "type", "systematicType", "name", "cellType")
#     for col in priority:
#         if col in df.columns:
#             val = df.iloc[0].get(col, None)
#             if pd.notna(val):
#                 sval = str(val).strip()
#                 if sval:
#                     return sval
#     return None


# # ---------- Full-name parsing ----------
# # Nerve → coarse segment hints
# NERVE_BASES = {
#     "CvN": "head",  # cervical nerve → head/neck/antenna class
#     "DProN": "T1",  "VProN": "T1",  "ProAN": "T1",  "ProLN": "T1",
#     "ADMN": "T2",   "PDMN": "T2",   "MesoAN": "T2", "MesoLN": "T2",
#     "DMetaN": "T3", "MetaLN": "T3",
#     "AbN1": "Ab", "AbN2": "Ab", "AbN3": "Ab", "AbN4": "Ab", "AbNT": "Ab",
# }

# # Limb group routing by nerve
# NERVE_GROUP = {
#     "ProLN": "leg", "MesoLN": "leg", "MetaLN": "leg",
#     "ADMN": "wing", "PDMN": "wing",
#     "CvN": "head", "MesoAN": "wing_accessory", "ProAN": "pro_accessory",
#     "DProN": "pro_dorsal", "VProN": "pro_ventral", "DMetaN": "meta_dorsal",
#     "AbN1": "abdomen", "AbN2": "abdomen", "AbN3": "abdomen", "AbN4": "abdomen", "AbNT": "abdomen",
# }

# # Regexes for parsing
# SIDE_RX   = re.compile(r'(?:^|[_\-])(L|R|ND)(?:$|[_\-])', re.I)
# NERVE_RX  = re.compile(r'\b(CvN|DProN|VProN|ProAN|ProLN|ADMN|PDMN[pd]?|MesoAN|MesoLN|DMetaN|MetaLN|AbN[1-4]|AbNT)\b', re.I)
# THORAX_RX = re.compile(r'\b(T1|T2|T3)\b', re.I)

# def parse_full_mn_name(full_type):
#     """
#     Parse strings like "MNwm34_PDMNp_L" into components.
#     Returns dict(type_short, instance, nerve_token, nerve_base, side, thorax, segment_hint, group)

#     - side: 'L' | 'R' | 'ND' | ''  (raw token)
#     - thorax: 'T1'|'T2'|'T3' if inferred; else ''
#     - segment_hint: same as thorax but may carry 'Ab' for abdomen
#     """
#     instance = str(full_type).strip()
#     tokens = instance.split('_') if instance else []

#     # side
#     side = ""
#     if tokens and tokens[-1] in ("L", "R", "ND"):
#         side = tokens[-1]
#     else:
#         m = SIDE_RX.search(instance)
#         if m:
#             side = m.group(1).upper()

#     # nerve token & base (strip trailing branch letters like 'p'/'d')
#     nerve_token = ""
#     m = NERVE_RX.search(instance)
#     if m:
#         nerve_token = m.group(1)
#     elif len(tokens) >= 2:
#         # heuristic: middle token often nerve; take last non-side token
#         nerve_token = tokens[-2] if (tokens[-1] in ("L","R","ND") and len(tokens) >= 2) else tokens[-1]

#     nerve_base = re.sub(r'[a-z]+$', '', nerve_token) if nerve_token else ""

#     # thorax/segment hint
#     thorax = ""
#     m_t = THORAX_RX.search(instance)
#     if m_t:
#         thorax = m_t.group(1).upper()
#     else:
#         hint = NERVE_BASES.get(nerve_base, "")
#         thorax = hint if hint in ("T1","T2","T3") else ""

#     segment_hint = thorax if thorax else NERVE_BASES.get(nerve_base, "")

#     group = NERVE_GROUP.get(nerve_base, "")
#     type_short = tokens[0] if tokens else instance

#     return dict(
#         type_short=type_short,
#         instance=instance,
#         nerve_token=nerve_token,
#         nerve_base=nerve_base,
#         side=side,
#         thorax=thorax,
#         segment_hint=segment_hint,
#         group=group
#     )


# # ---------- Filesystem ops ----------
# def sanitize_folder_name(s):
#     """Make a filesystem-safe folder name (alnum, _, -, .)."""
#     return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s)).strip('-_.') or "unnamed"

# def rename_mn_dirs(mn_root, client, dry_run=True, make_symlink=True, write_index_json=True):
#     """
#     Walk /MN/<short_type>/<id> and rename <short_type> → <instance> from NeuPrint.
#     Keeps <id> folder as-is. Optionally leaves a symlink from old to new and writes an index JSON.
#     """
#     mn_root = Path(mn_root).expanduser().resolve()
#     todo = []
#     index = []  # for optional JSON index
#     for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         short_type = type_dir.name
#         for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir() and p.name.isdigit()):
#             bid = int(id_dir.name)
#             full = get_full_type_from_neuprint(bid, client)
#             if not full:
#                 continue
#             dst_type = sanitize_folder_name(full)
#             if dst_type == short_type:
#                 continue
#             src = type_dir / str(bid)
#             dst = mn_root / dst_type / str(bid)
#             todo.append((src, dst, short_type, dst_type, bid, full))
#             index.append({
#                 "bodyId": bid,
#                 "old_type": short_type,
#                 "new_instance": full,
#                 "new_type_folder": dst_type,
#                 "old_path": str(src),
#                 "new_path": str(dst)
#             })

#     if not todo:
#         print("[rename] Nothing to rename.")
#         return []

#     print(f"[rename] Planned renames: {len(todo)}")
#     for src, dst, sshort, sfull_dir, bid, full in todo:
#         print(f"  {sshort}/{bid}  →  {sfull_dir}/{bid}")

#     if dry_run:
#         print("[rename] Dry run only. Re-run with dry_run=False to apply.")
#         return todo

#     for src, dst, sshort, sfull_dir, bid, full in todo:
#         dst.parent.mkdir(parents=True, exist_ok=True)
#         shutil.move(str(src), str(dst))

#         # remove empty old type dir
#         old_type_dir = src.parent
#         try:
#             if not any(old_type_dir.iterdir()):
#                 old_type_dir.rmdir()
#         except Exception:
#             pass

#         # optional back-compat symlink
#         if make_symlink:
#             try:
#                 old_type_dir.mkdir(parents=True, exist_ok=True)
#                 link = old_type_dir / str(bid)
#                 if not link.exists():
#                     link.symlink_to(dst)
#             except Exception as e:
#                 print(f"  [warn] symlink failed for {src} → {dst}: {e}")

#     if write_index_json:
#         idx_path = mn_root / "_rename_index.json"
#         try:
#             with open(idx_path, "w") as f:
#                 json.dump(index, f, indent=2)
#             print(f"[rename] Wrote index → {idx_path}")
#         except Exception as e:
#             print(f"[rename] Could not write index JSON: {e}")

#     print("[rename] Done.")
#     return todo


# # ---------- Mapping CSV from renamed dirs ----------
# # Toggle: for leg MNs, auto-fill a COARSE default joint/dof so you can run right away.
# COARSE_LEG_DEFAULTS = False   # set True to default to tibia/main, sign=+1

# def infer_joint_defaults_from_group(thorax, side_token, group):
#     """
#     For non-leg groups we can map to named actuators directly.
#     For legs we either leave joint/dof blank or give a coarse default.
#     Returns list of dict rows (may be empty).
#     """
#     side = {"L": "left", "R": "right"}.get(side_token, "")

#     if group == "wing":
#         if side == "left":  return [{"joint": "wing", "dof": "pitch", "actuator_name": "wing_pitch_left"}]
#         if side == "right": return [{"joint": "wing", "dof": "pitch", "actuator_name": "wing_pitch_right"}]
#         return []

#     if group == "abdomen":
#         return [{"joint": "abdomen", "dof": "main", "actuator_name": "abdomen"}]

#     if group == "head":
#         return [{"joint": "head", "dof": "main", "actuator_name": "head"}]

#     if group.endswith("accessory") or group.endswith("dorsal") or group.endswith("ventral"):
#         # ambiguous routes (leave blank for manual assignment)
#         return []

#     if group == "leg":
#         if not COARSE_LEG_DEFAULTS:
#             return []
#         # coarse default: tibia main on segment thorax/side
#         return [{"joint": "tibia", "dof": "main", "actuator_name": ""}]

#     return []





### Identify folders to rename

In [ ]:
# # MN_ROOT = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# # rename_mn_dirs(MN_ROOT, client, dry_run=True)


# # ===== Run it with your paths =====
# MN_ROOT = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# _ = build_mapping_from_dirs_with_targets(MN_ROOT, client, out_csv="mn_to_actuator_mapping.csv")

### Run Renaming

In [ ]:
# rename_mn_dirs(MN_ROOT, client, dry_run=False, make_symlink=True)

### Audit folder names

In [ ]:
# from pathlib import Path
# import pandas as pd

# def audit_mn_dirs(mn_root, client):
#     from neuprint import NeuronCriteria, fetch_neurons
#     mn_root = Path(mn_root).expanduser().resolve()
#     rows = []
#     for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         short = type_dir.name
#         for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir()):
#             bid = id_dir.name
#             if not bid.isdigit():
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: non-numeric id"))
#                 continue
#             df,_ = fetch_neurons(NeuronCriteria(bodyId=int(bid)), client=client)
#             if df is None or df.empty:
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: neuprint empty"))
#                 continue
#             full = None
#             for col in ("type","systematicType","instance","name","cellType"):
#                 if col in df.columns and pd.notna(df.iloc[0].get(col, None)):
#                     full = str(df.iloc[0][col]).strip(); 
#                     if full: break
#             if not full:
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: no full name"))
#             elif full == short:
#                 rows.append(dict(id=bid, short=short, neu_full=full, action="skip: same name"))
#             else:
#                 rows.append(dict(id=bid, short=short, neu_full=full, action="RENAME"))
#     audit_df = pd.DataFrame(rows, columns=["id","short","neu_full","action"])
#     print(audit_df.to_string(index=False))
#     return audit_df

# # usage
# MN_ROOT = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# _ = audit_mn_dirs(MN_ROOT, client)


## Build Muscle to Actuator control CSV (Up-to-Date, run all 4)

In [ ]:
# from pathlib import Path
# import json
# import pandas as pd

# def build_mapping_from_dirs(
#     mn_root,
#     out_csv="mn_to_actuator_mapping.csv",
#     prefer_metadata=True,
#     client=None,             # optional: only used if you want to refetch missing metadata
#     refetch_if_missing=False # set True to try neuPrint when metadata.json is missing/incomplete
# ):
#     """
#     Creates (or updates) a CSV with rows for each MN found under mn_root.
#     Assumes ID folders have been rebucketed under instance folders:
#         <mn_root>/<Instance>/<ID>
#     Reads metadata from <ID>/metadata.json (instance/type/origin/target) when present.

#     New columns added: 'origin', 'target'
#     """

#     mn_root = Path(mn_root).expanduser().resolve()
#     rows = []

#     # --- optional helper if you want to refetch missing metadata on the fly ---
#     def _maybe_refetch(bid: int):
#         if (client is None) or (not refetch_if_missing):
#             return {}
#         try:
#             # lightweight batched query replaced with one-body fallback
#             from neuprint import fetch_custom
#             cy = f"""
#             MATCH (n:Neuron) WHERE n.bodyId = {int(bid)}
#             RETURN n.instance AS instance, n.type AS type,
#                    coalesce(n.origin, n.Origin) AS origin,
#                    coalesce(n.target, n.Target) AS target
#             """
#             res = fetch_custom(cy, client=client)
#             df = res[0] if isinstance(res, tuple) else res
#             if df is None or df.empty:
#                 return {}
#             r = df.iloc[0].to_dict()
#             # normalize empties
#             for k in ("instance","type","origin","target"):
#                 v = r.get(k, None)
#                 if v is None: continue
#                 s = str(v).strip()
#                 r[k] = s if s else None
#             return r
#         except Exception:
#             return {}

#     for instance_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         instance_name = instance_dir.name  # expected sanitized INSTANCE (e.g., MNfl27_ProLN_L)

#         # derive thorax/side/group/etc from the instance folder name
#         info = parse_full_mn_name(instance_name)
#         thorax       = info.get("thorax","") or ""
#         side_token   = info.get("side","")
#         side         = {"L": "left", "R": "right"}.get(side_token, "")
#         group        = info.get("group","")
#         nerve_base   = info.get("nerve_base","")
#         segment_hint = info.get("segment_hint","")

#         for id_dir in sorted(p for p in instance_dir.iterdir() if p.is_dir() and p.name.isdigit()):
#             bid = int(id_dir.name)

#             # 1) Try metadata.json inside the ID folder
#             meta_path = id_dir / "metadata.json"
#             meta = {}
#             if prefer_metadata and meta_path.exists():
#                 try:
#                     with open(meta_path, "r") as f:
#                         meta = json.load(f) or {}
#                 except Exception:
#                     meta = {}

#             # normalize metadata fields; fall back to instance folder name
#             mn_instance = (str(meta.get("instance") or instance_name)).strip()
#             mn_type     = (str(meta.get("type") or "")).strip() or mn_instance  # keep your old behavior
#             origin      = (str(meta.get("origin") or "")).strip()
#             target      = (str(meta.get("target") or "")).strip()

#             # 2) If still missing and refetch is enabled, hit neuPrint
#             if refetch_if_missing and (not origin or not target or not mn_instance):
#                 ref = _maybe_refetch(bid)
#                 mn_instance = ref.get("instance", mn_instance)
#                 mn_type     = ref.get("type", mn_type)
#                 origin      = ref.get("origin", origin)
#                 target      = ref.get("target", target)

#             # 3) Build default rows as before (can be multiple rows via your helper)
#             defaults = infer_joint_defaults_from_group(thorax, side_token, group)

#             if not defaults:
#                 rows.append({
#                     "mn_type": mn_instance,        # same meaning as before (full instance)
#                     "mn_id": bid,
#                     "thorax": thorax,
#                     "side": side,
#                     "nerve_base": nerve_base,
#                     "segment_hint": segment_hint,
#                     "group": group,
#                     # NEW:
#                     "origin": origin,
#                     "target": target,
#                     # editable mapping fields:
#                     "joint": "",
#                     "dof": "",
#                     "action": "",
#                     "gain": 1.0,
#                     "bias": 0.0,
#                     "sign": "",
#                     "actuator_name": ""
#                 })
#             else:
#                 for d in defaults:
#                     rows.append({
#                         "mn_type": mn_instance,
#                         "mn_id": bid,
#                         "thorax": thorax,
#                         "side": side,
#                         "nerve_base": nerve_base,
#                         "segment_hint": segment_hint,
#                         "group": group,
#                         # NEW:
#                         "origin": origin,
#                         "target": target,
#                         # pre-filled mapping suggestions:
#                         "joint": d.get("joint", ""),
#                         "dof": d.get("dof", ""),
#                         "action": "",
#                         "gain": 1.0,
#                         "bias": 0.0,
#                         "sign": "",
#                         "actuator_name": d.get("actuator_name", "")
#                     })

#     cols = [
#         "mn_type", "mn_id", "thorax", "side",
#         "nerve_base", "segment_hint", "group",
#         # NEW columns (right after your anatomical hints):
#         "origin", "target",
#         # editable mapping fields (unchanged):
#         "joint", "dof", "action", "gain", "bias", "sign", "actuator_name"
#     ]
#     df = pd.DataFrame(rows, columns=cols)

#     # stable sort: by mn_type then mn_id
#     if not df.empty:
#         df = df.sort_values(["mn_type", "mn_id"]).reset_index(drop=True)

#     df.to_csv(out_csv, index=False)
#     print(f"[mapping] Wrote {len(df)} rows → {out_csv}")
#     print("  Fill any blanks (esp. leg joint/dof), set sign (+1/-1). Now you also have origin/target from neuPrint.")
#     return df


In [ ]:
# build_mapping_from_dirs(
#     "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates",
#     out_csv="/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv",
#     client=client,
#     refetch_if_missing=True   # only if you want to query neuPrint for gaps
# )



In [ ]:
# import re
# import pandas as pd
# import numpy as np

# # ---- SAFE string helpers ----
# def s(x):
#     """Return '' for None/NaN, else str(x)."""
#     if x is None: return ""
#     if isinstance(x, float) and (np.isnan(x)): return ""
#     return str(x)

# def slower(x):
#     v = s(x)
#     return v.lower()

# # ---- extractors ----
# def _seg_num(thorax_str: str) -> str:
#     m = re.search(r'T([123])', s(thorax_str), flags=re.I)
#     return m.group(1) if m else ""

# def _side_lr(side_str: str) -> str:
#     ss = slower(side_str)
#     if ss in ("l","left"):  return "left"
#     if ss in ("r","right"): return "right"
#     return ""

# def _contains(txt, *keys):
#     t = slower(txt)
#     return all(k.lower() in t for k in keys)

# def _any(txt, *keys):
#     t = slower(txt)
#     return any(k.lower() in t for k in keys)

# # Optional: if you have parse_full_mn_name() already defined, we can use it
# def _fallback_from_mn_type(mn_type):
#     # expects parse_full_mn_name to be available in your session
#     try:
#         info = parse_full_mn_name(s(mn_type))
#         thorax = info.get("thorax","") or info.get("segment_hint","")
#         side_token = info.get("side","")
#         side = {"L":"left","R":"right"}.get(side_token, "")
#         group = info.get("group","")
#         return thorax, side, group
#     except Exception:
#         return "", "", ""

# # ---- main mapper from Target text → actuator_name (+joint/dof hints) ----
# def map_target_to_actuator(thorax, side, target, limb_group=None):
#     seg = _seg_num(thorax)
#     lr  = _side_lr(side)
#     t   = slower(target)
    
#     # TTM (tergotrochanteral muscle) → use femur actuator as trochanter proxy
#     if _any(t, "ttm", "tergotrochanter", "tergo-trochanter"):
#         if not seg or not lr:
#             return "", "", ""
#         return "femur", "", f"femur_T{seg}_{lr}"

#     # Antenna / head / wing / abdomen first (don’t require seg/lr)
#     if _any(t, "antenna", "antennal"):
#         if _any(t, "abduct", "abductor"):
#             return "antenna", "abduct", f"antenna_abduct_{lr or 'left'}"
#         if _any(t, "twist", "rotat"):
#             return "antenna", "twist", f"antenna_twist_{lr or 'left'}"
#         return "antenna", "", f"antenna_{lr or 'left'}"

#     if _any(t, "wing"):
#         return "wing", "pitch", f"wing_pitch_{lr or 'left'}"

#     if _any(t, "abdomen", "abdominal"):
#         if _any(t, "abduct"):
#             return "abdomen", "abduct", "abdomen_abduct"
#         return "abdomen", "", "abdomen"

#     if _any(t, "head"):
#         if _any(t, "abduct"):
#             return "head", "abduct", "head_abduct"
#         if _any(t, "twist", "yaw"):
#             return "head", "twist", "head_twist"
#         return "head", "", "head"

#     # Legs (need segment + side)
#     lg = slower(limb_group)
#     if lg == "leg" or _any(t, "tibia", "femur", "coxa", "trochanter", "tarsus"):
#         if not seg or not lr:
#             return "", "", ""  # cannot choose a T{1/2/3}_{left/right} actuator

#         # Tarsus
#         if _any(t, "tarsus2", "tarsal2"):
#             return "tarsus", "", f"tarsus2_T{seg}_{lr}"
#         if _any(t, "tarsus", "tarsal"):
#             return "tarsus", "", f"tarsus_T{seg}_{lr}"

#         # Tibia
#         if _any(t, "tibia"):
#             return "tibia", "", f"tibia_T{seg}_{lr}"

#         # Coxa
#         if _any(t, "coxa"):
#             if _any(t, "abduct", "abductor"):
#                 return "coxa", "abduct", f"coxa_abduct_T{seg}_{lr}"
#             if _any(t, "twist", "rotat"):
#                 return "coxa", "twist", f"coxa_twist_T{seg}_{lr}"
#             return "coxa", "", f"coxa_T{seg}_{lr}"

#         # Trochanter (incl. accessory trochanter) → femur as safe proxy
#         if _any(t, "trochanter", "acc. tr", "accessory tr", "acc.tr", "tr "):
#             return "femur", "", f"femur_T{seg}_{lr}"

#         # Femur
#         if _any(t, "femur", "femoral"):
#             if _any(t, "twist", "rotat"):
#                 return "femur", "twist", f"femur_twist_T{seg}_{lr}"
#             return "femur", "", f"femur_T{seg}_{lr}"

#     # Unknown
#     return "", "", ""

# # ----- hard overrides (by instance or by bodyId) -----
# MN_OVERRIDES_BY_TYPE = {
#     # “wm” is a misnomer; these are T2 leg jump MNs
#     "MNwm34_PDMNp_L": {"group": "leg", "thorax": "T2", "side": "left",
#                        "joint": "femur", "dof": "", "actuator_name": "femur_T2_left"},
#     "MNwm34_PDMNp_R": {"group": "leg", "thorax": "T2", "side": "right",
#                        "joint": "femur", "dof": "", "actuator_name": "femur_T2_right"},
# }
# MN_OVERRIDES_BY_ID = {
#     # if you prefer to key by bodyId instead, add entries like:
#     # 10068: {"group":"leg","thorax":"T2","side":"left","joint":"femur","dof":"","actuator_name":"femur_T2_left"},
#     # 10110: {"group":"leg","thorax":"T2","side":"right","joint":"femur","dof":"","actuator_name":"femur_T2_right"},
# }


# def autofill_actuators(csv_path, out_csv=None, target_col="target"):
#     """
#     Fills joint/dof/actuator_name in your mapping CSV using:
#       1) hard overrides (MN_OVERRIDES_BY_TYPE / _BY_ID)
#       2) map_target_to_actuator() heuristics (incl. TTM rule)
#     Only writes when we can determine an actuator_name.
#     """
#     import pandas as pd
#     from pathlib import Path
#     def s(x): return "" if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x)

#     csv_path = Path(csv_path).expanduser().resolve()
#     if out_csv is None:
#         out_csv = csv_path
#     else:
#         out_csv = Path(out_csv).expanduser().resolve()

#     df = pd.read_csv(csv_path)

#     # make sure columns exist
#     for c in ["mn_type","mn_id","thorax","side","group","joint","dof","actuator_name"]:
#         if c not in df.columns:
#             df[c] = ""

#     filled = 0
#     for i, row in df.iterrows():
#         # ----- per-MN overrides -----
#         mn_type = s(row.get("mn_type",""))
#         mn_id   = row.get("mn_id", None)
#         try:
#             # normalize possible floats to int
#             if isinstance(mn_id, float) and not pd.isna(mn_id):
#                 mn_id = int(mn_id)
#         except Exception:
#             pass

#         # by instance
#         if mn_type in MN_OVERRIDES_BY_TYPE:
#             o = MN_OVERRIDES_BY_TYPE[mn_type]
#             # update context fields (thorax/side/group) if provided
#             for k in ("group","thorax","side"):
#                 if k in o and o[k]:
#                     df.at[i, k] = o[k]
#             df.at[i, "joint"]         = o.get("joint", s(row.get("joint","")))
#             df.at[i, "dof"]           = o.get("dof", s(row.get("dof","")))
#             df.at[i, "actuator_name"] = o.get("actuator_name", s(row.get("actuator_name","")))
#             filled += 1
#             continue

#         # by bodyId
#         if mn_id in MN_OVERRIDES_BY_ID:
#             o = MN_OVERRIDES_BY_ID[mn_id]
#             for k in ("group","thorax","side"):
#                 if k in o and o[k]:
#                     df.at[i, k] = o[k]
#             df.at[i, "joint"]         = o.get("joint", s(row.get("joint","")))
#             df.at[i, "dof"]           = o.get("dof", s(row.get("dof","")))
#             df.at[i, "actuator_name"] = o.get("actuator_name", s(row.get("actuator_name","")))
#             filled += 1
#             continue

#         # ----- heuristic mapping -----
#         tgt = s(row.get(target_col, ""))
#         joint, dof, name = map_target_to_actuator(
#             s(row.get("thorax","")),
#             s(row.get("side","")),
#             tgt,
#             s(row.get("group",""))
#         )
#         if name:
#             df.at[i, "joint"]         = joint
#             df.at[i, "dof"]           = dof
#             df.at[i, "actuator_name"] = name
#             filled += 1

#     df.to_csv(out_csv, index=False)
#     print(f"[autofill] Updated {filled} rows → {out_csv}")
#     return df



In [ ]:
# autofill_actuators(
#     "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv"
# )

## MN to MuJoCo full CSV Mapping

### Step 1: Fetch  labels for bodyIds

In [ ]:
import re, os, json, shutil
from pathlib import Path
import pandas as pd

def fetch_mn_labels(body_id, client):
    """
    Returns a dict with ALL the columns your dataset has that are useful for labeling,
    using only columns that exist in your schema.
    """
    from neuprint import fetch_neurons, NeuronCriteria
    df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
    if df is None or df.empty:
        return None

    row = df.iloc[0]
    # Pull only if present in columns
    def get(c, default=None):
        return row[c] if (c in df.columns and pd.notna(row[c])) else default

    D = {
        "bodyId":        int(body_id),
        "instance":      str(get("instance", "") or "").strip(),
        "type":          str(get("type", "") or "").strip(),
        "class":         str(get("class", "") or "").strip(),
        "subclass":      str(get("subclass", "") or "").strip(),
        "subclassabbr":  str(get("subclassabbr", "") or "").strip(),
        "group":         str(get("group", "") or "").strip(),
        "hemilineage":   str(get("hemilineage", "") or "").strip(),
        "exitNerve":     str(get("exitNerve", "") or "").strip(),
        "entryNerve":    str(get("entryNerve", "") or "").strip(),
        "somaNeuromere": str(get("somaNeuromere", "") or "").strip(),  # e.g. T1/T2/T3
        "somaSide":      str(get("somaSide", "") or "").strip(),        # e.g. left/right
        "vfbId":         str(get("vfbId", "") or "").strip(),
        "status":        str(get("status","") or "").strip(),
        "statusLabel":   str(get("statusLabel","") or "").strip(),
        "description":   str(get("description","") or "").strip(),
        "predictedNt":   str(get("predictedNt","") or "").strip(),
    }

    # If side is missing, infer from instance suffix _L/_R if present
    if not D["somaSide"]:
        m = re.search(r"_(L|R)(?:$|_)", D["instance"])
        if m:
            D["somaSide"] = {"L":"left","R":"right"}[m.group(1)]

    # Normalize a few convenience flags
    inst = D["instance"]
    D["is_motor"] = (D["class"].lower() == "wm" or "motor" in D["group"].lower())
    D["is_leg"]   = ("leg" in D["subclass"].lower()) or ("LegNp" in D["subclass"]) or ("ProLN" in D["exitNerve"] or "MesoLN" in D["exitNerve"] or "MetaLN" in D["exitNerve"])
    D["is_wing"]  = ("wing" in D["group"].lower()) or ("ADMN" in D["exitNerve"] or "PDMN" in D["exitNerve"])
    return D
    
# ----- smarter grouping from neuprint labels -----

# small, explicit exception table for stubborn mislabels
# keys can be an instance (preferred) or a short type
SMART_OVERRIDES = {
    r"^MNwm\d+": {"group": "leg"},  # <- GOOD
}


def _apply_overrides(text, default_group):
    for pat, fix in SMART_OVERRIDES.items():
        if re.search(pat, text or "", flags=re.IGNORECASE):
            return fix.get("group", default_group)
    return default_group

def smarter_group_from_labels(labels: dict) -> tuple[str, str]:
    """
    Return (group, thorax) using neuprint fields with sensible priority:
      1) subclass / systematicType / instance tokens (LegNp(T), Wing, etc.)
      2) class/group fallbacks
      3) exitNerve LAST (unreliable for MNwm34/TTMn cases)
    """
    inst = (labels.get("instance") or labels.get("type") or "").strip()
    syst = (labels.get("systematicType") or "").strip()
    subc = (labels.get("subclass") or "").strip()
    cls  = (labels.get("class") or "").strip()
    grp  = (labels.get("group") or "").strip()
    exitNerve = (labels.get("exitNerve") or "").strip()

    text = " | ".join([inst, syst, subc, cls, grp, exitNerve])

    # 1) explicit tokens that trump nerves
    # Leg neuropil
    m = re.search(r"LegNp\(T([123])\)", text, flags=re.IGNORECASE)
    if m:
        thorax = f"T{m.group(1)}"
        group  = "leg"
        return _apply_overrides(inst, group), thorax

    if re.search(r"\bLegNp(T?[123])?\b", text, flags=re.IGNORECASE):
        # if no digit, fall back to somaNeuromere if present
        thorax = labels.get("somaNeuromere", "")
        thorax = thorax if thorax in ("T1","T2","T3") else ""
        group  = "leg"
        return _apply_overrides(inst, group), thorax

    # Wing tokens
    if re.search(r"\bWing(Np|)|\bTTM\b|\bTTMn\b", text, flags=re.IGNORECASE):
        # T2 by default for wing motors unless neuromere says otherwise
        thorax = labels.get("somaNeuromere", "T2")
        thorax = thorax if thorax in ("T1","T2","T3") else "T2"
        group  = "wing"
        return _apply_overrides(inst, group), thorax

    # Antenna/head
    if re.search(r"\bantenna\b|\bCvN\b", text, flags=re.IGNORECASE):
        return "head", ""

    # Abdomen tokens
    if re.search(r"\bAbN[1-4]\b|\bAbNT\b|\babdomen\b", text, flags=re.IGNORECASE):
        return "abdomen", ""

    # 2) use neuromere for thorax when we already know it’s a motor MN
    if re.search(r"\bmotor\b", text, flags=re.IGNORECASE):
        nm = labels.get("somaNeuromere", "")
        if nm in ("T1","T2","T3"):
            # fall back group based on nerve if we can
            if "Leg" in subc or "Leg" in syst:
                return "leg", nm
            if "Wing" in subc or "Wing" in syst:
                return "wing", nm

    # 3) last-resort: exit nerve hints (weak!)
    if re.search(r"\b(ProLN|MesoLN|MetaLN)\b", exitNerve):
        thorax = dict(ProLN="T1", MesoLN="T2", MetaLN="T3")[re.search(r"(ProLN|MesoLN|MetaLN)", exitNerve).group(1)]
        return "leg", thorax
    if re.search(r"\b(ADMN|PDMN)\b", exitNerve):
        # many wing MNs, but NOT ALWAYS (e.g., MNwm34)
        nm = labels.get("somaNeuromere", "T2")
        thorax = nm if nm in ("T1","T2","T3") else "T2"
        group = _apply_overrides(inst, "wing")
        return group, thorax

    # If nothing matched, return neutral
    return _apply_overrides(inst, ""), labels.get("somaNeuromere", "") if labels.get("somaNeuromere","") in ("T1","T2","T3") else ""


### Step 2: Rename MN type directories to the instance label and write a JSON sidecar

In [ ]:
def sanitize_folder_name(s):
    return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s))

def ensure_label_json(dir_for_id: Path, labels: dict):
    sidecar = dir_for_id / "labels.json"
    with sidecar.open("w") as f:
        json.dump(labels, f, indent=2)
    return sidecar

def rename_mn_dirs_to_instance(mn_root, client, dry_run=True, make_symlink=True):
    """
    Walks MN/<type_like>/<bodyId> and renames <type_like> → <instance>
    (e.g., MNwm34 → MNwm34_PDMNp_L), leaving the bodyId subfolder intact.
    Also writes MN/<instance>/<bodyId>/labels.json with neuPrint labels.
    """
    from pathlib import Path
    import shutil

    mn_root = Path(mn_root).expanduser().resolve()
    todo = []

    for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
        # FIX: use 'p' consistently in the generator below (not 'id_dir')
        for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir() and p.name.isdigit()):
            bid = int(id_dir.name)
            lab = fetch_mn_labels(bid, client)
            if not lab:
                continue
            inst = lab["instance"] or lab["type"] or type_dir.name
            dst_type = sanitize_folder_name(inst)

            # If already named correctly, still ensure labels.json exists
            if dst_type == type_dir.name:
                ensure_label_json(id_dir, lab)
                continue

            src = id_dir
            dst = mn_root / dst_type / id_dir.name
            todo.append((src, dst, type_dir.name, dst_type, lab))

    if not todo:
        print("[rename] Nothing to rename.")
        return

    print(f"[rename] Planned renames: {len(todo)}")
    for src, dst, old, new, _ in todo:
        print(f"  {old}/{src.name}  →  {new}/{dst.name}")

    if dry_run:
        print("[rename] Dry run only. Re-run with dry_run=False to apply.")
        return

    for src, dst, old, new, lab in todo:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        ensure_label_json(dst, lab)

        # remove old type dir if empty
        old_dir = src.parent
        try:
            if not any(old_dir.iterdir()):
                old_dir.rmdir()
        except Exception:
            pass

        # optional symlink from old path to new
        if make_symlink:
            try:
                old_dir.mkdir(parents=True, exist_ok=True)
                link = old_dir / dst.name
                if not link.exists():
                    link.symlink_to(dst)
            except Exception as e:
                print(f"  [warn] symlink failed: {e}")

    print("[rename] Done.")


### Step 3: Smarter CSV enrichment

In [ ]:
# simple mapping from (group, side) → actuator names you showed earlier
WING_ACT = {"left":"wing_pitch_left", "right":"wing_pitch_right"}
HEAD_ACT = "head"
AB_ACT   = "abdomen"

def enrich_mapping_csv(csv_path, client, coarse_leg_defaults=False):
    df = pd.read_csv(csv_path)

    new_rows = []
    for _, r in df.iterrows():
        bid = int(r["mn_id"])
        lab = fetch_mn_labels(bid, client)  # your existing helper: returns dict with instance/type/subclass/exitNerve/...
        if not lab:
            new_rows.append(r)
            continue

        # recompute group/thorax smartly
        group, thorax = smarter_group_from_labels(lab)
        if group:
            r["group"] = group   # optional: keep this column
        if thorax:
            r["thorax"] = thorax

        # fill side from instance suffix if empty
        if not str(r.get("side","")).strip():
            side = "left" if re.search(r"[_-]L\b", (lab.get("instance") or ""), re.IGNORECASE) else \
                   ("right" if re.search(r"[_-]R\b", (lab.get("instance") or ""), re.IGNORECASE) else "")
            r["side"] = side

        # smarter default actuator guesses (only if actuator_name is blank)
        if not str(r.get("actuator_name","")).strip():
            if group == "wing":
                r["joint"], r["dof"], r["actuator_name"] = ("wing","pitch", "wing_pitch_left" if r["side"]=="left" else ("wing_pitch_right" if r["side"]=="right" else ""))
            elif group == "abdomen":
                r["joint"], r["dof"], r["actuator_name"] = ("abdomen","main","abdomen")
            elif group == "head":
                r["joint"], r["dof"], r["actuator_name"] = ("head","main","head")
            elif group == "leg" and coarse_leg_defaults:
                # give you something runnable; you can refine later
                r["joint"], r["dof"] = ("tibia","main")
                if thorax in ("T1","T2","T3") and r["side"] in ("left","right"):
                    r["actuator_name"] = f"tibia_{thorax}_{r['side']}"
                # leave sign blank for manual tuning
        new_rows.append(r)

    out = pd.DataFrame(new_rows, columns=df.columns)
    out.to_csv(csv_path, index=False)
    print(f"[enrich] updated → {csv_path}")


### Step 4: Run mapping

In [ ]:
MN_ROOT = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
CSV_PATH = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_SWC/MN/mn_to_actuator_mapping.csv"

# 1) Rename type folders to full neuPrint `instance` and write labels.json in each ID folder
# rename_mn_dirs_to_instance(MN_ROOT, client, dry_run=False)

# 2) Enrich your CSV with the correct thorax/side/group and easy actuator guesses
enrich_mapping_csv(CSV_PATH, client, coarse_leg_defaults=False)  # set True if you want tibia defaults


In [ ]:
import json, re
import pandas as pd

# ROI → coarse body region + thoracic segment + side
_ROI_PATTERNS = [
    # Legs
    (re.compile(r"LegNp\(T(?P<thorax>[123])\)\((?P<side>L|R)\)"), "leg"),
    # Wings (dorsal mesothoracic nerves)
    (re.compile(r"PDMN\((?P<side>L|R)\)"), "wing"),   # Posterior Dorsal Meso Nerve
    (re.compile(r"ADMN\((?P<side>L|R)\)"), "wing"),   # Anterior Dorsal Meso Nerve
    # Head / antenna via Cervical nerve
    (re.compile(r"CvN\((?P<side>L|R)\)"), "head"),
    # Abdomen nerves (AbN1..4, AbNT)
    (re.compile(r"AbN(?P<abnum>[1-4])\((?P<side>L|R)\)"), "abdomen"),
    (re.compile(r"AbNT"), "abdomen"),
]

def _first_match_roi(roi_keys):
    """
    Given a set of ROI keys (strings), return inferred (group, thorax, side).
    """
    group = thorax = side = ""
    for key in roi_keys:
        for pat, g in _ROI_PATTERNS:
            m = pat.search(key)
            if not m:
                continue
            group = g
            if "thorax" in m.groupdict():
                thorax = f"T{m.group('thorax')}"
            if "side" in m.groupdict():
                side = {"L": "left", "R": "right"}[m.group("side")]
            # once we have a decisive match, stop
            if group in ("leg", "wing", "head", "abdomen"):
                return group, thorax, side
    return group, thorax, side  # possibly empty strings

def _parse_roiinfo_field(val):
    """
    neuPrint 'roiInfo' can be dict-like or a JSON string; we just need its keys.
    Returns a set of ROI names present for this neuron.
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return set()
    if isinstance(val, dict):
        return set(val.keys())
    if isinstance(val, str):
        try:
            obj = json.loads(val)
            if isinstance(obj, dict):
                return set(obj.keys())
        except Exception:
            # Sometimes roiInfo is a comma-separated blob; split as a fallback
            return set(re.findall(r"[A-Za-z0-9()\-+_]+", val))
    return set()

def _coarse_actuator_guess(group, thorax, side):
    """
    Optional guess to make rows immediately runnable.
    Legs remain blank (you’ll map to the exact joint/axis later).
    """
    if group == "wing":
        return "wing_pitch_left"  if side == "left"  else ("wing_pitch_right" if side == "right" else "")
    if group == "abdomen":
        return "abdomen"
    if group == "head":
        return "head"
    return ""  # leg, accessory, unknown → leave blank

def enrich_mapping_with_roi(csv_path, client, backup=True):
    """
    Enrich your mn_to_actuator_mapping.csv using neuPrint fields:
    - instance, type, group, subclass, exitNerve, somaSide
    - roiInfo (to fix leg/wing/head/abdomen + thorax + side)

    Updates columns: group, thorax, side, actuator_name (coarse guess for non-legs).
    """
    from neuprint import fetch_neurons, NeuronCriteria as NC

    df = pd.read_csv(csv_path)
    if backup:
        df.to_csv(csv_path + ".bak", index=False)

    rows_out = []
    filled = 0

    # Iterate row dicts so we can easily mutate and compare
    for row in df.to_dict(orient="records"):
        try:
            bid = int(row.get("mn_id"))
        except Exception:
            rows_out.append(row)
            continue

        # start from existing row; copy so we can compare
        new_row = dict(row)

        # Fetch neuPrint metadata
        meta, _ = fetch_neurons(NC(bodyId=bid), client=client)
        if meta is None or meta.empty:
            rows_out.append(new_row)
            continue

        meta0 = meta.iloc[0]

        # 1) ROI-driven inference
        roi_keys = _parse_roiinfo_field(meta0.get("roiInfo"))
        g_roi, t_roi, s_roi = _first_match_roi(roi_keys)

        # 2) Side fallback: somaSide if ROI didn’t give it
        if not s_roi:
            soma_side = str(meta0.get("somaSide") or "").strip().lower()
            if soma_side in ("left", "right"):
                s_roi = soma_side

        # 3) Group fallback: neuPrint 'group' or 'subclass'
        g_np = str(meta0.get("group") or "").strip().lower()
        if not g_roi:
            if "leg" in g_np:      g_roi = "leg"
            elif "wing" in g_np:   g_roi = "wing"
            elif "head" in g_np:   g_roi = "head"
            elif "abd" in g_np:    g_roi = "abdomen"

        # 4) Special-case known types (e.g., TTMn are leg MNs in T2 via PDMN)
        inst = str(meta0.get("instance") or "")
        typ  = str(meta0.get("type") or "")
        desc = str(meta0.get("description") or "")
        if ("TTMn" in typ) or ("TTMn" in inst) or ("TTMn" in desc):
            # TRUST ROI for side/thorax; if that failed, force leg/T2
            g_roi = g_roi or "leg"
            t_roi = t_roi or "T2"

        # 5) Apply to row (only overwrite if we have something)
        if g_roi: new_row["group"]  = g_roi
        if t_roi: new_row["thorax"] = t_roi
        if s_roi: new_row["side"]   = s_roi

        # 6) Coarse actuator guess for non-leg systems
        if not str(new_row.get("actuator_name","")).strip():
            guess = _coarse_actuator_guess(new_row.get("group",""), new_row.get("thorax",""), new_row.get("side",""))
            if guess:
                new_row["actuator_name"] = guess

        # --- Did anything change? Compare specific columns safely ---
        changed = False
        for col in ("group","thorax","side","actuator_name"):
            old_val = str(row.get(col, "") or "")
            new_val = str(new_row.get(col, "") or "")
            if new_val and (new_val != old_val):
                changed = True

        if changed:
            filled += 1
        rows_out.append(new_row)

    df_out = pd.DataFrame(rows_out, columns=df.columns)
    df_out.to_csv(csv_path, index=False)
    print(f"[roi-enrich] Updated {filled} rows using neuPrint roiInfo/metadata → {csv_path}")
    return df_out


In [ ]:
# CSV_PATH = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_SWC/MN/mn_to_actuator_mapping.csv"
enrich_mapping_with_roi(CSV_PATH, client)  # makes a .bak and rewrites CSV in place


## Step 5: MuJoCo Output

In [ ]:
# ========= NEURON → MuJoCo adapter (no CONFIG/RES dependencies) =========
import os, math, numpy as np, pandas as pd, imageio
from pathlib import Path
import os
# If you're on a laptop / Jupyter, GLFW is usually safest. For headless servers use: os.environ["MUJOCO_GL"]="osmesa"
os.environ.setdefault("MUJOCO_GL", "glfw")
# --- YOU set these three paths ---
RESULT_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")
MAPPING_CSV  = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping.csv")
JCF_XML    = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets/fruitfly.xml")  # <-- this must be an XML
VIDEO_OUT   = RESULT_DIR / "mujoco_run.mp4"

# Optional one-off overrides (your TTM mn IDs → femur actuators)
# MN_OVERRIDES = {
#     10068: "femur_T2_left",
#     10110: "femur_T2_right",
# }

# ---- 1) Load mapping (mn_id → actuator_name, gain, sign, bias) ----
# --- FIX 1: robust mapping loader (drop NaN actuator names) ---
def load_mapping(csv_path, overrides=None):
    df = pd.read_csv(csv_path)

    # keep rows that actually have an actuator name
    if "actuator_name" not in df.columns:
        raise ValueError("mapping CSV must contain an 'actuator_name' column")
    df = df[df["actuator_name"].notna()]                      # <- drop NaN
    df = df[df["actuator_name"].astype(str).str.strip() != ""]# <- drop empty strings

    # normalize columns
    for c, default in (("gain", 1.0), ("bias", 0.0)):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default)
        else:
            df[c] = default

    if "sign" in df.columns:
        def _sgn(x):
            try:
                return 1.0 if float(x) >= 0 else -1.0
            except:
                return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
        df["sign"] = df["sign"].apply(_sgn)
    else:
        df["sign"] = 1.0

    # overrides (e.g., your TTM → femur remap)
    if overrides:
        for bid, name in overrides.items():
            msk = (df["mn_id"].astype(int) == int(bid)) if "mn_id" in df.columns else pd.Series(False, index=df.index)
            if msk.any():
                df.loc[msk, "actuator_name"] = name
            else:
                df = pd.concat([df, pd.DataFrame([{
                    "mn_type": "", "mn_id": int(bid), "actuator_name": name,
                    "gain": 1.0, "bias": 0.0, "sign": 1.0
                }])], ignore_index=True)

    # group by actuator
    by_act = {}
    for _, r in df.iterrows():
        bid = int(r["mn_id"])
        nm  = str(r["actuator_name"]).strip()
        by_act.setdefault(nm, []).append(
            dict(mn_id=bid, gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
        )
    return by_act


# ---- 2) Convert NEURON results → actuator control series ----
def spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
    """Sum alpha-like kernels at each spike. Returns activation(t) on t_grid_ms."""
    a = np.zeros_like(t_grid_ms, dtype=float)
    if len(spike_times_ms) == 0:
        return a
    dt = float(np.median(np.diff(t_grid_ms)))
    tmax = 6.0 * tau_decay
    k_t = np.arange(0.0, tmax + dt, dt)
    k = (1.0 - np.exp(-k_t / tau_rise)) * np.exp(-k_t / tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts))
        j = min(i + len(k), len(a))
        a[i:j] += k[:j-i]
    return a

def build_actuator_controls(result_dir: Path, mapping_by_act,
                            tau_rise=1.0, tau_decay=6.0, scale=1.0,
                            from_voltages=False, v_thresh=0.0):
    spikes_csv   = result_dir / "spike_times.csv"
    voltages_csv = result_dir / "voltages_somas.csv"
    if not voltages_csv.exists():
        raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")
    vdf = pd.read_csv(voltages_csv)
    t_ms = vdf["t_ms"].to_numpy(dtype=float)

    # load spikes (preferred); fallback: detect threshold crossings in soma V
    if (not from_voltages) and spikes_csv.exists():
        sdf = pd.read_csv(spikes_csv)
        if not sdf.empty:
            sdf["neuron_id"] = sdf["neuron_id"].astype(int)
            id2spk = {nid: grp["spike_time_ms"].to_numpy(dtype=float)
                      for nid, grp in sdf.groupby("neuron_id")}
        else:
            id2spk = {}
    else:
        id2spk = {}
        for col in vdf.columns:
            if col == "t_ms": continue
            v = vdf[col].to_numpy(dtype=float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            ts = t_ms[1:][above]
            try: nid = int(col)
            except: continue
            id2spk[nid] = ts

    # for each actuator, sum MN contributions
    act_controls = {}
    for act, sources in mapping_by_act.items():
        acc = np.zeros_like(t_ms, dtype=float)
        for s in sources:
            spikes = id2spk.get(int(s["mn_id"]), np.array([], float))
            a = spikes_to_activation(spikes, t_ms, tau_rise=tau_rise, tau_decay=tau_decay, scale=scale)
            acc += s["sign"] * s["gain"] * a
        # bias as DC offset
        total_bias = sum(s["bias"] for s in sources)
        acc += total_bias
        act_controls[act] = acc
    return t_ms, act_controls

# ---- 3) Run MuJoCo and render video ----
# --- version-agnostic renderer (works with old/new mujoco.Renderer) ---
import os
# pick ONE:
os.environ.setdefault("MUJOCO_GL", "glfw")    # local GPU with display
# os.environ.setdefault("MUJOCO_GL", "osmesa")  # headless CPU render
def run_mujoco(mjcf_path: Path, t_ms, act_controls, video_out: Path,
               sim_hz=1000, render_hz=60, width=768, height=512,
               camera_name=None, slowmo=10.0, render_every=None):
    """
    slowmo: >1 makes playback slower by that factor (video duration *= slowmo)
    render_every: if None, auto; if 1, write a frame every sim step for max frames.
    """
    import os, inspect, numpy as np, imageio, mujoco

    # (keep your MUJOCO_GL env set before importing mujoco)

    if mjcf_path.suffix.lower() not in (".xml", ".mjcf", ".mjb"):
        raise ValueError(f"MJCF/XML expected for MJCF_PATH, got: {mjcf_path}")

    m = mujoco.MjModel.from_xml_path(str(mjcf_path))
    d = mujoco.MjData(m)

    # actuator name map
    name2id = {mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or f"act{i}": i
               for i in range(m.nu)}
    missing = [nm for nm in act_controls.keys() if nm not in name2id]
    if missing:
        print("[mujoco] WARNING: missing actuators in model:", missing)

    # resample NEURON → sim steps
    dt_ms = max(1e-3, 1000.0 * float(m.opt.timestep))
    T = int(np.ceil((t_ms[-1] - t_ms[0]) / dt_ms)) + 1
    times_sim = t_ms[0] + np.arange(T) * dt_ms
    U = np.zeros((m.nu, T), dtype=float)
    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None:
            continue
        U[aid, :] = np.interp(times_sim, t_ms, series)
        lo, hi = m.actuator_ctrlrange[aid]
        if lo < hi:
            U[aid, :] = np.clip(U[aid, :], lo, hi)

    # decide how often to render frames
    if render_every is None:
        # default: similar cadence as before
        nominal = max(1, int(round((1000.0 / render_hz) / dt_ms)))
        render_every = nominal
    out_fps = max(1, int(round(render_hz / max(1e-6, slowmo))))  # lower fps → slower playback

    # Try high-level Renderer first (supports old/new APIs)
    renderer = None
    try:
        renderer = mujoco.Renderer(m, height=height, width=width)
        sig = inspect.signature(renderer.render)
        accepts_data = ("data" in sig.parameters)

        cam_id = -1
        if camera_name:
            try:
                cid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_CAMERA, camera_name)
                if cid >= 0:
                    cam_id = cid
            except Exception:
                pass

        # preview
        mujoco.mj_forward(m, d)
        if accepts_data:
            frame0 = renderer.render(data=d, camera_id=(cam_id if cam_id >= 0 else None))
        else:
            renderer.update_scene(d)
            frame0 = renderer.render()
        video_out.parent.mkdir(parents=True, exist_ok=True)
        imageio.imwrite(str(video_out.with_suffix(".preview.png")), frame0)

        # writer with slow-mo
        writer = imageio.get_writer(str(video_out), fps=out_fps)

        # simulate & render
        for k in range(T):
            d.ctrl[:] = U[:, k]
            mujoco.mj_step(m, d)
            if k % render_every == 0:
                if accepts_data:
                    img = renderer.render(data=d, camera_id=(cam_id if cam_id >= 0 else None))
                else:
                    renderer.update_scene(d)
                    img = renderer.render()
                writer.append_data(img)

        writer.close()
        print(f"[video] wrote {video_out}  (fps={out_fps}, frames≈{T//render_every}, slowmo×{slowmo})")
        return
    except Exception as e:
        print("[mujoco] Renderer path failed, falling back to low-level pipeline:", e)

    # Low-level fallback (Mjv/Mjr)
    scn = mujoco.MjvScene(m, maxgeom=10000)
    con = mujoco.MjrContext(m, mujoco.mjtFontScale.mjFONTSCALE_150)
    opt = mujoco.MjvOption()
    cam = mujoco.MjvCamera()
    if camera_name:
        cid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_CAMERA, camera_name)
        if cid >= 0:
            cam.fixedcamid = cid
            cam.type = mujoco.mjtCamera.mjCAMERA_FIXED

    video_out.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_out), fps=out_fps)
    rect = mujoco.MjrRect(0, 0, width, height)
    rgb = np.zeros((height, width, 3), dtype=np.uint8)

    mujoco.mj_forward(m, d)
    mujoco.mjv_updateScene(m, d, opt, None, cam, mujoco.mjtCatBit.mjCAT_ALL, scn)
    mujoco.mjr_render(rect, scn, con)
    mujoco.mjr_readPixels(rgb, None, rect, con)
    imageio.imwrite(str(video_out.with_suffix(".preview.png")), np.flipud(rgb))

    for k in range(T):
        d.ctrl[:] = U[:, k]
        mujoco.mj_step(m, d)
        if k % render_every == 0:
            mujoco.mjv_updateScene(m, d, opt, None, cam, mujoco.mjtCatBit.mjCAT_ALL, scn)
            mujoco.mjr_render(rect, scn, con)
            mujoco.mjr_readPixels(rgb, None, rect, con)
            writer.append_data(np.flipud(rgb))

    writer.close()
    print(f"[video] wrote {video_out}  (fps={out_fps}, frames≈{T//render_every}, slowmo×{slowmo})")


# # ---- Do it ----
# mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
# t_ms, act_controls = build_actuator_controls(RESULT_DIR, mapping_by_act,
#                                              tau_rise=1.0, tau_decay=6.0, scale=1.0,
#                                              from_voltages=False, v_thresh=0.0)
# run_mujoco(MJCF_XML, t_ms, act_controls, VIDEO_OUT,
#            render_hz=60, slowmo=20.0, render_every=0.05, camera_name="hero")


In [ ]:
# # Confirm mapping has no bogus actuator names:
# print("first mapping actuators:", list(mapping_by_act.keys())[:10])

# # Confirm model actuators (compare names)
# import mujoco
# m = mujoco.MjModel.from_xml_path(str(MJCF_XML))
# print("model actuators:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(m.nu)])

# # If your XML defines cameras, list them and try one explicitly:
# cams = [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_CAMERA, i) for i in range(m.ncam)]
# print("cameras:", cams)


## DIAGNOSTICS

In [ ]:
# ==== ONE-CELL, SELF-CONTAINED DIAGNOSTICS (defines all names it uses) ====
import os, math, numpy as np, pandas as pd

# --- EDIT THESE IF NEEDED ---
RESULT_DIR  = RESULT_DIR if 'RESULT_DIR' in globals() else \
    "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/DNp01_edges/results/demo_run"
MAPPING_CSV = MAPPING_CSV if 'MAPPING_CSV' in globals() else \
    "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv"
MJCF_XML    = MJCF_XML if 'MJCF_XML' in globals() else \
    "/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets/fruitfly.xml"

# AFTER: TTMn drive tibias instead of femurs
MN_OVERRIDES = { 10068: "tibia_T2_left", 10110: "tibia_T2_right" }



# MN_OVERRIDES = {
#     10068: "femur_T2_left",
#     10110: "femur_T2_right",
# }

# ---- helpers (local, no external deps) ----
import pandas as pd, numpy as np

def load_mapping_strict(csv_path, overrides=None):
    df = pd.read_csv(csv_path)

    # Clean actuator_name
    df["actuator_name"] = df.get("actuator_name", "").astype(str).str.strip()
    bad = df["actuator_name"].isin(["", "nan", "None"])
    df = df[~bad].copy()

    # Clean numerics
    for c, default in (("gain", 1.0), ("bias", 0.0)):
        df[c] = pd.to_numeric(df.get(c, default), errors="coerce").fillna(default)

    def _sgn(x):
        try: return 1.0 if float(x) >= 0 else -1.0
        except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
    df["sign"] = df.get("sign", 1.0).apply(_sgn)

    # Optional overrides
    if overrides:
        for bid, name in overrides.items():
            msk = pd.to_numeric(df["mn_id"], errors="coerce").astype("Int64") == int(bid)
            if msk.any():
                df.loc[msk, "actuator_name"] = str(name).strip()
            else:
                df = pd.concat([df, pd.DataFrame([{
                    "mn_type":"", "mn_id":int(bid), "actuator_name":str(name).strip(),
                    "gain":1.0, "bias":0.0, "sign":1.0
                }])], ignore_index=True)

    by_act = {}
    for _, r in df.iterrows():
        nm = str(r["actuator_name"]).strip()
        by_act.setdefault(nm, []).append(
            dict(mn_id=int(r["mn_id"]),
                 gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
        )
    return by_act




def _spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
    a = np.zeros_like(t_grid_ms, dtype=float)
    if len(spike_times_ms) == 0: return a
    dt = float(np.median(np.diff(t_grid_ms)))
    tmax = 6.0*tau_decay
    k_t = np.arange(0.0, tmax+dt, dt)
    k = (1.0 - np.exp(-k_t/tau_rise)) * np.exp(-k_t/tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts))
        j = min(i + len(k), len(a))
        if i < len(a): a[i:j] += k[:j-i]
    return a

def _build_act_controls(result_dir, mapping_by_act, tau_rise=1.0, tau_decay=6.0,
                        scale=1.0, from_voltages=False, v_thresh=0.0):
    result_dir = os.fspath(result_dir)
    vdf = pd.read_csv(os.path.join(result_dir, "voltages_somas.csv"))
    t_ms = vdf["t_ms"].to_numpy(dtype=float)
    id2spk = {}
    spikes_csv = os.path.join(result_dir, "spike_times.csv")
    if (not from_voltages) and os.path.exists(spikes_csv):
        sdf = pd.read_csv(spikes_csv)
        if not sdf.empty:
            sdf["neuron_id"] = sdf["neuron_id"].astype(int)
            id2spk = {nid: grp["spike_time_ms"].to_numpy(dtype=float)
                      for nid, grp in sdf.groupby("neuron_id")}
    if not id2spk:  # fallback from voltages
        for col in vdf.columns:
            if col == "t_ms": continue
            v = vdf[col].to_numpy(dtype=float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            ts = t_ms[1:][above]
            try: nid = int(col)
            except: continue
            id2spk[nid] = ts

    act_controls = {}
    for act, sources in mapping_by_act.items():
        acc = np.zeros_like(t_ms, dtype=float)
        for s in sources:
            spikes = id2spk.get(int(s["mn_id"]), np.array([], float))
            a = _spikes_to_activation(spikes, t_ms, tau_rise, tau_decay, scale=scale)
            acc += s["sign"] * s["gain"] * a
        acc += sum(s["bias"] for s in sources)
        act_controls[act] = acc
    return t_ms, act_controls

def _load_model_acts(xml_path):
    import mujoco
    m = mujoco.MjModel.from_xml_path(str(xml_path))
    names = [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(m.nu)]
    ctrl = [tuple(m.actuator_ctrlrange[i]) for i in range(m.nu)]
    return names, ctrl, m

# ---- run diagnostics ----


model_acts, ctrl_ranges, _m = _load_model_acts(MJCF_XML)
# Rebuild mapping → controls
mapping_by_act = load_mapping_strict(MAPPING_CSV, overrides=MN_OVERRIDES)

# use your existing builder
t_ms, act_controls = _build_act_controls(RESULT_DIR, mapping_by_act,
                                             tau_rise=1.0, tau_decay=6.0, scale=1.0,
                                             from_voltages=False, v_thresh=0.0)

# Safe magnitudes print
print("— Actuator signal magnitudes —")
nonzero_acts = []
for nm, sig in act_controls.items():
    nm_str = str(nm)
    mx = float(np.nanmax(np.abs(sig))) if len(sig) else 0.0
    print(f"{nm_str:24s} max|u|={mx:.6g}")
    if mx > 1e-6:
        nonzero_acts.append(nm_str)
print("nonzero actuators:", nonzero_acts or "NONE")

print("nonzero actuators:", nonzero_acts or "NONE")

print("\n— Mapping ∩ Model —")
mapped = sorted(set(act_controls.keys()))
present = sorted(set(mapped) & set(model_acts))
missing = sorted(set(mapped) - set(model_acts))
print("present:", present)
print("missing:", missing)

# (Optional) show first 10 model actuators + ranges
print("\n— First 10 model actuators & ctrlranges —")
for i,(nm,rg) in enumerate(list(zip(model_acts, ctrl_ranges))[:10]):
    lo, hi = rg
    print(f"{i:2d} {nm:24s} ctrlrange=({lo:.3g}, {hi:.3g})")

# (Optional) inject a visible pulse on a known actuator to prove movement
try:
    import mujoco
    test_act = None
    for candidate in ["abdomen", "femur_T2_left", "femur_T1_left", "wing_pitch_left"]:
        if candidate in model_acts:
            test_act = candidate; break
    if test_act:
        dt = float(np.median(np.diff(t_ms)))
        extra_ms = 1000.0
        nextra = int(round(extra_ms / dt))
        t_ms = np.concatenate([t_ms, t_ms[-1] + np.arange(1, nextra+1)*dt])
        for k in list(act_controls.keys()):
            last = act_controls[k][-1] if len(act_controls[k]) else 0.0
            act_controls[k] = np.concatenate([act_controls[k], np.full(nextra, last)])
        print(f"[pulse] You can now add a visible pulse to '{test_act}' before rendering.")
    else:
        print("[pulse] No familiar actuator found to test.")
except Exception as e:
    print("[pulse] Skipped (mujoco import failed):", e)
# ==== end one-cell diagnostics ====


In [ ]:
# # ========================== VISIBILITY FIX PACK ==========================
# import math, numpy as np, mujoco, imageio
# from pathlib import Path

# # --- 1) Extend signals to a longer window (e.g., 1500 ms) ---
# def extend_signals(t_ms, act_controls, total_ms=1500.0):
#     t0 = float(t_ms[0]); t_end = float(t_ms[-1])
#     if t_end - t0 >= total_ms - 1e-6:
#         return t_ms, act_controls  # already long enough
#     dt = float(np.median(np.diff(t_ms)))
#     new_times = np.arange(t0, t0 + total_ms + 1e-9, dt)
#     out = {}
#     for k, sig in act_controls.items():
#         sig = np.asarray(sig, float)
#         # hold last value (or 0) after original end
#         tail = np.full(len(new_times) - len(t_ms), (sig[-1] if len(sig) else 0.0), float)
#         out[k] = np.concatenate([sig, tail], axis=0)
#     return new_times, out

# # --- 2) Map dimensionless neuron signals → MuJoCo ctrlrange (and amplify) ---
# def remap_to_ctrlrange(act_controls, model, gain=2.0, bias_to_mid=True):
#     """
#     Treat act_controls values as "effort units". We:
#       - multiply by 'gain' to amplify
#       - optionally add DC bias to mid of ctrlrange (helps you see motion from neutral)
#       - clip to ctrlrange
#     """
#     out = {}
#     name2id = {mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i): i for i in range(model.nu)}
#     for nm, series in act_controls.items():
#         if nm not in name2id:
#             # keep as-is (shouldn't happen since you checked presence)
#             out[nm] = np.asarray(series, float)
#             continue
#         aid = name2id[nm]
#         lo, hi = model.actuator_ctrlrange[aid]
#         mid = 0.5*(lo+hi)
#         rng = max(hi-mid, mid-lo, 1e-6)

#         u = np.asarray(series, float) * float(gain)
#         if bias_to_mid:
#             u = mid + u*rng*0.5   # push around mid using half-range scaling
#         # clip to physical range
#         u = np.clip(u, lo, hi)
#         out[nm] = u
#     return out

# # --- 3) Optional: add a visible square pulse to a known actuator to confirm rendering ---
# def add_test_pulse(t_ms, act_controls, actuator_name, model, start_ms=200, dur_ms=400, strength=0.8):
#     """
#     Drive actuator to a large value within ctrlrange for visual confirmation.
#     """
#     if actuator_name not in act_controls:
#         return act_controls  # no-op
#     name2id = {mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i): i for i in range(model.nu)}
#     aid = name2id.get(actuator_name, None)
#     if aid is None:
#         return act_controls

#     lo, hi = model.actuator_ctrlrange[aid]
#     target = lo + strength*(hi-lo)   # e.g., 80% of the range
#     series = act_controls[actuator_name].copy()
#     i0 = int(np.searchsorted(t_ms, start_ms))
#     i1 = int(np.searchsorted(t_ms, start_ms+dur_ms))
#     series[i0:i1] = target
#     act_controls[actuator_name] = series
#     return act_controls

# # --- 4) Re-render with slower playback & longer duration ---
# def render_mujoco_video(mjcf_xml_path, t_ms, act_controls, video_out,
#                         render_hz=30, cam_name="hero",
#                         cam_back_factor=2.0,   # >1 = farther back
#                         cam_fovy_scale=0.8):   # <1 = wider angle
#     import mujoco, math, numpy as np, imageio
#     from pathlib import Path

#     m = mujoco.MjModel.from_xml_path(str(mjcf_xml_path))
#     d = mujoco.MjData(m)

#     # --- find camera id (or None for free-cam) ---
#     cam_id = -1
#     for i in range(m.ncam):
#         if mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_CAMERA, i) == cam_name:
#             cam_id = i; break

#     # --- OPTIONAL: push the model camera back & widen FOV ---
#     if cam_id >= 0:
#         # move along the line from model center to current camera position
#         center = np.array(m.stat.center, dtype=float)
#         pos = m.cam_pos[cam_id].copy()
#         delta = pos - center
#         if np.linalg.norm(delta) > 1e-9:
#             m.cam_pos[cam_id] = center + delta * float(cam_back_factor)
#         # widen the lens (smaller fovy = wider view)
#         m.cam_fovy[cam_id] = float(m.cam_fovy[cam_id]) * float(cam_fovy_scale)

#     # timings
#     dt_ms = 1000.0 * float(m.opt.timestep)
#     T = int(math.ceil((t_ms[-1]-t_ms[0]) / dt_ms)) + 1
#     times_sim = t_ms[0] + np.arange(T)*dt_ms

#     # controls → sim steps
#     name2id = {mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i): i for i in range(m.nu)}
#     U = np.zeros((m.nu, T), float)
#     for nm, series in act_controls.items():
#         if nm in name2id:
#             aid = name2id[nm]
#             U[aid, :] = np.interp(times_sim, t_ms, np.asarray(series, float))

#     # renderer (handles old/new API)
#     try:
#         renderer = mujoco.Renderer(m, 1280, 800)  # new API signature
#         use_new = True
#     except TypeError:
#         renderer = mujoco.Renderer(m)             # old API
#         use_new = False

#     mujoco.mj_forward(m, d)
#     try:
#         frame0 = renderer.render(data=d, camera_id=(cam_id if cam_id >= 0 else None))
#     except TypeError:
#         renderer.update_scene(d)
#         frame0 = renderer.render()
#     imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

#     # write video
#     writer = imageio.get_writer(str(video_out), fps=render_hz)
#     steps_per_frame = max(1, int(round((1000.0/render_hz) / dt_ms)))
#     for k in range(T):
#         d.ctrl[:] = U[:, k]
#         mujoco.mj_step(m, d)
#         if k % steps_per_frame == 0:
#             try:
#                 frame = renderer.render(data=d, camera_id=(cam_id if cam_id >= 0 else None))
#             except TypeError:
#                 renderer.update_scene(d)
#                 frame = renderer.render()
#             writer.append_data(frame)
#     writer.close()
#     print(f"[video] wrote {video_out} (cam='{cam_name}', back×{cam_back_factor}, fovy×{cam_fovy_scale})")


# # ========================== APPLY FIXES & RENDER ==========================
# # 0) Build/extend controls
# t_long, act_controls_long = extend_signals(t_ms, act_controls, total_ms=1500.0)

# # 1) Make a model to compute ctrlranges and remap
# _model = mujoco.MjModel.from_xml_path(str(MJCF_XML))
# act_controls_scaled = remap_to_ctrlrange(act_controls_long, _model, gain=3.0, bias_to_mid=True)

# # 2) (Optional) Add a visible pulse to a known actuator to prove rendering works
# #    Use one that definitely exists in the model; 'abdomen' is safe.
# act_controls_pulsed = add_test_pulse(t_long, act_controls_scaled, actuator_name="abdomen",
#                                      model=_model, start_ms=0, dur_ms=120, strength=0.85)

# # 3) Render (30 fps → ~1.5 seconds on screen, much easier to see)
# render_mujoco_video(
#     MJCF_XML, t_long, act_controls_pulsed,
#     VIDEO_OUT.with_name("mujoco_run_long.mp4"),
#     render_hz=30,
#     cam_name="side",          # or 'side', 'back', etc.
#     cam_back_factor=300.0,      # pull back more
#     cam_fovy_scale=0.7        # widen view a bit more
# )



In [ ]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")

SPIKES_CSV   = RESULT_DIR / "spike_times.csv"
VOLTAGES_CSV = RESULT_DIR / "voltages_somas.csv"
MAPPING_CSV  = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping.csv"

spikes   = pd.read_csv(SPIKES_CSV)
voltages = pd.read_csv(VOLTAGES_CSV)
mapping  = pd.read_csv(MAPPING_CSV)

spike_ids   = spikes["neuron_id"].astype(int).unique()
voltage_ids = [int(c) for c in voltages.columns if c != "t_ms"]
mapped_ids  = mapping["mn_id"].astype(int)

mn_diag = (
    mapping[["mn_id", "mn_type", "instance"]]
    .drop_duplicates()
    .assign(
        in_spikes=lambda df: df["mn_id"].astype(int).isin(spike_ids),
        in_voltages=lambda df: df["mn_id"].astype(int).isin(voltage_ids),
        fired=lambda df: df["mn_id"].astype(int).isin(spike_ids),  # same as in_spikes here
    )
)

print(mn_diag.head(20))
out_mn = RESULT_DIR / "mn_diagnostics.csv"
mn_diag.to_csv(out_mn, index=False)
print(f"\n[Saved] MN-level diagnostics → {out_mn}")


In [ ]:
import pandas as pd
from pathlib import Path

# -----------------------------------------------------
# 1) SET YOUR PATHS HERE
# -----------------------------------------------------
RESULT_DIR = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")

SPIKES_CSV   = RESULT_DIR / "spike_times.csv"
VOLTAGES_CSV = RESULT_DIR / "voltages_somas.csv"

MAPPING_CSV  = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping.csv"

print("SPIKES_CSV   =", SPIKES_CSV, type(SPIKES_CSV))
print("VOLTAGES_CSV =", VOLTAGES_CSV, type(VOLTAGES_CSV))
print("MAPPING_CSV  =", MAPPING_CSV, type(MAPPING_CSV))

# -----------------------------------------------------
# 2) LOAD DATA
# -----------------------------------------------------
spikes   = pd.read_csv(SPIKES_CSV)
voltages = pd.read_csv(VOLTAGES_CSV)
mapping  = pd.read_csv(MAPPING_CSV)

# sanity check for required columns
if "neuron_id" not in spikes.columns:
    raise ValueError("spike_times.csv must contain a 'neuron_id' column.")
if "mn_id" not in mapping.columns or "actuator_name" not in mapping.columns:
    raise ValueError("mn_to_actuator_mapping.csv must contain 'mn_id' and 'actuator_name' columns.")

# -----------------------------------------------------
# 3) BUILD ID SETS
# -----------------------------------------------------
spike_ids   = spikes["neuron_id"].astype(int).unique()
fired_ids   = set(spike_ids)  # anything in spike_times “fired at least once”
voltage_ids = [int(c) for c in voltages.columns if c != "t_ms"]
mapped_ids  = mapping["mn_id"].astype(int).unique()

print("Total mapped MNs:", len(mapped_ids))
print("Total neurons with spikes:", len(spike_ids))
print("Total neurons with voltages:", len(voltage_ids))

# -----------------------------------------------------
# 4) ACTUATOR-BY-ACTUATOR DIAGNOSTICS
# -----------------------------------------------------
rows = []

for act_name, grp in mapping.groupby("actuator_name"):
    if pd.isna(act_name):
        continue

    mn_mapped = set(grp["mn_id"].dropna().astype(int))

    mn_in_spikes   = sorted(mn_mapped & set(spike_ids))
    mn_in_voltages = sorted(mn_mapped & set(voltage_ids))
    mn_fired       = sorted(mn_mapped & fired_ids)

    row = {
        "actuator_name": act_name,
        "n_mn_mapped": len(mn_mapped),
        "n_mn_present_in_spikes": len(mn_in_spikes),
        "n_mn_present_in_voltages": len(mn_in_voltages),
        "n_mn_fired": len(mn_fired),
        "mn_mapped": mn_mapped,
        "mn_present_in_spikes": mn_in_spikes,
        "mn_present_in_voltages": mn_in_voltages,
        "mn_fired": mn_fired,
        "is_active_actuator": len(mn_fired) > 0,
    }
    rows.append(row)

act_diag = pd.DataFrame(rows).sort_values(
    ["is_active_actuator", "n_mn_fired", "n_mn_mapped"],
    ascending=[False, False, False]
)

# -----------------------------------------------------
# 5) PRINT SUMMARY + SAVE TO CSV
# -----------------------------------------------------
print("\n=== Actuator diagnostics summary ===")
print("Total actuators in mapping:", act_diag.shape[0])
print("Actuators with at least one firing MN:",
      act_diag["is_active_actuator"].sum())

print("\nExample rows:")
print(act_diag.head(15)[[
    "actuator_name",
    "n_mn_mapped",
    "n_mn_present_in_spikes",
    "n_mn_fired",
    "is_active_actuator"
]])

out_path = RESULT_DIR / "actuator_diagnostics.csv"
act_diag.to_csv(out_path, index=False)
print(f"\n[Saved] Detailed actuator diagnostics → {out_path}")


In [ ]:
# ===== MN → actuator mapping: full renewal helper =====
from pathlib import Path
import pandas as pd

def renew_mn_to_actuator_mapping(
    base_dir,
    hemi_name="hemi_09A",
    old_map_name="mn_to_actuator_mapping.csv",
    new_map_name="mn_to_actuator_mapping_updated.csv",
    verbose=True,
):
    """
    Rebuild mn_to_actuator_mapping_updated.csv by:
      - Taking all MN rows from mn_diagnostics.csv for a given hemi
      - Outer-merging with the old mapping to carry over existing metadata
      - Ensuring the same columns as the old mapping
    Does NOT auto-guess actuator_name (it preserves whatever you had and leaves new ones blank).
    """
    base_dir   = Path(base_dir)
    mn_folder  = base_dir / "MN"
    hemi_dir   = base_dir / "hemi_runs" / hemi_name

    old_map_csv = mn_folder / old_map_name
    mn_diag_csv = hemi_dir / "mn_diagnostics.csv"
    new_map_csv = mn_folder / new_map_name

    if verbose:
        print(f"[renew] BASE_DIR    = {base_dir}")
        print(f"[renew] MN_FOLDER   = {mn_folder}")
        print(f"[renew] HEMI_DIR    = {hemi_dir}")
        print(f"[renew] OLD_MAP_CSV = {old_map_csv}")
        print(f"[renew] MN_DIAG_CSV = {mn_diag_csv}")
        print(f"[renew] NEW_MAP_CSV = {new_map_csv}")

    # --- Load inputs ---
    old_map = pd.read_csv(old_map_csv)
    mn_diag = pd.read_csv(mn_diag_csv)

    if verbose:
        print(f"[renew] old_map rows: {len(old_map)}  cols: {list(old_map.columns)}")
        print(f"[renew] mn_diag rows: {len(mn_diag)}  cols: {list(mn_diag.columns)}")

    # --- Decide join keys: we want stable identity by mn_id (+ type/instance if present) ---
    join_keys = []
    for k in ["mn_id", "mn_type", "instance"]:
        if k in old_map.columns and k in mn_diag.columns:
            join_keys.append(k)
    if "mn_id" not in join_keys:
        raise ValueError("Need at least 'mn_id' in both old_map and mn_diag to merge.")

    if verbose:
        print(f"[renew] Join keys: {join_keys}")

    # --- Outer-merge so that every MN in mn_diag appears ---
    merged = pd.merge(
        mn_diag,
        old_map,
        on=join_keys,
        how="left",
        suffixes=("", "_old"),
    )

    # If some columns got duplicated as *_old, prefer the un-suffixed ones
    for col in list(merged.columns):
        if col.endswith("_old"):
            base = col[:-4]
            if base in merged.columns:
                merged.drop(columns=[col], inplace=True)

    # --- Ensure we have all columns from old_map ---
    required_cols = list(old_map.columns)
    for col in required_cols:
        if col not in merged.columns:
            merged[col] = pd.NA

    # Reorder columns to match the original mapping
    merged = merged[required_cols]

    # Optional: make sure mn_id is int
    if "mn_id" in merged.columns:
        merged["mn_id"] = merged["mn_id"].astype("Int64")

    # --- Save ---
    mn_folder.mkdir(parents=True, exist_ok=True)
    merged.to_csv(new_map_csv, index=False)

    if verbose:
        print(f"[renew] Wrote {len(merged)} rows → {new_map_csv}")
        print("[renew] You now have a complete mapping table with all MNs in mn_diagnostics.")
        print("        Next step is to fill/auto-fill actuator_name per MN type or instance.")

    return merged
BASE_DIR = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc"

mapping_updated = renew_mn_to_actuator_mapping(
    base_dir=BASE_DIR,
    hemi_name="hemi_09A",                 # adjust if you have multiple hemi runs
    old_map_name="mn_to_actuator_mapping.csv",
    new_map_name="mn_to_actuator_mapping_updated.csv",
)


In [ ]:
# ===== MN → actuator mapping: renew + auto-fill from MJCF =====

from pathlib import Path
import pandas as pd
import re

def renew_mn_to_actuator_mapping(
    base_dir,
    hemi_name="hemi_09A",
    old_map_name="mn_to_actuator_mapping.csv",
    new_map_name="mn_to_actuator_mapping_updated.csv",
    verbose=True,
):
    """
    Rebuild mn_to_actuator_mapping_updated.csv by:
      - Taking all MN rows from mn_diagnostics.csv for a given hemi
      - Outer-merging with the old mapping to carry over existing metadata
      - Ensuring the same columns as the old mapping
    """
    base_dir   = Path(base_dir)
    mn_folder  = base_dir / "MN"
    hemi_dir   = base_dir / "hemi_runs" / hemi_name

    old_map_csv = mn_folder / old_map_name
    mn_diag_csv = hemi_dir / "mn_diagnostics.csv"
    new_map_csv = mn_folder / new_map_name

    if verbose:
        print(f"[renew] BASE_DIR    = {base_dir}")
        print(f"[renew] MN_FOLDER   = {mn_folder}")
        print(f"[renew] HEMI_DIR    = {hemi_dir}")
        print(f"[renew] OLD_MAP_CSV = {old_map_csv}")
        print(f"[renew] MN_DIAG_CSV = {mn_diag_csv}")
        print(f"[renew] NEW_MAP_CSV = {new_map_csv}")

    old_map = pd.read_csv(old_map_csv)
    mn_diag = pd.read_csv(mn_diag_csv)

    if verbose:
        print(f"[renew] old_map rows: {len(old_map)}  cols: {list(old_map.columns)}")
        print(f"[renew] mn_diag rows: {len(mn_diag)}  cols: {list(mn_diag.columns)}")

    # Decide join keys
    join_keys = []
    for k in ["mn_id", "mn_type", "instance"]:
        if k in old_map.columns and k in mn_diag.columns:
            join_keys.append(k)
    if "mn_id" not in join_keys:
        raise ValueError("Need at least 'mn_id' in both old_map and mn_diag to merge.")

    if verbose:
        print(f"[renew] Join keys: {join_keys}")

    merged = pd.merge(
        mn_diag,
        old_map,
        on=join_keys,
        how="left",
        suffixes=("", "_old"),
    )

    # Drop *_old duplicates
    for col in list(merged.columns):
        if col.endswith("_old"):
            base = col[:-4]
            if base in merged.columns:
                merged.drop(columns=[col], inplace=True)

    # Ensure all columns from old_map exist
    required_cols = list(old_map.columns)
    for col in required_cols:
        if col not in merged.columns:
            merged[col] = pd.NA

    merged = merged[required_cols]

    if "mn_id" in merged.columns:
        merged["mn_id"] = merged["mn_id"].astype("Int64")

    mn_folder.mkdir(parents=True, exist_ok=True)
    merged.to_csv(new_map_csv, index=False)

    if verbose:
        print(f"[renew] Wrote {len(merged)} rows → {new_map_csv}")

    return merged, new_map_csv


def _keywords_from_row(row, cols):
    """Extract lowercase alphanumeric chunks from selected columns."""
    kws = []
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            s = str(row[c]).lower()
            # split into tokens (letters/numbers/underscores-ish)
            for token in re.split(r"[^a-z0-9]+", s):
                token = token.strip()
                if token:
                    kws.append(token)
    return kws


def autofill_actuator_names(
    mapping_df,
    xml_path,
    min_score=2,
    only_if_empty=True,
    verbose=True,
):
    """
    For each MN row in mapping_df, try to guess actuator_name from MJCF actuators
    by fuzzy matching metadata:
      - side, thorax, nerve_base, segment_hint, group, Target, mn_type, instance
    Does NOT overwrite existing actuator_name unless only_if_empty=False.
    """
    # Use your existing helper to list actuators
    df_act = list_mjcf_actuators(xml_path)

    if verbose:
        print(f"[auto] Loaded {len(df_act)} actuators from MJCF.")

    # Normalize actuator metadata
    df_act = df_act.copy()
    for c in ["actuator_name", "side", "limb_group", "segment", "axis_hint", "direction_hint"]:
        if c in df_act.columns:
            df_act[c] = df_act[c].astype(str).str.lower()

    act_rows = list(df_act.to_dict("records"))

    def guess_for_row(row):
        # if we keep existing names, skip non-empty ones
        if only_if_empty and pd.notna(row.get("actuator_name")) and str(row["actuator_name"]).strip() != "":
            return row["actuator_name"], 0

        side = str(row.get("side", "")).lower()
        kws = _keywords_from_row(
            row,
            cols=[
                "mn_type",
                "instance",
                "group",
                "Target",
                "segment_hint",
                "nerve_base",
                "thorax",
            ],
        )

        best_name = pd.NA
        best_score = -1

        for a in act_rows:
            score = 0
            # Side match is strong
            if side and side in a.get("side", ""):
                score += 3
            # limb / segment hint matches
            for key in ["limb_group", "segment", "axis_hint", "direction_hint"]:
                val = a.get(key, "")
                for kw in kws:
                    if kw and kw in val:
                        score += 1
            # actuator name substring matches
            aname = a.get("actuator_name", "")
            for kw in kws:
                if kw and kw in aname:
                    score += 1

            if score > best_score:
                best_score = score
                best_name = a.get("actuator_name")

        # Enforce a minimum score so we don't assign junk
        if best_score < min_score:
            return pd.NA, best_score
        return best_name, best_score

    # Apply guessing row-wise
    scores = []
    new_names = []

    for _, row in mapping_df.iterrows():
        name, score = guess_for_row(row)
        new_names.append(name)
        scores.append(score)

    out = mapping_df.copy()
    out["actuator_name_autofill_score"] = scores

    # Only override actuator_name where we actually guessed something
    mask = pd.isna(out["actuator_name"]) | (out["actuator_name"].astype(str).str.strip() == "")
    out.loc[mask, "actuator_name"] = [
        n if pd.notna(n) else out.loc[i, "actuator_name"]
        for i, n in enumerate(new_names)
    ]

    if verbose:
        filled = mask & out["actuator_name"].notna()
        print(f"[auto] Filled actuator_name for {filled.sum()} MN rows (score ≥ {min_score}).")

    return out
BASE_DIR = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc"
HEMI_NAME = "hemi_09A"
MJCF_XML = "/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/build_fruitfly/fruitfly.xml"

# 1) Renew mapping (all MNs from mn_diagnostics included)
mapping_renewed, new_map_csv = renew_mn_to_actuator_mapping(
    base_dir=BASE_DIR,
    hemi_name=HEMI_NAME,
    old_map_name="mn_to_actuator_mapping.csv",
    new_map_name="mn_to_actuator_mapping_updated.csv",
    verbose=True,
)

# 2) Auto-fill actuator_name from MJCF metadata
mapping_autofilled = autofill_actuator_names(
    mapping_renewed,
    xml_path=MJCF_XML,
    min_score=2,           # tighten/loosen if needed
    only_if_empty=True,    # keep any hand-edited names you might already have
    verbose=True,
)

# 3) Save final updated mapping back to the same CSV
final_csv = Path(new_map_csv)
mapping_autofilled.to_csv(final_csv, index=False)
print(f"[final] Wrote auto-filled mapping to: {final_csv}")


In [ ]:
import pandas as pd

mapping = pd.read_csv("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping_updated.csv")

print(mapping[["mn_id", "mn_type", "side", "group", "actuator_name"]].head(20))
print("Non-empty actuator_name rows:", mapping["actuator_name"].notna().sum())


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# ===== Paths (edit BASE to your setup) =====
BASE = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc")

SPIKES_CSV   = BASE / "hemi_runs" / "hemi_09A" / "spike_times.csv"
VOLTAGES_CSV = BASE / "hemi_runs" / "hemi_09A" / "voltages_somas.csv"
MAP_CSV      = BASE / "MN" / "mn_to_actuator_mapping_updated.csv"

print("SPIKES_CSV  :", SPIKES_CSV)
print("VOLTAGES_CSV:", VOLTAGES_CSV)
print("MAP_CSV     :", MAP_CSV)

# ===== Load data =====
spikes   = pd.read_csv(SPIKES_CSV)
voltages = pd.read_csv(VOLTAGES_CSV)
mapping  = pd.read_csv(MAP_CSV)

# ----- IDs from spikes -----
if "mn_id" in spikes.columns:
    spike_ids = spikes["mn_id"].astype(int).unique()
elif "neuron_id" in spikes.columns:
    spike_ids = spikes["neuron_id"].astype(int).unique()
else:
    raise ValueError("spike_times.csv must have 'neuron_id' or 'mn_id' column.")

# ----- IDs from voltages (columns except t_ms) -----
voltage_ids = np.array([int(c) for c in voltages.columns if c != "t_ms"])

# ----- IDs from mapping -----
if "mn_id" not in mapping.columns:
    raise ValueError("mn_to_actuator_mapping_updated.csv must contain 'mn_id' column.")

mapping["mn_id"] = mapping["mn_id"].astype(int)

spike_mn   = set(spike_ids)
voltage_mn = set(voltage_ids)
mapped_mn  = set(mapping["mn_id"].unique())

# ===== Build per-MN diagnostic for all simulated neurons =====
all_sim_ids = sorted(spike_mn | voltage_mn)
rows = []

for mn_id in all_sim_ids:
    has_spikes  = mn_id in spike_mn
    has_voltage = mn_id in voltage_mn
    rows_for_id = mapping[mapping["mn_id"] == mn_id]

    if len(rows_for_id) == 0:
        # neuron appears in sim but has no mapping row
        rows.append({
            "mn_id": mn_id,
            "has_spikes": has_spikes,
            "has_voltage": has_voltage,
            "mapped": False,
            "mn_type": None,
            "side": None,
            "group": None,
            "actuator_name": None,
        })
    else:
        # one or more mapping rows (e.g. multiple actuators)
        for _, r in rows_for_id.iterrows():
            rows.append({
                "mn_id": mn_id,
                "has_spikes": has_spikes,
                "has_voltage": has_voltage,
                "mapped": True,
                "mn_type": r.get("mn_type"),
                "side": r.get("side"),
                "group": r.get("group"),
                "actuator_name": r.get("actuator_name"),
            })

mn_act_diag = pd.DataFrame(rows)

# ===== Summary =====
wired = mn_act_diag[mn_act_diag["actuator_name"].notna()]
unmapped_sim = mn_act_diag[~mn_act_diag["mapped"]]
no_actuator = mn_act_diag[mn_act_diag["mapped"] & mn_act_diag["actuator_name"].isna()]

print("=========================================")
print("  MOTOR-NEURON → ACTUATOR DIAGNOSTIC")
print("=========================================")
print(f"Total unique simulated MNs (spikes ∪ voltages): {len(all_sim_ids)}")
print(f"Sim MNs that have mapping rows:                 {len(all_sim_ids) - len(unmapped_sim['mn_id'].unique())}")
print(f"Sim MNs mapped with actuator_name set:          {len(wired['mn_id'].unique())}")
print(f"Sim MNs with mapping row but no actuator_name:  {len(no_actuator['mn_id'].unique())}")
print()

# ===== 1) Firing MNs mapped to actuators (the ones that actually drive MuJoCo) =====
firing_wired = wired[wired["has_spikes"]]

print("=== FIRING MNs → ACTUATORS (WIRED) ===")
if firing_wired.empty:
    print("No firing MNs currently mapped to actuators.")
else:
    for _, r in firing_wired.sort_values(["actuator_name", "mn_id"]).iterrows():
        print(
            f"MN {r.mn_id:<6} type={r.mn_type} side={r.side} "
            f"-> actuator='{r.actuator_name}' "
            f"(spikes={r.has_spikes}, voltage={r.has_voltage})"
        )
print("\nTotal firing MNs mapped to actuators:", firing_wired['mn_id'].nunique())

# ===== 2) Firing MNs with NO mapping row at all =====
unmapped_firing_ids = sorted(spike_mn - mapped_mn)
print("\n=== FIRING MNs WITH NO MAPPING ROW (CRITICAL) ===")
if not unmapped_firing_ids:
    print("All firing MNs are present in the mapping table. ✔")
else:
    print(f"{len(unmapped_firing_ids)} firing MNs have NO mapping rows at all.")
    print("Example IDs:", unmapped_firing_ids[:25])

# ===== 3) Firing MNs that have mapping rows but actuator_name is NaN =====
mapped_rows_with_spikes = mn_act_diag[mn_act_diag["has_spikes"]]
unwired_firing = mapped_rows_with_spikes[mapped_rows_with_spikes["actuator_name"].isna()]

print("\n=== FIRING MNs WITH MAPPING ROW BUT NO actuator_name (NEED WIRING) ===")
if unwired_firing.empty:
    print("All mapped firing MNs have actuator_name assigned. ✔")
else:
    for _, r in unwired_firing.sort_values("mn_id").iterrows():
        print(
            f"MN {r.mn_id:<6} type={r.mn_type} side={r.side} "
            f"-> actuator_name=NaN (spikes={r.has_spikes}, voltage={r.has_voltage})"
        )

# ===== 4) Which actuators will actually move (have ≥1 firing MN) =====
print("\n=== ACTUATORS WITH ≥1 FIRING MN (CAN MOVE) ===")
act_counts = firing_wired.groupby("actuator_name").size().sort_values(ascending=False)

if act_counts.empty:
    print("⚠ No actuators currently receive input from firing MNs → MuJoCo will not move.")
else:
    for act, count in act_counts.items():
        print(f"{act}: {count} firing MN(s) mapped")

# ===== 5) Leg-wired MNs (actuators containing coxa/femur/tibia/tarsus) =====
leg_mask = firing_wired["actuator_name"].fillna("").str.contains(
    "coxa_|femur_|tibia_|tarsus_", case=False, regex=True
)
leg_wired = firing_wired[leg_mask]

print("\n=== LEG-WIRED FIRING MNs (LIKELY CONTROLLING LEGS) ===")
if leg_wired.empty:
    print("No firing MNs are currently mapped to obvious leg actuators.")
else:
    for _, r in leg_wired.sort_values(["actuator_name", "mn_id"]).iterrows():
        print(
            f"MN {r.mn_id:<6} type={r.mn_type} side={r.side} "
            f"-> leg actuator='{r.actuator_name}'"
        )

# ===== 6) Save full per-MN diagnostic =====
OUT = BASE / "MN" / "mn_to_actuator_diagnostic.csv"
mn_act_diag.to_csv(OUT, index=False)
print(f"\nFull per-MN diagnostic saved to:\n{OUT}")


In [ ]:
# ========= NEURON → MuJoCo using the SAME env style as your notebook =========
# Drives env.physics from NEURON spike→actuator series, with live camera control.

import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd, imageio

# --- paths you already have ---
RESULT_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/DNp01_edges/results/demo_run")
MAPPING_CSV  = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv")
FLY_ASSETS   = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets")
WORLD_XML    = FLY_ASSETS / "floor.xml"      # includes fruitfly.xml
VIDEO_OUT    = RESULT_DIR / "mujoco_run_env.mp4"

# Optional: where 'neuro_adapter' lives (so we can import the same helpers your notebook uses)
NEURO_FLY_SRC = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly")
if str(NEURO_FLY_SRC) not in sys.path:
    sys.path.insert(0, str(NEURO_FLY_SRC))

# Render backend: GLFW for on-screen; use "osmesa" if headless
os.environ.setdefault("MUJOCO_GL", "glfw")

# -------------------- mapping & NEURON→controls --------------------
MN_OVERRIDES = { 10068: "femur_T2_left", 10110: "femur_T2_right" }

def load_mapping(csv_path, overrides=None):
    df = pd.read_csv(csv_path)
    if "actuator_name" not in df.columns:
        raise ValueError("mapping CSV must contain 'actuator_name'")
    df = df[df["actuator_name"].notna()]
    df = df[df["actuator_name"].astype(str).str.strip() != ""]
    for c, default in (("gain", 1.0), ("bias", 0.0)):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default) if c in df.columns else default
    if "sign" in df.columns:
        def _sgn(x):
            try: return 1.0 if float(x) >= 0 else -1.0
            except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
        df["sign"] = df["sign"].apply(_sgn)
    else:
        df["sign"] = 1.0
    if overrides:
        for bid, name in overrides.items():
            msk = (df["mn_id"].astype(int) == int(bid)) if "mn_id" in df.columns else pd.Series(False, index=df.index)
            if msk.any():
                df.loc[msk, "actuator_name"] = name
            else:
                df = pd.concat([df, pd.DataFrame([{
                    "mn_type": "", "mn_id": int(bid), "actuator_name": name,
                    "gain": 1.0, "bias": 0.0, "sign": 1.0
                }])], ignore_index=True)
    by_act = {}
    for _, r in df.iterrows():
        by_act.setdefault(str(r["actuator_name"]).strip(), []).append(
            dict(mn_id=int(r["mn_id"]), gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
        )
    return by_act

def spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
    a = np.zeros_like(t_grid_ms, dtype=float)
    if len(spike_times_ms) == 0:
        return a
    dt = float(np.median(np.diff(t_grid_ms)))
    tmax = 6.0 * tau_decay
    k_t = np.arange(0.0, tmax + dt, dt)
    k = (1.0 - np.exp(-k_t / tau_rise)) * np.exp(-k_t / tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts))
        j = min(i + len(k), len(a))
        a[i:j] += k[:j-i]
    return a

def build_actuator_controls(result_dir: Path, mapping_by_act,
                            tau_rise=1.0, tau_decay=6.0, scale=1.0,
                            from_voltages=False, v_thresh=0.0):
    spikes_csv   = result_dir / "spike_times.csv"
    voltages_csv = result_dir / "voltages_somas.csv"
    if not voltages_csv.exists():
        raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")
    vdf = pd.read_csv(voltages_csv)
    t_ms = vdf["t_ms"].to_numpy(dtype=float)

    if (not from_voltages) and spikes_csv.exists():
        sdf = pd.read_csv(spikes_csv)
        if not sdf.empty:
            sdf["neuron_id"] = sdf["neuron_id"].astype(int)
            id2spk = {nid: grp["spike_time_ms"].to_numpy(dtype=float)
                      for nid, grp in sdf.groupby("neuron_id")}
        else:
            id2spk = {}
    else:
        id2spk = {}
        for col in vdf.columns:
            if col == "t_ms": continue
            v = vdf[col].to_numpy(dtype=float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            ts = t_ms[1:][above]
            try: nid = int(col)
            except: continue
            id2spk[nid] = ts

    act_controls = {}
    for act, sources in mapping_by_act.items():
        acc = np.zeros_like(t_ms, dtype=float)
        for s in sources:
            spikes = id2spk.get(int(s["mn_id"]), np.array([], float))
            a = spikes_to_activation(spikes, t_ms, tau_rise=tau_rise, tau_decay=tau_decay, scale=scale)
            acc += s["sign"] * s["gain"] * a
        acc += sum(s["bias"] for s in sources)
        act_controls[act] = acc
    return t_ms, act_controls

# -------------------- helper: extend & scale signals --------------------
def extend_signals(t_ms, act_controls, total_ms=120.0):
    t0 = float(t_ms[0]); dt = float(np.median(np.diff(t_ms)))
    new_times = np.arange(t0, total_ms + 1e-9, dt)
    if new_times.shape[0] <= len(t_ms):
        return t_ms, act_controls
    out = {}
    for k, sig in act_controls.items():
        sig = np.asarray(sig, float)
        tail = np.full(len(new_times) - len(t_ms), (sig[-1] if len(sig) else 0.0), float)
        out[k] = np.concatenate([sig, tail], axis=0)
    return new_times, out

# -------------------- ENV BUILDER (do it like your notebook) --------------------
def build_env_from_xml(world_xml: Path):
    """
    Tries to build the same style env you used in the notebook:
    1) neuro_adapter.env_loader.load_env (if available)
    2) flybody.fly_envs generic loader (if available)
    3) fallback: dm_control mujoco.Physics wrapped in a SimpleEnv
    """
    # 1) neuro_adapter
    try:
        from neuro_adapter.env_loader import load_env
        env = load_env(xml_path=str(world_xml))
        return env
    except Exception:
        pass
    # 2) flybody fallback (simple world loader)
    try:
        import flybody.fly_envs as fb
        # try a generic world loader if present
        if hasattr(fb, "xml_world"):
            env = fb.xml_world(str(world_xml))
            return env
    except Exception:
        pass
    # 3) final fallback: dm_control physics, wrapped to look like env
    from dm_control import mujoco as dm_mujoco
    physics = dm_mujoco.Physics.from_xml_path(str(world_xml))
    class SimpleEnv:
        def __init__(self, physics):
            self.physics = physics
        def reset(self): self.physics.reset(); return None
        def step(self, *_): self.physics.step(); return None
        def control_timestep(self): return float(self.physics.timestep())
        # dm_control-style action_spec if someone asks:
        class _Spec:
            def __init__(self, lo, hi): self.minimum, self.maximum = lo, hi
        def action_spec(self):
            m = self.physics.model
            lo = m.actuator_ctrlrange[:,0] if m.actuator_ctrlrange.size else np.full((m.nu,), -1.0)
            hi = m.actuator_ctrlrange[:,1] if m.actuator_ctrlrange.size else np.full((m.nu,),  1.0)
            return SimpleEnv._Spec(lo, hi)
    return SimpleEnv(physics)

# -------------------- camera control on the LIVE env.model --------------------
def set_env_camera(env, cam_name="side", back_factor=3.0, fovy_scale=0.8):
    ph = env.physics
    m = ph.model
    
    # find camera id
    cam_id = -1
    for i in range(m.ncam):
        nm = (m.id2name(i, 'camera') or "")
        if nm == cam_name:
            cam_id = i; break
    if cam_id < 0:
        print(f"[camera] '{cam_name}' not found; keeping default")
        return cam_name  # return name anyway so render uses same string
    
    # move camera & widen FOV
    center = np.array(m.stat.center, dtype=float)
    pos = m.cam_pos[cam_id].copy()
    delta = pos - center
    if np.linalg.norm(delta) > 1e-9:
        m.cam_pos[cam_id] = center + delta * float(back_factor)
    m.cam_fovy[cam_id] = float(m.cam_fovy[cam_id]) * float(fovy_scale)
    
    # APPLY: forward + reset cached dm_control render contexts
    ph.forward()
    try:
        ph._reset_contexts()   # <-- critical for dm_control
    except AttributeError:
        pass
    
    print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.2f}")
    return cam_name  # return the name to use in render()


# -------------------- option 2: BOOST actuator strength (gear/forcerange) ----
def boost_actuator_strength_on_env(env, targets=None, gear_mult=3.0, forcerange_mult=2.0):
    m = env.physics.model
    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}
    chosen = (targets or list(name2id.keys()))
    for nm in chosen:
        if nm not in name2id: continue
        aid = name2id[nm]
        if m.actuator_gear.shape[1] >= 1:
            m.actuator_gear[aid, 0] *= float(gear_mult)
        lo, hi = m.actuator_forcerange[aid]
        if lo < hi:
            span = (hi - lo) * float(forcerange_mult)
            mid = 0.5 * (hi + lo)
            m.actuator_forcerange[aid, 0] = mid - 0.5 * span
            m.actuator_forcerange[aid, 1] = mid + 0.5 * span


def zoom_out(env, cam_name="side", back_factor=2.0, fovy_add_deg=15.0):
    ph = env.physics
    m = ph.model

    cam_id = -1
    for i in range(m.ncam):
        nm = (m.id2name(i, 'camera') or "")
        if nm == cam_name:
            cam_id = i; break
    if cam_id < 0:
        print(f"[zoom] camera '{cam_name}' not found")
        return cam_name

    center = np.array(m.stat.center, dtype=float)
    pos = m.cam_pos[cam_id].copy()
    delta = pos - center
    if np.linalg.norm(delta) > 1e-9 and back_factor != 1.0:
        m.cam_pos[cam_id] = center + delta * float(back_factor)

    m.cam_fovy[cam_id] = float(m.cam_fovy[cam_id]) + float(fovy_add_deg)
    m.cam_fovy[cam_id] = np.clip(m.cam_fovy[cam_id], 10.0, 150.0)

    ph.forward()
    try:
        ph._reset_contexts()
    except AttributeError:
        pass

    print(f"[zoom] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.1f}°")
    return cam_name

def set_camera_zoom(env,
                    cam_name: str = "side",
                    *,
                    distance_abs: float | None = None,   # absolute distance in model units
                    distance_factor: float | None = 8.0, # OR factor×model.stat.extent
                    fovy_deg: float | None = 70.0,       # absolute FOV in degrees
                    along_current: bool = True):         # move along the camera's current direction
    """
    Set a named (fixed) camera to a *non-compounding*, zoomed-out view.
    - If distance_abs is given, use it.
    - Else if distance_factor is given, use distance = distance_factor * model.stat.extent.
    - If fovy_deg is given, set absolute FOV (clamped to [10, 150] deg).
    - 'along_current=True' keeps the camera pointing the same way, just farther.
    """
    ph = env.physics
    m = ph.model

    # find camera id
    cam_id = -1
    for i in range(m.ncam):
        if (m.id2name(i, 'camera') or "") == cam_name:
            cam_id = i; break
    if cam_id < 0:
        print(f"[camera] '{cam_name}' not found"); 
        return cam_name  # fall back to name

    center = np.array(m.stat.center, dtype=float)
    pos    = m.cam_pos[cam_id].copy()

    # pick direction to back away
    v = pos - center
    if (not along_current) or (np.linalg.norm(v) < 1e-9):
        # stable default direction if current vector is degenerate
        v = np.array([-1.0, 0.0, 1.0], dtype=float)
    v /= (np.linalg.norm(v) + 1e-12)

    # choose distance
    if distance_abs is not None:
        dist = float(distance_abs)
    else:
        if distance_factor is None:
            distance_factor = 8.0   # sensible default
        dist = float(m.stat.extent) * float(distance_factor)

    # set camera position & FOV *absolutely* (no compounding)
    m.cam_pos[cam_id] = center + v * dist
    if fovy_deg is not None:
        m.cam_fovy[cam_id] = float(np.clip(fovy_deg, 10.0, 150.0))

    # apply and refresh dm_control render contexts
    ph.forward()
    try: ph._reset_contexts()
    except AttributeError: pass

    print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.2f}")
    return cam_name


def set_all_cameras_zoom(env,
                         distance_factor: float = 8.0,
                         fovy_deg: float = 70.0,
                         skip_tracking: bool = True):
    """
    Apply the same zoom to *every* camera (useful if you switch cams).
    If skip_tracking=True, we skip cameras whose names suggest tracking (e.g. 'track1/2/3').
    """
    ph = env.physics
    m = ph.model
    names = [(m.id2name(i, 'camera') or "") for i in range(m.ncam)]
    for nm in names:
        low = nm.lower()
        if skip_tracking and ("track" in low):
            continue
        set_camera_zoom(env, nm, distance_factor=distance_factor, fovy_deg=fovy_deg)




# -------------------- render using env.physics (exactly like the notebook loop) ---
def render_env_with_controls(env, t_ms, act_controls, video_out,
                             cam_name="side", back_factor=3.0, fovy_scale=0.8,
                             slowmo=8.0, H=800, W=1280,
                             boost_targets=("femur_T2_left","femur_T2_right"),
                             gear_mult=28.0, forcerange_mult=10.0):
    """
    Feeds actuator controls into env.physics each step and renders frames via env.physics.render.
    Slowmo>1 increases video duration (more frames per unit sim time).
    """
    ph = env.physics
    m  = ph.model

    # choose/modify camera on LIVE model (so it actually shows up)
    cam_id = set_env_camera(env, cam_name=cam_name, back_factor=back_factor, fovy_scale=fovy_scale)

    # boost actuator strength on the env model
    boost_actuator_strength_on_env(env, targets=list(boost_targets) if boost_targets else None,
                                   gear_mult=gear_mult, forcerange_mult=forcerange_mult)

    # map actuator names → ids
    name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
    # sim timing
    dt = float(ph.timestep())                     # seconds
    dt_ms = 1000.0 * dt
    T = int(math.ceil((t_ms[-1] - t_ms[0]) / dt_ms)) + 1
    times_sim = t_ms[0] + np.arange(T) * dt_ms

    # controls resampled to sim steps
    U = np.zeros((m.nu, T), dtype=float)
    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None: continue
        U[aid, :] = np.interp(times_sim, t_ms, np.asarray(series, float))
        lo, hi = m.actuator_ctrlrange[aid]
        if lo < hi:
            U[aid, :] = np.clip(U[aid, :], lo, hi)

    # slow motion: write more frames per sim time
    render_hz = 30
    base_steps_per_frame = max(1, int(round((1.0 / render_hz) / dt)))
    steps_per_frame = max(1, int(round(base_steps_per_frame / max(1.0, slowmo))))
    out_fps = render_hz

    # writer
    video_out.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_out), fps=out_fps)

    # reset and preview
    try:
        env.reset()
    except Exception:
        ph.reset()
    # one preview frame
    frame0 = ph.render(height=H, width=W, camera_id=cam_name)
    imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

    # simulate & render
    frame_count = 0
    for k in range(T):
        ph.data.ctrl[:] = U[:, k]
        ph.step()  # advance physics (like your notebook loop)
        if k % steps_per_frame == 0:
            frame = ph.render(height=H, width=W, camera_id=cam_name)
            writer.append_data(frame)
            frame_count += 1

    writer.close()
    est_secs = frame_count / out_fps
    print(f"[video] wrote {video_out}  frames={frame_count}  fps={out_fps}  slowmo×{slowmo}  (~{est_secs:.1f}s)")

# -------------------- run everything --------------------
# 1) mapping + controls from your NEURON run folder
mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
t_ms, act_controls = build_actuator_controls(
    RESULT_DIR, mapping_by_act,
    tau_rise=1.0, tau_decay=6.0, scale=1.0,
    from_voltages=False, v_thresh=0.0
)

# 2) stretch signals so you can see behavior
t_long, act_controls_long = extend_signals(t_ms, act_controls, total_ms=300.0)

# 3) build env the same way your notebook does (env.physics present)

env = build_env_from_xml(WORLD_XML)

# pick which camera to render with
CAM = "track1"   # or "back", "hero", etc.

# apply absolute zoom so it stays fixed & wide
# Option A: set one camera
set_camera_zoom(env, CAM, distance_factor=10.0, fovy_deg=75.0)

# (optional) Option B: make *all* fixed cams wide in case you switch later
# set_all_cameras_zoom(env, distance_factor=10.0, fovy_deg=75.0)

# IMPORTANT: do not re-scale cameras again in the renderer.
render_env_with_controls(
    env, t_long, act_controls_long, VIDEO_OUT,
    cam_name=CAM,
    back_factor=1.0,      # keep 1.0 so we don't compound the zoom
    fovy_scale=1.0,       # keep 1.0 so we don't compound the zoom
    slowmo=8.0,
    boost_targets=("femur_T2_left","femur_T2_right"),
    gear_mult=27.95, forcerange_mult=10.0
)



In [ ]:
# # ========= NEURON → MuJoCo (notebook-style env) : COMPACT + KNOBS =========
# import os, sys, math
# from pathlib import Path
# import numpy as np, pandas as pd, imageio

# # -------------------- PATHS --------------------
# RESULT_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")
# MAPPING_CSV = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping_updated.csv")
# FLY_ASSETS   = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets")
# WORLD_XML    = FLY_ASSETS / "floor.xml"      # includes fruitfly.xml
# VIDEO_OUT    = RESULT_DIR / "mujoco_run_env.mp4"

# # Optional (so `import neuro_adapter` etc. works)
# NEURO_FLY_SRC = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly")
# if str(NEURO_FLY_SRC) not in sys.path:
#     sys.path.insert(0, str(NEURO_FLY_SRC))

# os.environ.setdefault("MUJOCO_GL", "glfw")

# # -------------------- TOP-LEVEL KNOBS --------------------
# # Input → activation
# USE_SPIKES        = True        # True: use spike_times.csv; False: detect crossings from voltages_somas.csv
# V_THRESH_MV       = -20.0       # used if USE_SPIKES=False
# TAU_RISE_MS       = 0.8         # spike→activation kernel (ms)
# TAU_DECAY_MS      = 6.0
# KERNEL_SCALE      = 1.0         # multiply kernel
# POST_SMOOTH_WIN   = 0           # moving-average window (samples) after summing (0 = off)

# # Time & visibility
# EXTEND_TOTAL_MS   = 120.0       # extend signals so motion is visible
# SLOWMO_FACTOR     = 60.0         # >1 = slower video (writes more frames)

# # Control remap into MuJoCo ctrlrange
# CTRL_GAIN         = 8.0         # amplifies dimensionless activation→control
# CTRL_BIAS_TO_MID  = False        # add mid-bias so you see motion around neutral

# # Camera (absolute zoom, non-compounding)
# CAM_NAME          = "track2"    # "hero", "side", "back", "track1", ...
# CAM_DIST_FACTOR   = 10.0        # distance = model.stat.extent * this factor
# CAM_FOVY_DEG      = 75.0        # absolute FOV in degrees

# # Actuator strength (T2 extensor chain both sides)
# GEAR_MULT         = 2.5         # modest gear multiplier (too large → unstable)
# FORCERANGE_MULT   = 10.0        # widen force caps so gear isn't just saturating
# BOOST_TARGETS     = (
#     "coxa_T2_left","femur_T2_left","tibia_T2_left",
#     "coxa_T2_right","femur_T2_right","tibia_T2_right",
# )

# # Loss/traction knobs (reduce losses along the chain and improve ground grip)
# LOSS_DAMPING_MULT = 0.45        # <1 reduces damping on selected DOFs
# LOSS_ARMATURE_MULT= 0.5         # <1 reduces armature on selected DOFs
# LEG_JOINTS        = ("coxa_T2_left","femur_T2_left","tibia_T2_left",
#                      "coxa_T2_right","femur_T2_right","tibia_T2_right")
# FLOOR_FRICTION    = (1.8, 0.005, 0.002)   # (slide, spin, roll) for floor geom

# # Explosive push knobs (multiplicative burst on selected actuators)
# ADD_EXPLOSIVE_BURST = True
# BURST_T0_MS         = None       # None = auto (first nonzero); or a number
# BURST_WINDOW_MS     = (0.0, 45.0)  # relative window [offset, offset+dur] from t0
# BURST_GAIN          = 10.0        # 4..10 typical

# # Optional debug pulse to guarantee a visible push
# ADD_JUMP_PULSE    = False       # keep False if using ADD_EXPLOSIVE_BURST
# PULSE_MS_START    = 0.0
# PULSE_MS_DUR      = 120.0
# PULSE_STRENGTH    = 0.9         # 0..1, pushes toward ctrl upper end during pulse window

# # -------------------- MAP: TTMn → full T2 extensor chain --------------------
# def build_ttmn_overrides(mn_left=10068, mn_right=10110):
#     return {
#         int(mn_left):  ["coxa_T2_left","femur_T2_left","tibia_T2_left"],
#         int(mn_right): ["coxa_T2_right","femur_T2_right","tibia_T2_right"],
#     }
# MN_OVERRIDES = build_ttmn_overrides()

# # -------------------- CSV mapping → {actuator:[sources]} --------------------
# def load_mapping(csv_path, overrides=None):
#     df = pd.read_csv(csv_path)
#     if "actuator_name" not in df.columns: raise ValueError("mapping CSV must contain 'actuator_name'")
#     df = df[df["actuator_name"].astype(str).str.strip().ne("")]
#     for c, default in (("gain",1.0),("bias",0.0)):
#         df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default) if c in df.columns else default
#     df["sign"] = (df["sign"] if "sign" in df.columns else 1.0)
#     def _sgn(x):
#         try: return 1.0 if float(x)>=0 else -1.0
#         except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
#     df["sign"] = df["sign"].apply(_sgn)

#     # apply overrides (replace those MNs, then add one row per target actuator)
#     if overrides:
#         try:
#             ids = set(int(k) for k in overrides.keys())
#             if "mn_id" in df.columns: df = df[~df["mn_id"].astype(int).isin(ids)].copy()
#         except: pass
#         add = []
#         for bid, names in overrides.items():
#             names = names if isinstance(names,(list,tuple)) else [names]
#             for name in names:
#                 add.append({"mn_type":"","mn_id":int(bid),"actuator_name":str(name),"gain":1.0,"bias":0.0,"sign":1.0})
#         if add: df = pd.concat([df, pd.DataFrame(add)], ignore_index=True)

#     by_act = {}
#     for _, r in df.iterrows():
#         by_act.setdefault(str(r["actuator_name"]).strip(), []).append(
#             dict(mn_id=int(r["mn_id"]), gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
#         )
#     return by_act

# # -------------------- spikes/voltages → activation series --------------------
# def spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
#     a = np.zeros_like(t_grid_ms, float)
#     if len(spike_times_ms)==0: return a
#     dt = float(np.median(np.diff(t_grid_ms))); tmax = 6.0*tau_decay
#     k_t = np.arange(0.0, tmax+dt, dt)
#     k = (1.0 - np.exp(-k_t/tau_rise)) * np.exp(-k_t/tau_decay) * scale
#     for ts in spike_times_ms:
#         i = int(np.searchsorted(t_grid_ms, ts)); j = min(i+len(k), len(a))
#         a[i:j] += k[:j-i]
#     return a

# def build_actuator_controls(result_dir: Path, mapping_by_act,
#                             use_spikes=True, v_thresh=0.0,
#                             tau_rise=1.0, tau_decay=6.0, scale=1.0,
#                             post_smooth_win=0):
#     spikes_csv, volts_csv = result_dir/"spike_times.csv", result_dir/"voltages_somas.csv"
#     if not volts_csv.exists(): raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")
#     vdf = pd.read_csv(volts_csv); t_ms = vdf["t_ms"].to_numpy(float)

#     if use_spikes and spikes_csv.exists():
#         sdf = pd.read_csv(spikes_csv)
#         id2spk = {int(nid): grp["spike_time_ms"].to_numpy(float) for nid, grp in sdf.groupby("neuron_id")} if not sdf.empty else {}
#     else:
#         id2spk = {}
#         for col in vdf.columns:
#             if col=="t_ms": continue
#             v = vdf[col].to_numpy(float)
#             above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
#             id2spk[int(col)] = t_ms[1:][above]

#     acts = {}
#     for act, sources in mapping_by_act.items():
#         acc = np.zeros_like(t_ms, float)
#         for s in sources:
#             a = spikes_to_activation(id2spk.get(int(s["mn_id"]), np.array([],float)), t_ms,
#                                      tau_rise=tau_rise, tau_decay=tau_decay, scale=scale)
#             acc += s["sign"]*s["gain"]*a
#         acc += sum(s["bias"] for s in sources)
#         if post_smooth_win and post_smooth_win>1:
#             w = int(post_smooth_win); acc = np.convolve(acc, np.ones(w)/w, mode="same")
#         acts[act] = acc
#     return t_ms, acts

# # -------------------- time stretch + ctrl remap --------------------
# def extend_signals(t_ms, act_controls, total_ms=150.0):
#     t0 = float(t_ms[0]); dt = float(np.median(np.diff(t_ms)))
#     new_t = np.arange(t0, total_ms+1e-9, dt)
#     if len(new_t) <= len(t_ms): return t_ms, act_controls
#     out = {}
#     for k, sig in act_controls.items():
#         sig = np.asarray(sig, float)
#         tail = np.full(len(new_t)-len(sig), (sig[-1] if len(sig) else 0.0), float)
#         out[k] = np.concatenate([sig, tail])
#     return new_t, out

# def remap_to_ctrlrange(act_controls, env, gain=3.0, bias_to_mid=True):
#     m = env.physics.model; out = {}
#     name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
#     for nm, series in act_controls.items():
#         aid = name2id.get(nm)
#         if aid is None: out[nm] = np.asarray(series, float); continue
#         lo, hi = m.actuator_ctrlrange[aid]; mid = 0.5*(lo+hi); rng = max(hi-mid, mid-lo, 1e-6)
#         u = np.asarray(series, float) * float(gain)
#         if bias_to_mid: u = mid + 0.5*rng*u
#         out[nm] = np.clip(u, lo, hi) if lo<hi else u
#     return out

# # -------------------- env build (notebook-style) --------------------
# def build_env_from_xml(world_xml: Path):
#     try:
#         from neuro_adapter.env_loader import load_env
#         return load_env(xml_path=str(world_xml))
#     except Exception: pass
#     try:
#         import flybody.fly_envs as fb
#         if hasattr(fb,"xml_world"): return fb.xml_world(str(world_xml))
#     except Exception: pass
#     from dm_control import mujoco as dm_mujoco
#     physics = dm_mujoco.Physics.from_xml_path(str(world_xml))
#     class SimpleEnv:
#         def __init__(self, physics): self.physics=physics
#         def reset(self): self.physics.reset()
#         def step(self,*_): self.physics.step()
#         def control_timestep(self): return float(self.physics.timestep())
#         class _Spec: 
#             def __init__(self,lo,hi): self.minimum,self.maximum=lo,hi
#         def action_spec(self):
#             m=self.physics.model
#             lo=m.actuator_ctrlrange[:,0] if m.actuator_ctrlrange.size else np.full((m.nu,),-1.0)
#             hi=m.actuator_ctrlrange[:,1] if m.actuator_ctrlrange.size else np.full((m.nu,), 1.0)
#             return SimpleEnv._Spec(lo,hi)
#     return SimpleEnv(physics)

# # -------------------- camera: absolute zoom (non-compounding) --------------------
# def set_camera_zoom(env, cam_name="track2", distance_factor=8.0, fovy_deg=70.0):
#     ph = env.physics; m = ph.model
#     cam_id = next((i for i in range(m.ncam) if (m.id2name(i,'camera') or "")==cam_name), -1)
#     if cam_id<0: print(f"[camera] '{cam_name}' not found"); return cam_name
#     center = np.array(m.stat.center, float)
#     v = m.cam_pos[cam_id].copy() - center
#     if np.linalg.norm(v)<1e-9: v = np.array([-1.0,0.0,1.0], float)
#     v /= (np.linalg.norm(v)+1e-12)
#     dist = float(m.stat.extent)*float(distance_factor)
#     m.cam_pos[cam_id] = center + v*dist
#     m.cam_fovy[cam_id] = float(np.clip(fovy_deg, 10.0, 150.0))
#     ph.forward()
#     try: ph._reset_contexts()
#     except AttributeError: pass
#     print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.1f}")
#     return cam_name

# # -------------------- actuator strength (cap first, then gear) --------------------
# def boost_actuator_strength_on_env(env, targets, gear_mult=2.5, forcerange_mult=12.0):
#     m = env.physics.model; name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
#     changed=[]
#     for nm in targets:
#         aid = name2id.get(nm)
#         if aid is None: continue
#         lo,hi = m.actuator_forcerange[aid]
#         if lo<hi:
#             span=(hi-lo)*float(forcerange_mult); mid=0.5*(hi+lo)
#             m.actuator_forcerange[aid,0]=mid-0.5*span; m.actuator_forcerange[aid,1]=mid+0.5*span
#         if m.actuator_gear.shape[1]>=1: m.actuator_gear[aid,0]*=float(gear_mult)
#         changed.append(nm)
#     env.physics.forward()
#     try: env.physics._reset_contexts()
#     except AttributeError: pass
#     print("[boost] actuators:", changed)

# # -------------------- losses & traction --------------------
# def ease_leg_losses(env, joint_names, damping_mult=0.5, armature_mult=0.6):
#     m = env.physics.model
#     jname2id = {m.id2name(i,'joint'): i for i in range(m.njnt)}
#     changed=[]
#     for jn in joint_names:
#         jid = jname2id.get(jn)
#         if jid is None: continue
#         dof = int(m.jnt_dofadr[jid])
#         if 0 <= dof < m.dof_damping.size:
#             m.dof_damping[dof] *= float(damping_mult)
#         if 0 <= dof < m.dof_armature.size:
#             m.dof_armature[dof] *= float(armature_mult)
#         changed.append(jn)
#     env.physics.forward()
#     print("[loss] adjusted joints:", changed)

# def improve_floor_grip(env, friction=(1.5, 0.005, 0.002)):
#     m = env.physics.model; changed=False
#     for i in range(m.ngeom):
#         nm = (m.id2name(i,'geom') or "").lower()
#         if 'floor' in nm or 'ground' in nm:
#             m.geom_friction[i,:] = friction
#             changed=True
#     if changed: env.physics.forward()
#     print(f"[floor] friction set to {friction}")

# # -------------------- optional: explosive burst & debug pulse --------------------
# def add_explosive_burst(t_ms, act_controls, targets, t0_ms=None,
#                         burst_ms=(10.0, 45.0), burst_gain=6.0):
#     if not targets: return act_controls
#     t = np.asarray(t_ms, float)
#     # auto start
#     if t0_ms is None:
#         starts=[]
#         for nm in targets:
#             u = np.asarray(act_controls.get(nm, np.zeros_like(t)))
#             nz = np.where(np.abs(u) > (0.02*np.nanmax(np.abs(u))+1e-9))[0]
#             if len(nz): starts.append(t[nz[0]])
#         t0_ms = (min(starts) if starts else t[0])
#     start = t0_ms + float(burst_ms[0])
#     end   = t0_ms + float(burst_ms[1])
#     w = (t >= start) & (t <= end)
#     out = {k: np.array(v, float, copy=True) for k,v in act_controls.items()}
#     for nm in targets:
#         if nm in out: out[nm][w] *= float(burst_gain)
#     return out

# def add_jump_pulse(t_ms, act_controls, targets, start_ms=0.0, dur_ms=120.0, strength=0.9):
#     from copy import deepcopy
#     out = deepcopy(act_controls)
#     i0 = int(np.searchsorted(t_ms, start_ms)); i1 = int(np.searchsorted(t_ms, start_ms+dur_ms))
#     for nm in targets:
#         if nm not in out: continue
#         x = np.asarray(out[nm], float); lo, hi = np.nanmin(x), np.nanmax(x)
#         x[i0:i1] = (1.0-strength)*lo + strength*hi
#         out[nm] = x
#     return out

# # -------------------- render (env.physics loop, no camera tweaks here) --------------------
# def render_env_with_controls(env, t_ms, act_controls, video_out,
#                              cam_name="back", slowmo=20.0, H=1920, W=1280, render_hz=120):
#     ph = env.physics; m = ph.model
#     # resample controls to sim steps
#     dt = float(ph.timestep()); dt_ms = 1000.0*dt
#     T = int(math.ceil((t_ms[-1]-t_ms[0]) / dt_ms)) + 1
#     times_sim = t_ms[0] + np.arange(T)*dt_ms
#     name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
#     U = np.zeros((m.nu, T), float)
#     for nm, series in act_controls.items():
#         aid = name2id.get(nm)
#         if aid is None: continue
#         U[aid,:] = np.interp(times_sim, t_ms, np.asarray(series,float))
#         lo,hi = m.actuator_ctrlrange[aid]
#         if lo<hi: U[aid,:] = np.clip(U[aid,:], lo, hi)

#     video_out.parent.mkdir(parents=True, exist_ok=True)
#     writer = imageio.get_writer(str(video_out), fps=render_hz)
#     try: env.reset()
#     except Exception: ph.reset()
#     frame0 = ph.render(height=H, width=W, camera_id=cam_name)
#     imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

#     base_steps_per_frame = max(1, int(round((1.0/render_hz)/dt)))
#     steps_per_frame = max(1, int(round(base_steps_per_frame/max(1.0, slowmo))))
#     frames=0
#     for k in range(T):
#         ph.data.ctrl[:] = U[:,k]; ph.step()
#         if k % steps_per_frame == 0:
#             writer.append_data(ph.render(height=H, width=W, camera_id=cam_name)); frames+=1
#     writer.close()
#     print(f"[video] {video_out}  frames={frames}  fps={render_hz}  slowmo×{slowmo}  (~{frames/render_hz:.1f}s)")

# # ===================== RUN =====================
# # 1) mapping + controls (choose spikes vs voltages)
# mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
# t_ms, act_controls = build_actuator_controls(
#     RESULT_DIR, mapping_by_act,
#     use_spikes=USE_SPIKES, v_thresh=V_THRESH_MV,
#     tau_rise=TAU_RISE_MS, tau_decay=TAU_DECAY_MS, scale=KERNEL_SCALE,
#     post_smooth_win=POST_SMOOTH_WIN
# )

# # 2) extend window for visibility
# t_long, act_controls_long = extend_signals(t_ms, act_controls, total_ms=EXTEND_TOTAL_MS)

# # 3) env + traction/loss + boosts + camera
# env = build_env_from_xml(WORLD_XML)
# improve_floor_grip(env, friction=FLOOR_FRICTION)
# ease_leg_losses(env, LEG_JOINTS, damping_mult=LOSS_DAMPING_MULT, armature_mult=LOSS_ARMATURE_MULT)
# boost_actuator_strength_on_env(env, BOOST_TARGETS, gear_mult=GEAR_MULT, forcerange_mult=FORCERANGE_MULT)
# CAM = set_camera_zoom(env, cam_name=CAM_NAME, distance_factor=CAM_DIST_FACTOR, fovy_deg=CAM_FOVY_DEG)

# # 4) (optional) add explosive burst OR debug pulse on the *dimensionless* signals
# act_controls_mod = dict(act_controls_long)
# if ADD_EXPLOSIVE_BURST:
#     act_controls_mod = add_explosive_burst(
#         t_long, act_controls_mod, BOOST_TARGETS,
#         t0_ms=BURST_T0_MS, burst_ms=BURST_WINDOW_MS, burst_gain=BURST_GAIN
#     )
# if ADD_JUMP_PULSE:
#     act_controls_mod = add_jump_pulse(
#         t_long, act_controls_mod, BOOST_TARGETS,
#         start_ms=PULSE_MS_START, dur_ms=PULSE_MS_DUR, strength=PULSE_STRENGTH
#     )

# # 5) remap to ctrlrange (use env model so ranges are exact)
# act_controls_scaled = remap_to_ctrlrange(act_controls_mod, env, gain=CTRL_GAIN, bias_to_mid=CTRL_BIAS_TO_MID)

# # 6) render
# render_env_with_controls(env, t_long, act_controls_scaled, VIDEO_OUT,
#                          cam_name=CAM, slowmo=SLOWMO_FACTOR, H=1920, W=1280, render_hz=120)


In [ ]:
# ========= NEURON → MuJoCo (notebook-style env) : COMPACT + KNOBS =========
import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd, imageio

# -------------------- PATHS --------------------
RESULT_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")
MAPPING_CSV = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping_phased.csv")
FLY_ASSETS   = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets")
WORLD_XML    = FLY_ASSETS / "floor.xml"      # includes fruitfly.xml
VIDEO_OUT    = RESULT_DIR / "mujoco_run_env.mp4"
ROLES_CSV = Path(
    "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_muscle_roles_from_neuprint_minimal_reclassified.csv"
)

# Optional (so `import neuro_adapter` etc. works)
NEURO_FLY_SRC = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly")
if str(NEURO_FLY_SRC) not in sys.path:
    sys.path.insert(0, str(NEURO_FLY_SRC))

os.environ.setdefault("MUJOCO_GL", "glfw")

# -------------------- TOP-LEVEL KNOBS --------------------
# Input → activation
USE_SPIKES        = True        # True: use spike_times.csv; False: detect crossings from voltages_somas.csv
V_THRESH_MV       = -20.0       # used if USE_SPIKES=False
TAU_RISE_MS       = 0.8         # spike→activation kernel (ms)
TAU_DECAY_MS      = 6.0
KERNEL_SCALE      = 3.0         # multiply kernel
POST_SMOOTH_WIN   = 0           # moving-average window (samples) after summing (0 = off)

# Time & visibility
EXTEND_TOTAL_MS   = 200.0       # longer window so motion is clearly visible
SLOWMO_FACTOR     = 60.0        # >1 = slower video (more frames per unit time)

# Control remap into MuJoCo ctrlrange
CTRL_GAIN         = 8.0        # was 8.0: strongly amplify dimensionless activation→control
CTRL_BIAS_TO_MID  = True        # center signals around mid ctrlrange so joints actually swing

# Camera (absolute zoom, non-compounding)
CAM_NAME          = "track2"    # "hero", "side", "back", "track1", ...
CAM_DIST_FACTOR   = 10.0        # distance = model.stat.extent * this factor
CAM_FOVY_DEG      = 75.0        # absolute FOV in degrees

# Actuator strength (legs; all thoracic segments left & right)
# These are strong debug values so you can *see* motion. Dial back later for realism.
GEAR_MULT       = 4.0   # was 12.0
FORCERANGE_MULT = 20.0  # was 50.0




# Leg actuators we want to boost (names will be canonicalized to the model, case-insensitive).
# If some of these don’t exist in the XML, they’ll just be skipped with a warning.
BOOST_TARGETS     = (
    # T1 (forelegs)
    "coxa_T1_left","femur_T1_left","tibia_T1_left",
    "coxa_T1_right","femur_T1_right","tibia_T1_right",

    # T2 (midlegs)
    "coxa_T2_left","femur_T2_left","tibia_T2_left",
    "coxa_T2_right","femur_T2_right","tibia_T2_right",

    # T3 (hindlegs)
    "coxa_T3_left","femur_T3_left","tibia_T3_left",
    "coxa_T3_right","femur_T3_right","tibia_T3_right",
)

# Loss/traction knobs (make contacts LESS bouncy & joints more damped)
LOSS_DAMPING_MULT  = 2.0   # >1 → more damping
LOSS_ARMATURE_MULT = 2.0   # >1 → more joint inertia

LEG_JOINTS         = (
    "coxa_T1_left","femur_T1_left","tibia_T1_left",
    "coxa_T2_left","femur_T2_left","tibia_T2_left",
    "coxa_T3_left","femur_T3_left","tibia_T3_left",
    "coxa_T1_right","femur_T1_right","tibia_T1_right",
    "coxa_T2_right","femur_T2_right","tibia_T2_right",
    "coxa_T3_right","femur_T3_right","tibia_T3_right",
)
FLOOR_FRICTION     = (1.8, 0.005, 0.002)   # (slide, spin, roll) for floor geom

# Explosive push knobs (we’ll disable for debugging and rely on jump pulse)
ADD_EXPLOSIVE_BURST = False      # off for now while we debug base mapping/strength
BURST_T0_MS         = None       # None = auto (first nonzero); or a number
BURST_WINDOW_MS     = (0.0, 45.0)
BURST_GAIN          = 10.0       # unused while ADD_EXPLOSIVE_BURST=False

# Optional debug pulse to guarantee a visible push across boosted legs
ADD_JUMP_PULSE    = False         # force a big pulse to see if legs can move at all
PULSE_MS_START    = 0.0
PULSE_MS_DUR      = 200.0        # over the full EXTEND_TOTAL_MS window
PULSE_STRENGTH  = 0.7   # was 1.0


# ---------- Time scaling knobs ----------
# Biological sim window (ms) you care about
SIM_WINDOW_MS      = 30.0      # the MN activation lives in ~0–30 ms

# How many times to repeat that 30 ms pattern
N_LOOPS            = 10        # e.g. 10 cycles → 300 ms total physical time

# Total physical ms the actuator signals should cover (for looping)
EXTEND_TOTAL_MS    = SIM_WINDOW_MS * N_LOOPS   # 30 * 10 = 300 ms

# How long you want the *whole* extended window to appear in the video (seconds)
TARGET_VIDEO_SEC   = 30.0      # ~30 second video for 300 ms of motion

# Slow-motion factor so that:
#   video_duration ≈ SLOWMO_FACTOR * (EXTEND_TOTAL_MS / 1000)
SLOWMO_FACTOR      = TARGET_VIDEO_SEC / (EXTEND_TOTAL_MS / 1000.0)
# → 30.0 / 0.3 = 100.0

print(f"[time] SIM_WINDOW_MS={SIM_WINDOW_MS} ms, "
      f"N_LOOPS={N_LOOPS}, "
      f"EXTEND_TOTAL_MS={EXTEND_TOTAL_MS} ms, "
      f"TARGET_VIDEO_SEC={TARGET_VIDEO_SEC} s, "
      f"SLOWMO_FACTOR={SLOWMO_FACTOR:.1f}")




# -------------------- MAP: TTMn → full T2 extensor chain --------------------
def build_ttmn_overrides(mn_left=10068, mn_right=10110):
    return {
        int(mn_left):  ["coxa_T2_left","femur_T2_left","tibia_T2_left"],
        int(mn_right): ["coxa_T2_right","femur_T2_right","tibia_T2_right"],
    }
MN_OVERRIDES = build_ttmn_overrides()

# -------------------- CSV mapping → {actuator:[sources]} (with muscle roles) --------------------
def load_mapping(csv_path, overrides=None, roles_csv=ROLES_CSV):
    # base mapping
    df = pd.read_csv(csv_path)
    if "actuator_name" not in df.columns:
        raise ValueError("mapping CSV must contain 'actuator_name'")

    # drop empty actuator_name rows
    df = df[df["actuator_name"].astype(str).str.strip().ne("")].copy()

    # normalize numeric fields
    for c, default in (("gain", 1.0), ("bias", 0.0)):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default)
        else:
            df[c] = default

    # sign: keep column if present, otherwise 1.0
    if "sign" in df.columns:
        df["sign"] = df["sign"]
    else:
        df["sign"] = 1.0

    def _sgn(x):
        try:
            return 1.0 if float(x) >= 0 else -1.0
        except Exception:
            s = str(x).strip()
            if s in ("-1", "-", "minus"):
                return -1.0
            return 1.0

    df["sign"] = df["sign"].apply(_sgn)

    # ensure mn_id is numeric (important for merge)
    df["mn_id"] = pd.to_numeric(df["mn_id"], errors="coerce").astype("Int64")

    # ---- merge neuPrint muscle roles ----
    if roles_csv is not None and roles_csv.exists():
        roles = pd.read_csv(roles_csv)
        # normalize column name
        if "bodyId" in roles.columns and "mn_id" not in roles.columns:
            roles = roles.rename(columns={"bodyId": "mn_id"})
        roles["mn_id"] = pd.to_numeric(roles["mn_id"], errors="coerce").astype("Int64")

        if "muscle_role" not in roles.columns:
            raise ValueError(f"{roles_csv} must contain 'muscle_role'")

        roles = roles[["mn_id", "muscle_role"]].drop_duplicates()

        df = df.merge(roles, on="mn_id", how="left")
        # default anything missing to "extensor"
        df["muscle_role"] = df["muscle_role"].fillna("extensor").astype(str).str.lower()

        print("\n[roles] coverage in mapping (after merge with neuPrint):")
        print(df["muscle_role"].value_counts(dropna=False))
    else:
        # fallback: everything is treated as extensor if no roles file
        df["muscle_role"] = "extensor"
        print("\n[roles] WARNING: roles CSV not found; defaulting all MNs to 'extensor'.")

    # apply overrides (TTMn → full T2 chain, etc.)
    if overrides:
        try:
            ids = set(int(k) for k in overrides.keys())
            if "mn_id" in df.columns:
                df = df[~df["mn_id"].astype(int).isin(ids)].copy()
        except Exception:
            pass

        add = []
        for bid, names in overrides.items():
            names = names if isinstance(names, (list, tuple)) else [names]
            for name in names:
                add.append({
                    "mn_type": "",
                    "mn_id": int(bid),
                    "actuator_name": str(name),
                    "gain": 1.0,
                    "bias": 0.0,
                    "sign": 1.0,
                    "muscle_role": "extensor",  # TTMn override: treat as extensor-ish push
                })
        if add:
            df = pd.concat([df, pd.DataFrame(add)], ignore_index=True)

    # build actuator -> list of MN sources, now including role
    by_act = {}
    for _, r in df.iterrows():
        act_name = str(r["actuator_name"]).strip()
        by_act.setdefault(act_name, []).append(
            dict(
                mn_id=int(r["mn_id"]),
                gain=float(r["gain"]),
                bias=float(r["bias"]),
                sign=float(r["sign"]),
                role=str(r["muscle_role"]).lower(),
            )
        )

    return by_act

# -------------------- spikes/voltages → activation series (role-aware) --------------------
def spikes_to_activation(spike_times_ms, t_grid_ms,
                         tau_rise=1.0, tau_decay=6.0, scale=1.0):
    a = np.zeros_like(t_grid_ms, float)
    if len(spike_times_ms) == 0:
        return a
    dt = float(np.median(np.diff(t_grid_ms)))
    tmax = 6.0 * tau_decay
    k_t = np.arange(0.0, tmax + dt, dt)
    k = (1.0 - np.exp(-k_t / tau_rise)) * np.exp(-k_t / tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts))
        j = min(i + len(k), len(a))
        a[i:j] += k[:j - i]
    return a


def build_actuator_controls(result_dir: Path, mapping_by_act,
                            use_spikes=True, v_thresh=0.0,
                            tau_rise=1.0, tau_decay=6.0, scale=1.0,
                            post_smooth_win=0):
    """Build actuator control signals from MN spikes/voltages, using neuPrint roles.

    For each actuator:
        net_signal = sum(extensor-like MN activations) - sum(flexor-like MN activations)
    """
    spikes_csv = result_dir / "spike_times.csv"
    volts_csv  = result_dir / "voltages_somas.csv"

    if not volts_csv.exists():
        raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")

    vdf = pd.read_csv(volts_csv)
    t_ms = vdf["t_ms"].to_numpy(float)

    # --- gather spike times per neuron ---
    if use_spikes and spikes_csv.exists():
        sdf = pd.read_csv(spikes_csv)
        if sdf.empty:
            id2spk = {}
        else:
            id2spk = {
                int(nid): grp["spike_time_ms"].to_numpy(float)
                for nid, grp in sdf.groupby("neuron_id")
            }
    else:
        # derive from voltage crossings
        id2spk = {}
        for col in vdf.columns:
            if col == "t_ms":
                continue
            v = vdf[col].to_numpy(float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            id2spk[int(col)] = t_ms[1:][above]

    # role → sign mapping
    FLEX_ROLES = {"flexor", "depressor", "retractor"}
    EXT_ROLES  = {"extensor", "levator", "protractor"}

    acts = {}
    debug_rows = []

    for act, sources in mapping_by_act.items():
        acc_ext = np.zeros_like(t_ms, float)
        acc_flex = np.zeros_like(t_ms, float)
        base_bias = 0.0

        for s in sources:
            mn_id = int(s["mn_id"])
            role = str(s.get("role", "extensor")).lower()
            g = float(s.get("gain", 1.0))
            b = float(s.get("bias", 0.0))

            spikes = id2spk.get(mn_id, np.array([], float))
            a = spikes_to_activation(spikes, t_ms,
                                     tau_rise=tau_rise,
                                     tau_decay=tau_decay,
                                     scale=scale)

            if role in FLEX_ROLES:
                acc_flex += g * a
            elif role in EXT_ROLES:
                acc_ext += g * a
            else:
                # unknown role: use sign as a fallback
                sgn = float(s.get("sign", 1.0))
                if sgn >= 0:
                    acc_ext += g * a
                else:
                    acc_flex += g * a

            base_bias += b

        net = acc_ext - acc_flex + base_bias

        # optional smoothing
        if post_smooth_win and post_smooth_win > 1:
            w = int(post_smooth_win)
            net = np.convolve(net, np.ones(w) / w, mode="same")

        acts[act] = net

        # collect debug info
        n_ext = sum(1 for s in sources if str(s.get("role", "")).lower() in EXT_ROLES)
        n_flex = sum(1 for s in sources if str(s.get("role", "")).lower() in FLEX_ROLES)
        debug_rows.append({
            "actuator": act,
            "n_sources": len(sources),
            "n_extensor_like": n_ext,
            "n_flexor_like": n_flex,
        })

    # print per-actuator role summary for sanity
    dbg_df = pd.DataFrame(debug_rows).sort_values("actuator")
    print("\n[role-debug] per-actuator MN pool:")
    print(dbg_df.to_string(index=False))

    # Also save to CSV next to result_dir
    dbg_out = result_dir / "actuator_role_pool_summary.csv"
    dbg_df.to_csv(dbg_out, index=False)
    print(f"[role-debug] saved → {dbg_out}")

    return t_ms, acts


# -------------------- time stretch + ctrl remap --------------------
def extend_signals(t_ms, act_controls, total_ms=150.0):
    t0 = float(t_ms[0])
    dt = float(np.median(np.diff(t_ms)))
    new_t = np.arange(t0, total_ms + 1e-9, dt)
    if len(new_t) <= len(t_ms):
        return t_ms, act_controls
    out = {}
    for k, sig in act_controls.items():
        sig = np.asarray(sig, float)
        tail = np.full(len(new_t) - len(sig),
                       (sig[-1] if len(sig) else 0.0), float)
        out[k] = np.concatenate([sig, tail])
    return new_t, out

def remap_to_ctrlrange(act_controls, env, gain=3.0, bias_to_mid=True):
    m = env.physics.model
    out = {}
    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}
    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None:
            # actuator name not in model → skip
            continue
        lo, hi = m.actuator_ctrlrange[aid]
        mid = 0.5 * (lo + hi)
        rng = max(hi - mid, mid - lo, 1e-6)
        u = np.asarray(series, float) * float(gain)
        if bias_to_mid:
            u = mid + 0.5 * rng * u
        out[nm] = np.clip(u, lo, hi) if lo < hi else u
    return out

# -------------------- env build (notebook-style) --------------------
def build_env_from_xml(world_xml: Path):
    try:
        from neuro_adapter.env_loader import load_env
        return load_env(xml_path=str(world_xml))
    except Exception:
        pass
    try:
        import flybody.fly_envs as fb
        if hasattr(fb, "xml_world"):
            return fb.xml_world(str(world_xml))
    except Exception:
        pass

    from dm_control import mujoco as dm_mujoco
    physics = dm_mujoco.Physics.from_xml_path(str(world_xml))

    class SimpleEnv:
        def __init__(self, physics):
            self.physics = physics
        def reset(self):
            self.physics.reset()
        def step(self, *_):
            self.physics.step()
        def control_timestep(self):
            return float(self.physics.timestep())
        class _Spec:
            def __init__(self, lo, hi):
                self.minimum, self.maximum = lo, hi
        def action_spec(self):
            m = self.physics.model
            if m.actuator_ctrlrange.size:
                lo = m.actuator_ctrlrange[:, 0]
                hi = m.actuator_ctrlrange[:, 1]
            else:
                lo = np.full((m.nu,), -1.0)
                hi = np.full((m.nu,),  1.0)
            return SimpleEnv._Spec(lo, hi)

    return SimpleEnv(physics)

# -------------------- camera: absolute zoom (non-compounding) --------------------
def set_camera_zoom(env, cam_name="track2",
                    distance_factor=8.0, fovy_deg=70.0):
    ph = env.physics
    m = ph.model
    cam_id = next(
        (i for i in range(m.ncam)
         if (m.id2name(i, 'camera') or "") == cam_name),
        -1
    )
    if cam_id < 0:
        print(f"[camera] '{cam_name}' not found")
        return cam_name
    center = np.array(m.stat.center, float)
    v = m.cam_pos[cam_id].copy() - center
    if np.linalg.norm(v) < 1e-9:
        v = np.array([-1.0, 0.0, 1.0], float)
    v /= (np.linalg.norm(v) + 1e-12)
    dist = float(m.stat.extent) * float(distance_factor)
    m.cam_pos[cam_id] = center + v * dist
    m.cam_fovy[cam_id] = float(np.clip(fovy_deg, 10.0, 150.0))
    ph.forward()
    try:
        ph._reset_contexts()
    except AttributeError:
        pass
    print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.1f}")
    return cam_name

# -------------------- actuator strength (cap first, then gear) --------------------
def boost_actuator_strength_on_env(env, targets,
                                   gear_mult=2.5,
                                   forcerange_mult=12.0):
    m = env.physics.model
    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}
    changed = []
    for nm in targets:
        aid = name2id.get(nm)
        if aid is None:
            continue
        lo, hi = m.actuator_forcerange[aid]
        if lo < hi:
            span = (hi - lo) * float(forcerange_mult)
            mid = 0.5 * (hi + lo)
            m.actuator_forcerange[aid, 0] = mid - 0.5 * span
            m.actuator_forcerange[aid, 1] = mid + 0.5 * span
        if m.actuator_gear.shape[1] >= 1:
            m.actuator_gear[aid, 0] *= float(gear_mult)
        changed.append(nm)
    env.physics.forward()
    try:
        env.physics._reset_contexts()
    except AttributeError:
        pass
    print("[boost] actuators:", changed)

# -------------------- losses & traction --------------------
def ease_leg_losses(env, joint_names,
                    damping_mult=0.5, armature_mult=0.6):
    m = env.physics.model
    jname2id = {m.id2name(i, 'joint'): i for i in range(m.njnt)}
    changed = []
    for jn in joint_names:
        jid = jname2id.get(jn)
        if jid is None:
            continue
        dof = int(m.jnt_dofadr[jid])
        if 0 <= dof < m.dof_damping.size:
            m.dof_damping[dof] *= float(damping_mult)
        if 0 <= dof < m.dof_armature.size:
            m.dof_armature[dof] *= float(armature_mult)
        changed.append(jn)
    env.physics.forward()
    print("[loss] adjusted joints:", changed)

def improve_floor_grip(env, friction=(1.5, 0.005, 0.002), floor_drop_z=-0.02):
    """
    Set floor friction & optionally move the floor down a bit to avoid
    legs spawning slightly inside the contact plane.

    floor_drop_z < 0 → move floor DOWN by |floor_drop_z| meters.
    """
    m = env.physics.model
    changed = False

    for i in range(m.ngeom):
        nm = (m.id2name(i, 'geom') or "").lower()
        if 'floor' in nm or 'ground' in nm:
            # friction
            m.geom_friction[i, :] = friction
            # small vertical offset
            if floor_drop_z != 0.0:
                m.geom_pos[i, 2] += float(floor_drop_z)
            changed = True

    if changed:
        env.physics.forward()
    print(f"[floor] friction set to {friction}, floor_drop_z={floor_drop_z}")

# -------------------- optional: explosive burst & debug pulse --------------------
def add_explosive_burst(t_ms, act_controls, targets, t0_ms=None,
                        burst_ms=(10.0, 45.0), burst_gain=6.0):
    if not targets:
        return act_controls
    t = np.asarray(t_ms, float)
    # auto start
    if t0_ms is None:
        starts = []
        for nm in targets:
            u = np.asarray(act_controls.get(nm, np.zeros_like(t)))
            if not np.any(u):
                continue
            nz = np.where(np.abs(u) > (0.02 * np.nanmax(np.abs(u)) + 1e-9))[0]
            if len(nz):
                starts.append(t[nz[0]])
        t0_ms = (min(starts) if starts else t[0])
    start = t0_ms + float(burst_ms[0])
    end   = t0_ms + float(burst_ms[1])
    w = (t >= start) & (t <= end)
    out = {k: np.array(v, float, copy=True) for k, v in act_controls.items()}
    for nm in targets:
        if nm in out:
            out[nm][w] *= float(burst_gain)
    return out

def add_jump_pulse(t_ms, act_controls, targets,
                   start_ms=0.0, dur_ms=120.0, strength=1.0):
    """
    Overwrite the selected actuators with a strong, dimensionless pulse
    in [start_ms, start_ms + dur_ms]. This does NOT depend on existing
    lo/hi values – it just sets them to `strength` in that window.
    """
    from copy import deepcopy

    t = np.asarray(t_ms, float)
    out = deepcopy(act_controls)

    i0 = int(np.searchsorted(t, start_ms))
    i1 = int(np.searchsorted(t, start_ms + dur_ms))

    for nm in targets:
        if nm not in out:
            continue
        x = np.asarray(out[nm], float)
        # Ensure indices are in bounds
        i0_clamp = max(0, min(i0, x.size))
        i1_clamp = max(0, min(i1, x.size))
        if i1_clamp <= i0_clamp:
            continue
        # Overwrite with a strong pulse
        x[i0_clamp:i1_clamp] = float(strength)
        out[nm] = x

    return out

# -------------------- NEW: canonicalize actuator names to match MuJoCo model ----
def canonicalize_actuator_controls_to_env(act_controls, env):
    """
    Re-key act_controls so names match MuJoCo's actuator names exactly,
    using case-insensitive matching. Any unmatched names are skipped with a warning.
    """
    m = env.physics.model
    model_names = [(m.id2name(i, 'actuator') or "") for i in range(m.nu)]
    lc_to_name = {n.lower(): n for n in model_names}

    new_controls = {}
    for nm, series in act_controls.items():
        key = lc_to_name.get(str(nm).lower())
        if key is None:
            print(f"[warn] actuator '{nm}' not found in model; dropping its signal")
            continue
        series = np.asarray(series, float)
        if key in new_controls:
            new_controls[key] += series
        else:
            new_controls[key] = series
    return new_controls

def canonicalize_actuator_name_list(names, env):
    """
    Map a list of actuator names (BOOST_TARGETS, etc.) to actual model names,
    again in a case-insensitive way.
    """
    m = env.physics.model
    model_names = [(m.id2name(i, 'actuator') or "") for i in range(m.nu)]
    lc_to_name = {n.lower(): n for n in model_names}
    out = []
    for nm in names:
        key = lc_to_name.get(str(nm).lower())
        if key is None:
            print(f"[warn] actuator '{nm}' not found in model; skipping")
            continue
        out.append(key)
    return tuple(out)

# -------------------- render (env.physics loop, no camera tweaks here) ---------
def render_env_with_controls_and_debug(env, t_ms, act_controls, video_out,
                                       cam_name="back", slowmo=20.0,
                                       H=1920, W=1280, render_hz=120,
                                       debug_out_dir=None):
    """
    Render MuJoCo video AND log kinematics (body position + leg joints)
    so we can 'see' the motion numerically.

    Saves:
      - kinematics_debug.csv  (timeseries)
    Prints:
      - COM/torso displacement, bounce (z-range)
      - per-joint min/max/range
    """
    ph = env.physics
    m = ph.model

    # ---------- 1) Build control timeline (same as before) ----------
    dt = float(ph.timestep())
    dt_ms = 1000.0 * dt

    T = int(math.ceil((t_ms[-1] - t_ms[0]) / dt_ms)) + 1
    times_sim = t_ms[0] + np.arange(T) * dt_ms

    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}
    U = np.zeros((m.nu, T), float)

    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None:
            continue
        series = np.asarray(series, float)
        U[aid, :] = np.interp(times_sim, t_ms, series)
        lo, hi = m.actuator_ctrlrange[aid]
        if lo < hi:
            U[aid, :] = np.clip(U[aid, :], lo, hi)

    # ---------- 2) Prepare video writer ----------
    video_out.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_out), fps=render_hz)

    try:
        env.reset()
    except Exception:
        ph.reset()

    # initial preview frame
    frame0 = ph.render(height=H, width=W, camera_id=cam_name)
    imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

    base_steps_per_frame = max(1, int(round((1.0 / render_hz) / dt)))
    steps_per_frame = max(1, int(round(base_steps_per_frame / max(1.0, slowmo))))

    # ---------- 3) Prepare kinematic logging ----------
    debug_rows = []
    if debug_out_dir is None:
        debug_out_dir = video_out.parent

    # pick a "torso" body (for position)
    root_body_id = 1  # fallback
    for i in range(m.nbody):
        nm = (m.id2name(i, 'body') or "").lower()
        if "thorax" in nm or "torso" in nm or "fly" in nm:
            root_body_id = i
            break

    # map leg joints to DOF indices (same naming as you already use)
    leg_joint_names = [
        "coxa_T1_left", "femur_T1_left", "tibia_T1_left",
        "coxa_T2_left", "femur_T2_left", "tibia_T2_left",
        "coxa_T3_left", "femur_T3_left", "tibia_T3_left",
        "coxa_T1_right", "femur_T1_right", "tibia_T1_right",
        "coxa_T2_right", "femur_T2_right", "tibia_T2_right",
        "coxa_T3_right", "femur_T3_right", "tibia_T3_right",
    ]

    jname2dof = {}
    for jn in leg_joint_names:
        jid = None
        for i in range(m.njnt):
            if (m.id2name(i, 'joint') or "") == jn:
                jid = i
                break
        if jid is None:
            continue
        dof = int(m.jnt_dofadr[jid])
        jname2dof[jn] = dof

    # ---------- 4) Simulate, render, and log ----------
    frames = 0
    for k in range(T):
        # apply control
        ph.data.ctrl[:] = U[:, k]
        ph.step()

        # log kinematics at the same cadence we render
        if k % steps_per_frame == 0:
            t_now = times_sim[k]  # in ms

            # body position
            pos = ph.data.xpos[root_body_id].copy()  # (x, y, z)
            row = {
                "t_ms": float(t_now),
                "body_x": float(pos[0]),
                "body_y": float(pos[1]),
                "body_z": float(pos[2]),
            }

            # joint angles
            qpos = ph.data.qpos
            for jn, dof in jname2dof.items():
                if 0 <= dof < qpos.size:
                    row[f"{jn}_angle"] = float(qpos[dof])
                else:
                    row[f"{jn}_angle"] = np.nan

            debug_rows.append(row)

            # render frame
            frame = ph.render(height=H, width=W, camera_id=cam_name)
            writer.append_data(frame)
            frames += 1

    writer.close()
    print(f"[video] {video_out}  frames={frames}  fps={render_hz}  slowmo×{slowmo}  (~{frames/render_hz:.1f}s)")

    # ---------- 5) Save debug CSV & print summary ----------
    if debug_rows:
        dbg_df = pd.DataFrame(debug_rows)

        dbg_path = Path(debug_out_dir) / "kinematics_debug.csv"
        dbg_df.to_csv(dbg_path, index=False)
        print(f"[debug] kinematics timeseries → {dbg_path}")

        # basic body motion stats
        dx = dbg_df["body_x"].iloc[-1] - dbg_df["body_x"].iloc[0]
        dy = dbg_df["body_y"].iloc[-1] - dbg_df["body_y"].iloc[0]
        dz = dbg_df["body_z"].iloc[-1] - dbg_df["body_z"].iloc[0]
        z_min = dbg_df["body_z"].min()
        z_max = dbg_df["body_z"].max()

        print("\n[debug] body motion summary:")
        print(f"  Δx = {dx:.4f}, Δy = {dy:.4f}, Δz = {dz:.4f}")
        print(f"  body_z range = [{z_min:.4f}, {z_max:.4f}] (bounce height ≈ {z_max - z_min:.4f})")

        # per-joint angle ranges
        joint_cols = [c for c in dbg_df.columns if c.endswith("_angle")]
        print("\n[debug] leg joint angle ranges (rad-ish, depending on model):")
        for jc in joint_cols:
            vals = dbg_df[jc].to_numpy()
            if np.all(np.isnan(vals)):
                continue
            vmin, vmax = np.nanmin(vals), np.nanmax(vals)
            print(f"  {jc:<20} min={vmin:.3f}, max={vmax:.3f}, range={vmax - vmin:.3f}")
    else:
        print("[debug] no kinematic rows logged?! (check steps_per_frame / T)")





def debug_actuator_signal_stats(t_ms, act_controls, env, sample_targets=None, label="(pre-remap)"):
    """Print basic stats about actuator signals vs model ctrlrange."""
    m = env.physics.model
    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}

    print("\n[debug] actuator signal stats", label)
    for nm, sig in act_controls.items():
        aid = name2id.get(nm)
        if aid is None:
            continue
        sig = np.asarray(sig, float)
        if not np.any(np.isfinite(sig)):
            continue
        lo, hi = m.actuator_ctrlrange[aid]
        print(
            f"  {nm:15s} | ctrlrange=({lo:.3g},{hi:.3g}) "
            f"sig[min={np.nanmin(sig):.3g}, max={np.nanmax(sig):.3g}]"
        )

    if sample_targets:
        print("\n[debug] sample targets:")
        for nm in sample_targets:
            aid = name2id.get(nm)
            if aid is None:
                print(f"  {nm}: not found in model")
            else:
                lo, hi = m.actuator_ctrlrange[aid]
                print(f"  {nm}: ctrlrange=({lo:.3g},{hi:.3g})")




def summarize_actuator_signals(t_ms, act_controls, env, label=""):
    """
    Print min/max, fraction nonzero, and a tiny sample of values
    for each actuator that actually exists in the MuJoCo model.
    """
    m = env.physics.model
    name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}

    print(f"\n[summary] actuator signals {label}")
    print("  (only actuators that exist in the MuJoCo model are shown)")
    for nm, sig in sorted(act_controls.items()):
        aid = name2id.get(nm)
        if aid is None:
            continue  # this name doesn't exist in the XML model
        sig = np.asarray(sig, float)
        if sig.size == 0:
            continue
        finite = np.isfinite(sig)
        if not finite.any():
            continue
        s = sig[finite]
        frac_nz = np.count_nonzero(np.abs(s) > 1e-9) / s.size
        print(
            f"  {nm:16s} | min={s.min(): .3g}, max={s.max(): .3g}, "
            f"mean={s.mean(): .3g}, frac_nonzero={frac_nz: .2f}"
        )

def save_actuator_signals_csv(t_ms, act_controls, out_path):
    """
    Save a time × actuator table with one column per actuator.
    This is whatever you pass in (raw or scaled).
    """
    t_ms = np.asarray(t_ms, float)
    df = pd.DataFrame({"t_ms": t_ms})
    for nm, sig in sorted(act_controls.items()):
        df[str(nm)] = np.asarray(sig, float)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"[save] actuator signals → {out_path}")

def plot_actuator_signals(t_ms, act_controls, actuators=None, tlim=None):
    """
    Quick inline visualization of a few actuator signals.
    Example:
        plot_actuator_signals(t_long, act_controls_scaled,
                              actuators=['coxa_T2_left','femur_T2_left'])
    """
    import matplotlib.pyplot as plt  # safe to re-import

    t = np.asarray(t_ms, float)
    if actuators is None:
        actuators = sorted(act_controls.keys())

    plt.figure(figsize=(10, 6))
    for nm in actuators:
        if nm not in act_controls:
            print(f"[plot] actuator '{nm}' not in act_controls dict; skipping")
            continue
        sig = np.asarray(act_controls[nm], float)
        plt.plot(t, sig, label=nm)

    if tlim is not None:
        plt.xlim(tlim)
    plt.xlabel("time (ms)")
    plt.ylabel("signal")
    plt.legend()
    plt.tight_layout()
    plt.show()

    
def ensure_series_for_boost_targets(act_controls, t_ms, boost_targets):
    """
    Make sure every boosted actuator has a time series.
    If it's missing, create a zeros series so pulses/bursts can still drive it.
    """
    t_ms = np.asarray(t_ms, float)
    for nm in boost_targets:
        if nm not in act_controls:
            act_controls[nm] = np.zeros_like(t_ms, dtype=float)
    return act_controls

def ramp_in_controls(t_ms, act_controls, ramp_ms=5.0):
    """
    Multiply each actuator signal by a 0→1 ramp over `ramp_ms`
    starting at t_ms[0]. This avoids instant full-force kicks.
    """
    t = np.asarray(t_ms, float)
    ramp = np.clip((t - t[0]) / float(ramp_ms), 0.0, 1.0)

    out = {}
    for nm, sig in act_controls.items():
        x = np.asarray(sig, float)
        if x.shape != ramp.shape:
            # just in case shapes drift
            r = np.interp(t, t, ramp)
        else:
            r = ramp
        out[nm] = x * r
    return out

def mirror_left_to_right_actuators(act_controls, env=None):
    """
    For each actuator ending in 'left', mirror its signal onto the matching
    'right' actuator if that actuator exists and is basically flat/unused.

    This gives you bilateral motion even if spikes/mapping are left-biased.
    """
    from copy import deepcopy
    out = deepcopy(act_controls)

    # if env is passed, only mirror actuators that actually exist in the model
    valid_acts = None
    if env is not None:
        m = env.physics.model
        valid_acts = {m.id2name(i, 'actuator') for i in range(m.nu)}

    def is_flat(sig, eps=1e-6):
        sig = np.asarray(sig, float)
        return np.all(np.abs(sig) < eps)

    keys = list(act_controls.keys())
    for nm in keys:
        if "left" not in nm:
            continue
        right_name = nm.replace("left", "right")
        if valid_acts is not None and right_name not in valid_acts:
            continue

        left_sig = act_controls[nm]
        right_sig = out.get(right_name, None)

        # If right side missing or essentially zero → clone from left
        if right_sig is None or is_flat(right_sig):
            out[right_name] = np.asarray(left_sig, float).copy()

    print("[mirror] mirrored left→right for any flat/missing right actuators.")
    return out

def apply_ramp(t_ms, act_controls, ramp_ms=5.0):
    """
    Multiply all actuator signals by a 0→1 ramp over the first ramp_ms of sim time
    to avoid a huge impulse at t=0.
    """
    t = np.asarray(t_ms, float)
    t0 = float(t[0])
    ramp = np.clip((t - t0) / float(ramp_ms), 0.0, 1.0)

    out = {}
    for nm, sig in act_controls.items():
        s = np.asarray(sig, float)
        if s.shape != ramp.shape:
            # safety: shape mismatch → interpolate
            s = np.interp(t, t, s)
        out[nm] = s * ramp
    print(f"[ramp] applied {ramp_ms} ms fade-in to all actuator signals.")
    return out


from copy import deepcopy

def balance_left_right_sources(mapping_by_act, verbose=True):
    """
    Make left/right leg actuators share the same MN pools if one side is weaker.
    This does NOT change MN IDs, just mirrors the list from the stronger side.
    """
    # MuJoCo actuator names (what model.id2name('actuator') returns)
    lr_pairs = [
        ("coxa_T1_left",  "coxa_T1_right"),
        ("femur_T1_left", "femur_T1_right"),
        ("tibia_T1_left", "tibia_T1_right"),
        ("coxa_T2_left",  "coxa_T2_right"),
        ("femur_T2_left", "femur_T2_right"),
        ("tibia_T2_left", "tibia_T2_right"),
        ("coxa_T3_left",  "coxa_T3_right"),
        ("femur_T3_left", "femur_T3_right"),
        ("tibia_T3_left", "tibia_T3_right"),
    ]

    def _nsrc(lst):
        return len(lst) if isinstance(lst, list) else 0

    if verbose:
        print("[balance] left/right source counts BEFORE:")
        for L, R in lr_pairs:
            print(f"  {L:14s}: {_nsrc(mapping_by_act.get(L))}  |  {R:14s}: {_nsrc(mapping_by_act.get(R))}")

    for L, R in lr_pairs:
        srcL = mapping_by_act.get(L, [])
        srcR = mapping_by_act.get(R, [])
        nL, nR = _nsrc(srcL), _nsrc(srcR)

        # If one side has no sources but the other does, mirror the pool
        if nL == 0 and nR > 0:
            mapping_by_act[L] = deepcopy(srcR)
        elif nR == 0 and nL > 0:
            mapping_by_act[R] = deepcopy(srcL)

        # Optional: if one side has *much* fewer sources than the other, also mirror
        elif 0 < nL < 0.3 * nR:
            mapping_by_act[L] = deepcopy(srcR)
        elif 0 < nR < 0.3 * nL:
            mapping_by_act[R] = deepcopy(srcL)

    if verbose:
        print("[balance] left/right source counts AFTER:")
        for L, R in lr_pairs:
            print(f"  {L:14s}: {_nsrc(mapping_by_act.get(L))}  |  {R:14s}: {_nsrc(mapping_by_act.get(R))}")

    return mapping_by_act





def loop_signals_simple(t_ms, act_controls, total_ms=200.0):
    """
    Tile the original activation pattern back-to-back until we cover total_ms.
    No ramps, no scaling, just literal repetition.

    t_ms:          original time grid (e.g. 0–30 ms)
    act_controls:  {actuator_name -> np.array(signal over t_ms)}
    total_ms:      desired total duration in ms for the looped controls
    """
    t_ms = np.asarray(t_ms, float)
    if t_ms.size < 2:
        # degenerate case: nothing to loop
        return t_ms.copy(), {k: np.asarray(v, float).copy() for k, v in act_controls.items()}

    dt = float(np.median(np.diff(t_ms)))
    t0 = float(t_ms[0])
    base_dur = float(t_ms[-1] - t0)

    if base_dur <= 0:
        # weird time axis: just return original
        return t_ms.copy(), {k: np.asarray(v, float).copy() for k, v in act_controls.items()}

    # how many repeats do we need to cover total_ms?
    total_ms = float(total_ms)
    n_loops = max(1, int(np.ceil(total_ms / base_dur)))

    # build new time axis
    base_rel = t_ms - t0  # start from 0
    t_pieces = []
    for k in range(n_loops):
        t_pieces.append(base_rel + k * base_dur)
    new_t = t0 + np.concatenate(t_pieces)

    # tile each actuator's signal the same way
    out = {}
    for name, sig in act_controls.items():
        sig = np.asarray(sig, float)
        if sig.size == 0:
            out[name] = np.zeros_like(new_t)
            continue
        out[name] = np.tile(sig, n_loops)

    # trim to exact total_ms window
    mask = new_t <= (t0 + total_ms + 1e-9)
    new_t = new_t[mask]
    for name in out:
        out[name] = out[name][: mask.sum()]

    return new_t, out



# ===================== RUN =====================
# 1) mapping + controls
mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
mapping_by_act = balance_left_right_sources(mapping_by_act, verbose=True)

t_ms, act_controls = build_actuator_controls(
    RESULT_DIR, mapping_by_act,
    use_spikes=USE_SPIKES, v_thresh=V_THRESH_MV,
    tau_rise=TAU_RISE_MS, tau_decay=TAU_DECAY_MS, scale=KERNEL_SCALE,
    post_smooth_win=POST_SMOOTH_WIN
)

# 2) simply loop the original activation pattern over a longer window
#    EXTEND_TOTAL_MS is the *physical* ms we want to cover with repeated cycles
t_long, act_controls_long = loop_signals_simple(
    t_ms,
    act_controls,
    total_ms=EXTEND_TOTAL_MS   # e.g. 200.0 or 300.0
)


# 2b) TEMP: mirror left leg signals to right legs if right is flat
# (do this *before* we pass into remap_to_ctrlrange)
env = build_env_from_xml(WORLD_XML)  # build env a bit earlier now
act_controls_long = mirror_left_to_right_actuators(act_controls_long, env=env)

# 2c) ramp in forces over first few ms to reduce bounce
act_controls_long = apply_ramp(t_long, act_controls_long, ramp_ms=5.0)

if "abdomen" in act_controls_long:
    print("[hack] scaling abdomen control by 0.2 to reduce bounce")
    act_controls_long["abdomen"] = 0.2 * act_controls_long["abdomen"]

# 3) env + traction/loss + boosts + camera  (env already built above)
improve_floor_grip(env, friction=FLOOR_FRICTION)
ease_leg_losses(env, LEG_JOINTS, damping_mult=LOSS_DAMPING_MULT, armature_mult=LOSS_ARMATURE_MULT)
boost_actuator_strength_on_env(env, BOOST_TARGETS, gear_mult=GEAR_MULT, forcerange_mult=FORCERANGE_MULT)
CAM = set_camera_zoom(env, cam_name=CAM_NAME, distance_factor=CAM_DIST_FACTOR, fovy_deg=CAM_FOVY_DEG)

# Canonicalize BOOST_TARGETS to actual actuator names
BOOST_TARGETS_CANON = canonicalize_actuator_name_list(BOOST_TARGETS, env)


# 4) canonicalize names, ensure boost targets exist
act_controls_mod = canonicalize_actuator_controls_to_env(
    act_controls_long, env
)
act_controls_mod = ensure_series_for_boost_targets(
    act_controls_mod, t_long, BOOST_TARGETS_CANON
)

# Debug pulses / bursts on dimensionless signals
if ADD_EXPLOSIVE_BURST:
    act_controls_mod = add_explosive_burst(
        t_long, act_controls_mod, BOOST_TARGETS_CANON,
        t0_ms=BURST_T0_MS, burst_ms=BURST_WINDOW_MS, burst_gain=BURST_GAIN
    )

if ADD_JUMP_PULSE:
    act_controls_mod = add_jump_pulse(
        t_long, act_controls_mod, BOOST_TARGETS_CANON,
        start_ms=PULSE_MS_START,
        dur_ms=PULSE_MS_DUR,
        strength=PULSE_STRENGTH
    )


# Show what *raw, dimensionless* signals each actuator is getting
summarize_actuator_signals(
    t_long,
    act_controls_mod,
    env,
    label="(raw MN activation, before remap)"
)

# Optional: save to CSV for offline inspection
save_actuator_signals_csv(
    t_long,
    act_controls_mod,
    RESULT_DIR / "actuator_signals_raw.csv"
)

# Optional quick plot of key actuators
plot_actuator_signals(
    t_long,
    act_controls_mod,
    actuators=list(BOOST_TARGETS_CANON),  # or pick a subset
    tlim=(0, EXTEND_TOTAL_MS)
)





# DEBUG: see what signals we're actually sending (before remap)
debug_actuator_signal_stats(
    t_long, act_controls_mod, env,
    sample_targets=BOOST_TARGETS_CANON,
    label="before remap"
)

# debug pulse / burst
if ADD_EXPLOSIVE_BURST:
    act_controls_mod = add_explosive_burst(
        t_long, act_controls_mod, BOOST_TARGETS_CANON,
        t0_ms=BURST_T0_MS, burst_ms=BURST_WINDOW_MS, burst_gain=BURST_GAIN
    )
if ADD_JUMP_PULSE:
    act_controls_mod = add_jump_pulse(
        t_long, act_controls_mod, BOOST_TARGETS_CANON,
        start_ms=PULSE_MS_START, dur_ms=PULSE_MS_DUR, strength=PULSE_STRENGTH
    )

# NEW: ramp forces in over the first few ms
act_controls_mod = ramp_in_controls(t_long, act_controls_mod, ramp_ms=5.0)
# 5) remap to ctrlrange (use env model so ranges are exact)
act_controls_scaled = remap_to_ctrlrange(
    act_controls_mod, env,
    gain=CTRL_GAIN,
    bias_to_mid=CTRL_BIAS_TO_MID
)

# Show what signals are actually in env.physics.data.ctrl over time
summarize_actuator_signals(
    t_long,
    act_controls_scaled,
    env,
    label="(scaled to ctrlrange, what MuJoCo sees)"
)

save_actuator_signals_csv(
    t_long,
    act_controls_scaled,
    RESULT_DIR / "actuator_signals_scaled.csv"
)

# Optional plot again, but now in control units
plot_actuator_signals(
    t_long,
    act_controls_scaled,
    actuators=list(BOOST_TARGETS_CANON),
    tlim=(0, EXTEND_TOTAL_MS)
)


# 6) render
render_env_with_controls_and_debug(
    env, t_long, act_controls_scaled, VIDEO_OUT,
    cam_name=CAM, slowmo=SLOWMO_FACTOR, H=1920, W=1280, render_hz=120,
    debug_out_dir=RESULT_DIR
)



In [ ]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
MAPPING_CSV = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping_updated.csv")

OUT_MAPPING_CSV  = MAPPING_CSV.with_name("mn_to_actuator_mapping_with_groups.csv")
OUT_SUMMARY_CSV  = MAPPING_CSV.with_name("actuator_flex_ext_summary.csv")

print(f"[load] mapping: {MAPPING_CSV}")
df = pd.read_csv(MAPPING_CSV)

# ---------------------------------------------------------------------
# 1) Basic sanity checks
# ---------------------------------------------------------------------
required_cols = ["mn_id", "actuator_name"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Mapping CSV must contain column '{c}'")

if "muscle_group" in df.columns:
    print("[info] 'muscle_group' column already exists; will overwrite/infer values.")
else:
    df["muscle_group"] = None

# ---------------------------------------------------------------------
# 2) Infer flexor vs extensor per MN
#    Priority:
#      (a) existing 'muscle_group' if valid
#      (b) numeric 'sign' (<0 = flexor, >=0 = extensor)
#      (c) mn_type / neuron_type text heuristics ('flex' / 'exten')
#      (d) default to 'extensor'
# ---------------------------------------------------------------------
type_cols = [c for c in df.columns if c.lower() in ("mn_type", "neuron_type", "cell_type", "type")]
type_col  = type_cols[0] if type_cols else None

def infer_group(row):
    # a) keep existing valid label
    g = str(row.get("muscle_group", "")).strip().lower()
    if g in ("flexor", "extensor"):
        return g

    # b) use sign if present
    if "sign" in row and pd.notna(row["sign"]):
        try:
            s = float(row["sign"])
            return "flexor" if s < 0 else "extensor"
        except Exception:
            pass

    # c) heuristic from type name
    if type_col is not None:
        name = str(row[type_col]).lower()
        if "flex" in name:      # e.g. "tibia_flexor"
            return "flexor"
        if "exten" in name:     # e.g. "femur_extensor"
            return "extensor"
        if "retract" in name or "depressor" in name:
            # treat retractors/depressors as flexor-ish
            return "flexor"
        if "protract" in name or "levator" in name:
            # treat protractors/levators as extensor-ish
            return "extensor"

    # d) conservative default
    return "extensor"

df["muscle_group"] = df.apply(infer_group, axis=1)

# Any unknowns?
unknown_mask = ~df["muscle_group"].isin(["flexor", "extensor"])
n_unknown = int(unknown_mask.sum())
if n_unknown > 0:
    print(f"[warn] {n_unknown} rows still have unknown muscle_group; defaulting to 'extensor'.")
    df.loc[unknown_mask, "muscle_group"] = "extensor"

# Final sanity: ensure only flexor/extensor remain
assert set(df["muscle_group"].unique()).issubset({"flexor", "extensor"})

# ---------------------------------------------------------------------
# 3) Global counts
# ---------------------------------------------------------------------
n_total   = len(df)
n_flex    = int((df["muscle_group"] == "flexor").sum())
n_ext     = int((df["muscle_group"] == "extensor").sum())

print("\n[global muscle_group counts]")
print(f"  total rows:    {n_total}")
print(f"  flexors:       {n_flex}")
print(f"  extensors:     {n_ext}")

# ---------------------------------------------------------------------
# 4) Per-actuator summary: does each joint have both flexor & extensor?
# ---------------------------------------------------------------------
summary = (
    df
    .groupby("actuator_name")["muscle_group"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"flexor": "n_flexor", "extensor": "n_extensor"})
)

if "n_flexor" not in summary.columns:   summary["n_flexor"] = 0
if "n_extensor" not in summary.columns: summary["n_extensor"] = 0

summary["n_total"]       = summary["n_flexor"] + summary["n_extensor"]
summary["has_flexor"]    = summary["n_flexor"] > 0
summary["has_extensor"]  = summary["n_extensor"] > 0
summary["has_both"]      = summary["has_flexor"] & summary["has_extensor"]

# Save summary
summary = summary.sort_index()
summary.to_csv(OUT_SUMMARY_CSV)
print(f"\n[save] actuator flex/ext summary → {OUT_SUMMARY_CSV}")

# ---------------------------------------------------------------------
# 5) Print concise diagnostics
# ---------------------------------------------------------------------
n_actuators_total  = summary.shape[0]
n_both             = int(summary["has_both"].sum())
n_only_ext         = int((summary["has_extensor"] & ~summary["has_flexor"]).sum())
n_only_flex        = int((summary["has_flexor"] & ~summary["has_extensor"]).sum())
n_neither          = int((~summary["has_extensor"] & ~summary["has_flexor"]).sum())

print("\n[per-actuator diagnostic]")
print(f"  total actuators:           {n_actuators_total}")
print(f"  with BOTH flexor/extensor: {n_both}")
print(f"  extensor ONLY:             {n_only_ext}")
print(f"  flexor ONLY:               {n_only_flex}")
print(f"  neither (should be 0):     {n_neither}")

if n_only_ext or n_only_flex or n_neither:
    print("\n[details] actuators missing a flexor or extensor group:")
    bad = summary[~summary["has_both"]].copy()
    print(bad[["n_flexor", "n_extensor", "n_total"]])

# ---------------------------------------------------------------------
# 6) Save updated mapping with muscle_group
# ---------------------------------------------------------------------
df.to_csv(OUT_MAPPING_CSV, index=False)
print(f"\n[save] updated mapping with muscle_group → {OUT_MAPPING_CSV}")


In [ ]:
# ------------------------------------------------------------
# DEBUG: cross-check MN identity + reclassify muscle roles
# ------------------------------------------------------------
from pathlib import Path

MN_DIR = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN")

MAPPING_CSV      = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping_updated.csv")
ROLES_FULL_CSV   = MN_DIR / "mn_muscle_roles_from_neuprint_full.csv"
ROLES_MIN_OUT    = MN_DIR / "mn_muscle_roles_from_neuprint_minimal_reclassified.csv"
ROLE_DEBUG_CSV   = MN_DIR / "mn_roles_debug_sample.csv"

SPIKES_CSV       = RESULT_DIR / "spike_times.csv"
VOLTAGES_CSV     = RESULT_DIR / "voltages_somas.csv"

def _detect_id_col(df, candidates=("mn_id","bodyId","neuron_id","neuronId","id")):
    cols = [c for c in candidates if c in df.columns]
    if not cols:
        raise ValueError(f"Could not find any ID column in {df.columns.tolist()}")
    return cols[0]

def debug_mn_identity_and_roles():
    print("[debug] loading CSVs...")
    mapping  = pd.read_csv(MAPPING_CSV)
    roles_f  = pd.read_csv(ROLES_FULL_CSV)
    spikes   = pd.read_csv(SPIKES_CSV)
    volts    = pd.read_csv(VOLTAGES_CSV)

    # ---------------- ID sets from each source ----------------
    # mapping: mn_id
    if "mn_id" not in mapping.columns:
        raise ValueError("mapping CSV must contain 'mn_id'")
    ids_mapping = set(pd.to_numeric(mapping["mn_id"], errors="coerce").dropna().astype(int))

    # roles_full: bodyId or mn_id
    roles_id_col = "mn_id" if "mn_id" in roles_f.columns else "bodyId"
    roles_f[roles_id_col] = pd.to_numeric(roles_f[roles_id_col], errors="coerce")
    ids_roles = set(roles_f[roles_id_col].dropna().astype(int))

    # spikes: neuron_id (or similar)
    spikes_id_col = _detect_id_col(spikes, candidates=("neuron_id","mn_id","bodyId"))
    spikes[spikes_id_col] = pd.to_numeric(spikes[spikes_id_col], errors="coerce")
    ids_spikes = set(spikes[spikes_id_col].dropna().astype(int))

    # volts: columns other than t_ms
    ids_volts = set()
    for c in volts.columns:
        if c == "t_ms":
            continue
        try:
            ids_volts.add(int(c))
        except Exception:
            pass

    print("\n[ID SUMMARY]")
    print(f"  mapping mn_id count:   {len(ids_mapping)}")
    print(f"  roles bodyId count:    {len(ids_roles)}")
    print(f"  spikes neuron_id count:{len(ids_spikes)}")
    print(f"  volts soma cols count: {len(ids_volts)}")

    print("\n[ID INTERSECTIONS w/ mapping.mn_id]")
    print(f"  mapping ∩ roles:   {len(ids_mapping & ids_roles)}")
    print(f"  mapping ∩ spikes:  {len(ids_mapping & ids_spikes)}")
    print(f"  mapping ∩ volts:   {len(ids_mapping & ids_volts)}")

    print("\n[IDs in mapping but missing elsewhere]")
    print(f"  in mapping, not in roles:   {len(ids_mapping - ids_roles)}")
    print(f"  in mapping, not in spikes:  {len(ids_mapping - ids_spikes)}")
    print(f"  in mapping, not in volts:   {len(ids_mapping - ids_volts)}")

    # ---------------- aggressive re-classification of roles ----------------
    # Look across ALL text columns for each neuron
    import re

    def classify_from_any_text(row):
        texts = []
        for col in roles_f.columns:
            v = row[col]
            if isinstance(v, str):
                texts.append(v)
        s = " ".join(texts).lower()

        # early exit if no useful text
        if not s.strip():
            return "unknown"

        # make sure we don't mis-handle 'ext' vs 'flex'
        # Check longer, more specific tokens first.
        if "extensor" in s:
            return "extensor"
        if "flexor" in s:
            return "flexor"
        if "depressor" in s:
            return "depressor"
        if "levator" in s:
            return "levator"
        if "retractor" in s:
            return "retractor"
        if "protractor" in s:
            return "protractor"

        # More relaxed patterns (you can tighten/loosen these depending on what you see)
        # Use regex word boundaries where possible.
        if re.search(r"\bext\b", s) or " ext " in s or "extend" in s:
            return "extensor"
        if re.search(r"\bflx\b", s) or " flex " in s or "flx " in s:
            return "flexor"
        if "dep " in s or "dep-" in s:
            return "depressor"
        if "lev " in s or "lev-" in s:
            return "levator"

        return "unknown"

    print("\n[reclassify] scanning neuPrint roles_full for labels...")
    roles_f["muscle_role_reclassified"] = roles_f.apply(classify_from_any_text, axis=1)

    print("\n[role counts AFTER reclassification (before filtering to mapping IDs)]")
    print(roles_f["muscle_role_reclassified"].value_counts(dropna=False).sort_index())

    # save a small sample for manual inspection
    sample_cols = [roles_id_col, "instance", "type", "status"] if all(
        c in roles_f.columns for c in ("instance","type","status")
    ) else roles_f.columns.tolist()
    sample = roles_f[sample_cols + ["muscle_role_reclassified"]].head(50)
    sample.to_csv(ROLE_DEBUG_CSV, index=False)
    print(f"\n[reclassify] saved sample rows → {ROLE_DEBUG_CSV}")

    # Now restrict to MNs that appear in the mapping, and build new minimal roles file
    roles_sub = roles_f[roles_f[roles_id_col].isin(ids_mapping)].copy()
    roles_min = (
        roles_sub[[roles_id_col, "muscle_role_reclassified"]]
        .drop_duplicates()
        .rename(columns={roles_id_col: "mn_id", "muscle_role_reclassified": "muscle_role"})
    )

    print("\n[role counts restricted to mapping MNs]")
    print(roles_min["muscle_role"].value_counts(dropna=False).sort_index())

    roles_min.to_csv(ROLES_MIN_OUT, index=False)
    print(f"\n[save] minimal roles for mapping MNs → {ROLES_MIN_OUT}")

debug_mn_identity_and_roles()


In [ ]:
import pandas as pd
from pathlib import Path
import re

# ------------ PATHS (adjust if needed) ------------
BASE_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc")
RESULT_DIR = BASE_DIR / "hemi_runs" / "hemi_09A"
MN_DIR     = BASE_DIR / "MN"

SPIKES_CSV   = RESULT_DIR / "spike_times.csv"
VOLTS_CSV    = RESULT_DIR / "voltages_somas.csv"
MAPPING_CSV  = MN_DIR / "mn_to_actuator_mapping_updated.csv"
ROLES_FULL_CSV = MN_DIR / "mn_muscle_roles_from_neuprint_full.csv"

OUT_DIAG_CSV = MN_DIR / "mn_mapping_spikes_volts_roles_diagnostic.csv"

print("[debug] loading CSVs...")
spikes   = pd.read_csv(SPIKES_CSV)
volts    = pd.read_csv(VOLTS_CSV)
mapping  = pd.read_csv(MAPPING_CSV)
roles_f  = pd.read_csv(ROLES_FULL_CSV)

# ------------ NORMALIZE ID COLUMNS ------------
# mapping: assume mn_id column exists
if "mn_id" not in mapping.columns:
    raise ValueError(f"'mn_id' column not found in mapping CSV columns: {mapping.columns.tolist()}")
mapping["mn_id"] = pd.to_numeric(mapping["mn_id"], errors="coerce")

# spikes: assume neuron_id column exists
if "neuron_id" not in spikes.columns:
    raise ValueError(f"'neuron_id' column not found in spikes CSV columns: {spikes.columns.tolist()}")
spikes["neuron_id"] = pd.to_numeric(spikes["neuron_id"], errors="coerce")

# volts: neuron IDs are the column names except t_ms
voltage_ids = {int(c) for c in volts.columns if c != "t_ms"}

# roles_full: detect bodyId vs mn_id
if "mn_id" in roles_f.columns:
    roles_id_col = "mn_id"
elif "bodyId" in roles_f.columns:
    roles_id_col = "bodyId"
else:
    raise ValueError(f"No 'mn_id' or 'bodyId' column found in roles CSV columns: {roles_f.columns.tolist()}")

roles_f[roles_id_col] = pd.to_numeric(roles_f[roles_id_col], errors="coerce")

mapped_ids  = set(mapping["mn_id"].dropna().astype(int))
spike_ids   = set(spikes["neuron_id"].dropna().astype(int))
roles_ids   = set(roles_f[roles_id_col].dropna().astype(int))

# ------------ INTERSECTIONS ------------
mapped_spike_ids  = sorted(mapped_ids & spike_ids)
mapped_volt_ids   = sorted(mapped_ids & voltage_ids)
mapped_roles_ids  = sorted(mapped_ids & roles_ids)

print("\n[ID SUMMARY]")
print(f"  mapping mn_id count:   {len(mapped_ids)}")
print(f"  spikes neuron_id count:{len(spike_ids)}")
print(f"  volts neuron_id count: {len(voltage_ids)}")
print(f"  roles bodyId count:    {len(roles_ids)}")

print("\n[ID INTERSECTIONS w/ mapping.mn_id]")
print(f"  mapping ∩ spikes:  {len(mapped_spike_ids)}")
print(f"  mapping ∩ volts:   {len(mapped_volt_ids)}")
print(f"  mapping ∩ roles:   {len(mapped_roles_ids)}")

print("\n[mapped & spiking MN IDs]")
print(mapped_spike_ids)

print("\n[mapped & in voltages_somas MN IDs]")
print(mapped_volt_ids)

# ------------ RECLASSIFY MUSCLE ROLE (flexor/extensor/etc.) ------------
def classify_muscle_role(row):
    """Look across all string columns and classify muscle role from free text."""
    texts = []
    for col in roles_f.columns:
        v = row[col]
        if isinstance(v, str):
            texts.append(v)
    s = " ".join(texts).lower()

    if not s.strip():
        return "unknown"

    # direct labels first
    if "extensor"  in s: return "extensor"
    if "flexor"    in s: return "flexor"
    if "depressor" in s: return "depressor"
    if "levator"   in s: return "levator"
    if "retractor" in s: return "retractor"
    if "protractor" in s: return "protractor"

    # more relaxed matches
    if re.search(r"\bext\b", s) or " extend" in s or " ext " in s:
        return "extensor"
    if re.search(r"\bflx\b", s) or " flex " in s or "flx " in s:
        return "flexor"
    if " dep " in s or "dep-" in s:
        return "depressor"
    if " lev " in s or "lev-" in s:
        return "levator"

    return "unknown"

print("\n[reclassify] scanning neuPrint roles_full for labels...")
roles_f["muscle_role_reclassified"] = roles_f.apply(classify_muscle_role, axis=1)

print("\n[role counts AFTER reclassification (all neuPrint MNs)]")
print(roles_f["muscle_role_reclassified"].value_counts().sort_index())

# restrict roles table to just columns we care about
roles_min = roles_f[[roles_id_col, "muscle_role_reclassified"]].drop_duplicates()
roles_min = roles_min.rename(columns={roles_id_col: "mn_id"})

# ------------ BUILD DIAGNOSTIC TABLE FOR MAPPED & ACTIVE MNS ------------
# union of mapped∩spikes and mapped∩volts
ids_of_interest = sorted(set(mapped_spike_ids) | set(mapped_volt_ids))

diag_rows = []
for nid in ids_of_interest:
    in_map   = nid in mapped_ids
    in_spk   = nid in spike_ids
    in_volt  = nid in voltage_ids

    role_row = roles_min[roles_min["mn_id"] == nid]
    if len(role_row):
        role = role_row["muscle_role_reclassified"].iloc[0]
    else:
        role = "unknown"

    diag_rows.append({
        "mn_id": int(nid),
        "in_mapping": in_map,
        "in_spikes": in_spk,
        "in_voltages_somas": in_volt,
        "muscle_role": role,
    })

diag = pd.DataFrame(diag_rows).sort_values("mn_id")

print("\n[DIAGNOSTIC TABLE: mapped MNs that appear in spikes and/or voltages]")
print(f"  total MNs of interest: {len(diag)}")
print(diag.head(30))  # show first 30 rows as a sample

print("\n[role counts for MNs of interest]")
print(diag["muscle_role"].value_counts().sort_index())

diag.to_csv(OUT_DIAG_CSV, index=False)
print(f"\n[save] full diagnostic → {OUT_DIAG_CSV}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- PATHS ----------
BASE_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc")
RESULT_DIR = BASE_DIR / "hemi_runs" / "hemi_09A"
MN_DIR     = BASE_DIR / "MN"

SPIKES_CSV   = RESULT_DIR / "spike_times.csv"
MAPPING_CSV  = MN_DIR / "mn_to_actuator_mapping_updated.csv"  # or with_roles_cheated if you prefer
ROLES_FULL   = MN_DIR / "mn_muscle_roles_from_neuprint_full.csv"

OUT_CSV      = RESULT_DIR / "mn_spike_order_by_leg.csv"

print("[debug] loading CSVs...")
print(f"  spikes:   {SPIKES_CSV}")
print(f"  mapping:  {MAPPING_CSV}")
print(f"  roles:    {ROLES_FULL}")

spikes  = pd.read_csv(SPIKES_CSV)
mapping = pd.read_csv(MAPPING_CSV)
roles_f = pd.read_csv(ROLES_FULL)

# ---------- NORMALIZE ID COLUMNS ----------
# spikes: neuron_id
if "neuron_id" not in spikes.columns:
    raise ValueError(f"Expected 'neuron_id' in {SPIKES_CSV}, got: {spikes.columns.tolist()}")
spikes["neuron_id"] = pd.to_numeric(spikes["neuron_id"], errors="coerce")

# mapping: mn_id
if "mn_id" not in mapping.columns:
    raise ValueError(f"Expected 'mn_id' in {MAPPING_CSV}, got: {mapping.columns.tolist()}")
mapping["mn_id"] = pd.to_numeric(mapping["mn_id"], errors="coerce")

# roles: bodyId or mn_id → neuPrint IDs, plus instance label if present
if "mn_id" in roles_f.columns:
    roles_id_col = "mn_id"
elif "bodyId" in roles_f.columns:
    roles_id_col = "bodyId"
else:
    raise ValueError(f"Roles CSV missing id column; has: {roles_f.columns.tolist()}")

roles_f[roles_id_col] = pd.to_numeric(roles_f[roles_id_col], errors="coerce")

# try to find instance-like column for full label
INSTANCE_COL = None
for cand in ["instance", "name", "bodyName", "body_name"]:
    if cand in roles_f.columns:
        INSTANCE_COL = cand
        break
if INSTANCE_COL is None:
    # fallback: make a synthetic label
    INSTANCE_COL = "_synthetic_instance"
    roles_f[INSTANCE_COL] = roles_f[roles_id_col].astype(str).radd("MN_")

# ---------- MERGE: mapping ↔ spikes ↔ roles ----------
# mapped + spiking MNs
spike_mapped = spikes.merge(
    mapping[["mn_id", "actuator_name"]],
    left_on="neuron_id",
    right_on="mn_id",
    how="inner",
)

# attach instance label from neuPrint
roles_min = roles_f[[roles_id_col, INSTANCE_COL]].drop_duplicates()
roles_min = roles_min.rename(columns={roles_id_col: "mn_id"})
spike_mapped = spike_mapped.merge(
    roles_min,
    on="mn_id",
    how="left",
)

# ---------- INFER LEG FROM ACTUATOR NAME ----------
def infer_leg_from_actuator(act_name: str) -> str:
    """Return a coarse leg label like 'front_left', 'middle_right', 'hind_left', or 'body_midline'."""
    s = str(act_name).lower()
    # thoracic segment
    if "t1" in s:
        seg = "front"
    elif "t2" in s:
        seg = "middle"
    elif "t3" in s:
        seg = "hind"
    else:
        seg = "body"
    # side
    if "left" in s:
        side = "left"
    elif "right" in s:
        side = "right"
    else:
        side = "midline"
    return f"{seg}_{side}"

spike_mapped["leg_id"] = spike_mapped["actuator_name"].apply(infer_leg_from_actuator)

# ---------- COMPUTE FIRST / MEAN SPIKE TIMES PER MN ----------
# We only care about MNs that spike at least once
grp = spike_mapped.groupby(["leg_id", "mn_id", "actuator_name", INSTANCE_COL])

summary_rows = []
for (leg_id, mn_id, act, inst), g in grp:
    times = g["spike_time_ms"].to_numpy(float)
    if times.size == 0:
        continue
    first_t = float(times.min())
    mean_t  = float(times.mean())
    n_spikes = int(times.size)
    summary_rows.append({
        "leg_id": leg_id,
        "mn_id": int(mn_id),
        "actuator_name": act,
        "instance": inst,
        "n_spikes": n_spikes,
        "first_spike_ms": first_t,
        "mean_spike_ms":  mean_t,
    })

mn_leg_timing = pd.DataFrame(summary_rows)

if mn_leg_timing.empty:
    print("\n[WARN] No mapped MNs with spikes found. Double-check spike_times and mapping.")
else:
    # order within each leg by first spike time, then mean time
    mn_leg_timing = mn_leg_timing.sort_values(
        ["leg_id", "first_spike_ms", "mean_spike_ms", "mn_id"]
    ).reset_index(drop=True)

    # ---------- PRINT A HUMAN-READABLE SUMMARY ----------
    print("\n[SUMMARY] spiking mapped MNs by leg (ordered by first spike):")
    for leg_id, df_leg in mn_leg_timing.groupby("leg_id"):
        print(f"\n=== {leg_id} ===  (n={len(df_leg)})")
        for _, row in df_leg.iterrows():
            print(f"  t_first={row['first_spike_ms']:7.3f} ms  "
                  f"t_mean={row['mean_spike_ms']:7.3f} ms  "
                  f"mn_id={row['mn_id']:>6}  "
                  f"act={row['actuator_name']}  "
                  f"instance={row['instance']}")

    # ---------- SAVE FOR INSPECTION IN EXCEL ----------
    mn_leg_timing.to_csv(OUT_CSV, index=False)
    print(f"\n[save] ordered MN leg timing → {OUT_CSV}")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------ PATHS ------------------
SPIKE_ORDER_CSV = Path(
    "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A/mn_spike_order_by_leg.csv"
)  # <-- update if needed

df = pd.read_csv(SPIKE_ORDER_CSV)

# sanity
required_cols = {"leg_id", "mn_id", "actuator_name", "instance", "first_spike_ms"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"mn_spike_order_by_leg.csv is missing columns: {missing}")

# ------------------ 1D two-phase splitter ------------------
def split_into_two_phases(times: np.ndarray):
    """
    Given an array of spike times (1D), find the best split index k (1..n-1)
    that minimizes within-cluster squared error. Returns:
      phase_labels: list of 'early_phase' or 'late_phase' for each time
    """
    n = len(times)
    if n < 2:
        return ["unknown"] * n

    # sort times but remember original order
    idx_sorted = np.argsort(times)
    t_sorted = times[idx_sorted]

    best_k = None
    best_sse = np.inf

    # try all possible splits 1..n-1
    for k in range(1, n):
        left = t_sorted[:k]
        right = t_sorted[k:]
        muL = left.mean()
        muR = right.mean()
        sse = ((left - muL) ** 2).sum() + ((right - muR) ** 2).sum()
        if sse < best_sse:
            best_sse = sse
            best_k = k

    # if for some reason we didn't find anything better, mark all unknown
    if best_k is None:
        return ["unknown"] * n

    # early = first best_k in the *sorted* order
    early_mask_sorted = np.zeros(n, dtype=bool)
    early_mask_sorted[:best_k] = True

    # map back to original order
    phase_labels = [""] * n
    for original_i, sorted_i in enumerate(idx_sorted):
        phase_labels[original_i] = "early_phase" if early_mask_sorted[original_i] else "late_phase"

    return phase_labels

# ------------------ APPLY PER LEG / ACTUATOR ------------------
rows = []
for (leg_id, actuator), grp in df.groupby(["leg_id", "actuator_name"], sort=False):
    grp = grp.sort_values("first_spike_ms").reset_index(drop=True)

    times = grp["first_spike_ms"].to_numpy(dtype=float)
    if len(times) < 2 or np.allclose(times, times[0]):
        phase_labels = ["unknown"] * len(times)
    else:
        phase_labels = split_into_two_phases(times)

    # map phases to candidate roles (purely by timing)
    # early_phase  → candidate_flexor
    # late_phase   → candidate_extensor
    candidate_roles = []
    for ph in phase_labels:
        if ph == "early_phase":
            candidate_roles.append("candidate_flexor")
        elif ph == "late_phase":
            candidate_roles.append("candidate_extensor")
        else:
            candidate_roles.append("unknown")

    grp_out = grp.copy()
    grp_out["phase_label"] = phase_labels
    grp_out["candidate_role"] = candidate_roles
    rows.append(grp_out)

df_phased = pd.concat(rows, ignore_index=True)

# nice ordering
df_phased = df_phased.sort_values(
    ["leg_id", "actuator_name", "first_spike_ms", "mn_id"]
).reset_index(drop=True)

# ------------------ SAVE + DIAGNOSTICS ------------------
OUT_CSV = SPIKE_ORDER_CSV.with_name("mn_spike_order_by_leg_with_phases.csv")
df_phased.to_csv(OUT_CSV, index=False)
print(f"[save] phased spike table → {OUT_CSV}")

print("\n[summary] candidate flexor/extensor counts per leg & actuator:")
summary = (
    df_phased
    .groupby(["leg_id", "actuator_name", "candidate_role"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
print(summary)

print("\n[example] first 10 rows (for sanity):")
print(df_phased.head(10))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# -------- PATHS (adjust if needed) ----------
MN_DIR       = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN")
MAPPING_IN   = MN_DIR / "mn_to_actuator_mapping_updated.csv"
PHASES_CSV   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A/mn_spike_order_by_leg_with_phases.csv")   # from previous step
MAPPING_OUT  = MN_DIR / "mn_to_actuator_mapping_phased.csv"

print("[paths]")
print("  mapping_in :", MAPPING_IN)
print("  phases_csv :", PHASES_CSV)
print("  mapping_out:", MAPPING_OUT)

# -------- LOAD --------
mapping = pd.read_csv(MAPPING_IN)
phased  = pd.read_csv(PHASES_CSV)

# sanity
if "mn_id" not in mapping.columns:
    raise ValueError("mapping CSV must have column 'mn_id'")
if "mn_id" not in phased.columns or "candidate_role" not in phased.columns:
    raise ValueError("phased CSV must have 'mn_id' and 'candidate_role' columns")

# ensure numeric ids
mapping["mn_id"] = pd.to_numeric(mapping["mn_id"], errors="coerce").astype("Int64")
phased["mn_id"]  = pd.to_numeric(phased["mn_id"], errors="coerce").astype("Int64")

# keep only rows with valid ids
mapping = mapping.dropna(subset=["mn_id"]).copy()
phased  = phased.dropna(subset=["mn_id"]).copy()

# if mapping has sign column, keep; otherwise init with +1
if "sign" not in mapping.columns:
    mapping["sign"] = 1.0
mapping["sign"] = pd.to_numeric(mapping["sign"], errors="coerce").fillna(1.0)

# we only need unique (mn_id → candidate_role); if multiple rows per mn_id, use first non-unknown
phased_role = phased[["mn_id", "candidate_role"]].copy()
phased_role["candidate_role"] = phased_role["candidate_role"].fillna("unknown")

# collapse: prefer extensor/flexor over unknown if there are multiple rows
def _collapse_roles(group):
    # priority: flexor > extensor > unknown
    roles = group["candidate_role"].tolist()
    if "candidate_flexor" in roles:
        return "candidate_flexor"
    if "candidate_extensor" in roles:
        return "candidate_extensor"
    return "unknown"

roles_collapsed = (
    phased_role
    .groupby("mn_id", as_index=False)
    .apply(_collapse_roles)
)
roles_collapsed = roles_collapsed.rename(columns={None: "candidate_role"})

# merge roles into mapping
mapping = mapping.merge(roles_collapsed, on="mn_id", how="left")

# ----------------- ASSIGN NEW SIGN / MUSCLE_GROUP -----------------
# create/overwrite a muscle_group_auto column so you can inspect later
def role_to_sign_and_group(role, old_sign):
    role = (role or "unknown")
    if role == "candidate_flexor":
        return -1.0, "flexor_auto"
    if role == "candidate_extensor":
        return  1.0, "extensor_auto"
    # unknown: keep old sign
    return float(old_sign), "unknown_auto"

new_signs = []
groups    = []
for _, row in mapping.iterrows():
    sign, grp = role_to_sign_and_group(row.get("candidate_role", "unknown"), row["sign"])
    new_signs.append(sign)
    groups.append(grp)

mapping["sign"] = new_signs
mapping["muscle_group_auto"] = groups

# ------------- DIAGNOSTICS -------------
print("\n[diagnostic] role counts in mapping after phase-based assignment:")
print(mapping["candidate_role"].value_counts(dropna=False))

print("\n[diagnostic] sign summary (how many +1 vs -1):")
print(mapping["sign"].value_counts())

print("\n[diagnostic] flexor/extensor per actuator (auto):")
act_summary = (
    mapping
    .groupby(["actuator_name", "muscle_group_auto"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
print(act_summary)

OUT = MAPPING_OUT
mapping.to_csv(OUT, index=False)
print(f"\n[save] updated, phase-aware mapping → {OUT}")


In [ ]:
# # ========= NEURON → MuJoCo using the SAME env style as your notebook =========
# # Drives env.physics from NEURON spike→actuator series, with live camera control.

# import os, sys, math
# from pathlib import Path
# import numpy as np, pandas as pd, imageio

# # --- paths you already have ---
# RESULT_DIR   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/DNp01_edges/results/demo_run")
# MAPPING_CSV  = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv")
# FLY_ASSETS   = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets")
# WORLD_XML    = FLY_ASSETS / "floor.xml"      # includes fruitfly.xml
# VIDEO_OUT    = RESULT_DIR / "mujoco_run_env.mp4"

# # Optional: where 'neuro_adapter' lives (so we can import the same helpers your notebook uses)
# NEURO_FLY_SRC = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly")
# if str(NEURO_FLY_SRC) not in sys.path:
#     sys.path.insert(0, str(NEURO_FLY_SRC))

# # Render backend: GLFW for on-screen; use "osmesa" if headless
# os.environ.setdefault("MUJOCO_GL", "glfw")

# # -------------------- mapping & NEURON→controls --------------------
# # MN_OVERRIDES = { 10068: "femur_T2_left", 10110: "femur_T2_right" }
# MN_OVERRIDES = {
#     10068: ["tibia_T2_left",  "femur_T2_left"],
#     10110: ["tibia_T2_right", "femur_T2_right"],
# }

# def load_mapping(csv_path, overrides=None):
#     df = pd.read_csv(csv_path)
#     if "actuator_name" not in df.columns:
#         raise ValueError("mapping CSV must contain 'actuator_name'")
#     df = df[df["actuator_name"].notna()]
#     df = df[df["actuator_name"].astype(str).str.strip() != ""]
#     for c, default in (("gain", 1.0), ("bias", 0.0)):
#         df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default) if c in df.columns else default
#     if "sign" in df.columns:
#         def _sgn(x):
#             try: return 1.0 if float(x) >= 0 else -1.0
#             except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
#         df["sign"] = df["sign"].apply(_sgn)
#     else:
#         df["sign"] = 1.0
#     # --- inside load_mapping(), replace your current "apply overrides" block ---
#     if overrides:
#         # remove any existing rows for these MNs (so we don't double-map unless you want to)
#         try:
#             mn_ids_to_replace = set(int(k) for k in overrides.keys())
#             if "mn_id" in df.columns:
#                 df = df[~df["mn_id"].astype(int).isin(mn_ids_to_replace)].copy()
#         except Exception:
#             pass
    
#         # add one row per (MN, actuator) target
#         add_rows = []
#         for bid, names in overrides.items():
#             bid = int(bid)
#             if not isinstance(names, (list, tuple)):
#                 names = [names]
#             for name in names:
#                 add_rows.append({
#                     "mn_type": "",
#                     "mn_id": bid,
#                     "actuator_name": str(name),
#                     "gain": 1.0,
#                     "bias": 0.0,
#                     "sign": 1.0,
#                 })
#         if add_rows:
#             df = pd.concat([df, pd.DataFrame(add_rows)], ignore_index=True)

#     by_act = {}
#     for _, r in df.iterrows():
#         by_act.setdefault(str(r["actuator_name"]).strip(), []).append(
#             dict(mn_id=int(r["mn_id"]), gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
#         )
#     return by_act

# def spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
#     a = np.zeros_like(t_grid_ms, dtype=float)
#     if len(spike_times_ms) == 0:
#         return a
#     dt = float(np.median(np.diff(t_grid_ms)))
#     tmax = 6.0 * tau_decay
#     k_t = np.arange(0.0, tmax + dt, dt)
#     k = (1.0 - np.exp(-k_t / tau_rise)) * np.exp(-k_t / tau_decay) * scale
#     for ts in spike_times_ms:
#         i = int(np.searchsorted(t_grid_ms, ts))
#         j = min(i + len(k), len(a))
#         a[i:j] += k[:j-i]
#     return a

# def build_actuator_controls(result_dir: Path, mapping_by_act,
#                             tau_rise=1.0, tau_decay=6.0, scale=1.0,
#                             from_voltages=False, v_thresh=0.0):
#     spikes_csv   = result_dir / "spike_times.csv"
#     voltages_csv = result_dir / "voltages_somas.csv"
#     if not voltages_csv.exists():
#         raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")
#     vdf = pd.read_csv(voltages_csv)
#     t_ms = vdf["t_ms"].to_numpy(dtype=float)

#     if (not from_voltages) and spikes_csv.exists():
#         sdf = pd.read_csv(spikes_csv)
#         if not sdf.empty:
#             sdf["neuron_id"] = sdf["neuron_id"].astype(int)
#             id2spk = {nid: grp["spike_time_ms"].to_numpy(dtype=float)
#                       for nid, grp in sdf.groupby("neuron_id")}
#         else:
#             id2spk = {}
#     else:
#         id2spk = {}
#         for col in vdf.columns:
#             if col == "t_ms": continue
#             v = vdf[col].to_numpy(dtype=float)
#             above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
#             ts = t_ms[1:][above]
#             try: nid = int(col)
#             except: continue
#             id2spk[nid] = ts

#     act_controls = {}
#     for act, sources in mapping_by_act.items():
#         acc = np.zeros_like(t_ms, dtype=float)
#         for s in sources:
#             spikes = id2spk.get(int(s["mn_id"]), np.array([], float))
#             a = spikes_to_activation(spikes, t_ms, tau_rise=tau_rise, tau_decay=tau_decay, scale=scale)
#             acc += s["sign"] * s["gain"] * a
#         acc += sum(s["bias"] for s in sources)
#         act_controls[act] = acc
#     return t_ms, act_controls

# # -------------------- helper: extend & scale signals --------------------
# def extend_signals(t_ms, act_controls, total_ms=120.0):
#     t0 = float(t_ms[0]); dt = float(np.median(np.diff(t_ms)))
#     new_times = np.arange(t0, total_ms + 1e-9, dt)
#     if new_times.shape[0] <= len(t_ms):
#         return t_ms, act_controls
#     out = {}
#     for k, sig in act_controls.items():
#         sig = np.asarray(sig, float)
#         tail = np.full(len(new_times) - len(t_ms), (sig[-1] if len(sig) else 0.0), float)
#         out[k] = np.concatenate([sig, tail], axis=0)
#     return new_times, out

# # -------------------- ENV BUILDER (do it like your notebook) --------------------
# def build_env_from_xml(world_xml: Path):
#     """
#     Tries to build the same style env you used in the notebook:
#     1) neuro_adapter.env_loader.load_env (if available)
#     2) flybody.fly_envs generic loader (if available)
#     3) fallback: dm_control mujoco.Physics wrapped in a SimpleEnv
#     """
#     # 1) neuro_adapter
#     try:
#         from neuro_adapter.env_loader import load_env
#         env = load_env(xml_path=str(world_xml))
#         return env
#     except Exception:
#         pass
#     # 2) flybody fallback (simple world loader)
#     try:
#         import flybody.fly_envs as fb
#         # try a generic world loader if present
#         if hasattr(fb, "xml_world"):
#             env = fb.xml_world(str(world_xml))
#             return env
#     except Exception:
#         pass
#     # 3) final fallback: dm_control physics, wrapped to look like env
#     from dm_control import mujoco as dm_mujoco
#     physics = dm_mujoco.Physics.from_xml_path(str(world_xml))
#     class SimpleEnv:
#         def __init__(self, physics):
#             self.physics = physics
#         def reset(self): self.physics.reset(); return None
#         def step(self, *_): self.physics.step(); return None
#         def control_timestep(self): return float(self.physics.timestep())
#         # dm_control-style action_spec if someone asks:
#         class _Spec:
#             def __init__(self, lo, hi): self.minimum, self.maximum = lo, hi
#         def action_spec(self):
#             m = self.physics.model
#             lo = m.actuator_ctrlrange[:,0] if m.actuator_ctrlrange.size else np.full((m.nu,), -1.0)
#             hi = m.actuator_ctrlrange[:,1] if m.actuator_ctrlrange.size else np.full((m.nu,),  1.0)
#             return SimpleEnv._Spec(lo, hi)
#     return SimpleEnv(physics)

# # -------------------- camera control on the LIVE env.model --------------------
# def set_env_camera(env, cam_name="side", back_factor=3.0, fovy_scale=0.8):
#     ph = env.physics
#     m = ph.model
    
#     # find camera id
#     cam_id = -1
#     for i in range(m.ncam):
#         nm = (m.id2name(i, 'camera') or "")
#         if nm == cam_name:
#             cam_id = i; break
#     if cam_id < 0:
#         print(f"[camera] '{cam_name}' not found; keeping default")
#         return cam_name  # return name anyway so render uses same string
    
#     # move camera & widen FOV
#     center = np.array(m.stat.center, dtype=float)
#     pos = m.cam_pos[cam_id].copy()
#     delta = pos - center
#     if np.linalg.norm(delta) > 1e-9:
#         m.cam_pos[cam_id] = center + delta * float(back_factor)
#     m.cam_fovy[cam_id] = float(m.cam_fovy[cam_id]) * float(fovy_scale)
    
#     # APPLY: forward + reset cached dm_control render contexts
#     ph.forward()
#     try:
#         ph._reset_contexts()   # <-- critical for dm_control
#     except AttributeError:
#         pass
    
#     print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.2f}")
#     return cam_name  # return the name to use in render()


# # -------------------- option 2: BOOST actuator strength (gear/forcerange) ----
# def boost_actuator_strength_on_env(env, targets=None, gear_mult=3.0, forcerange_mult=2.0):
#     m = env.physics.model
#     name2id = {m.id2name(i, 'actuator'): i for i in range(m.nu)}
#     chosen = (targets or list(name2id.keys()))
#     for nm in chosen:
#         if nm not in name2id: continue
#         aid = name2id[nm]
#         if m.actuator_gear.shape[1] >= 1:
#             m.actuator_gear[aid, 0] *= float(gear_mult)
#         lo, hi = m.actuator_forcerange[aid]
#         if lo < hi:
#             span = (hi - lo) * float(forcerange_mult)
#             mid = 0.5 * (hi + lo)
#             m.actuator_forcerange[aid, 0] = mid - 0.5 * span
#             m.actuator_forcerange[aid, 1] = mid + 0.5 * span


# def zoom_out(env, cam_name="side", back_factor=2.0, fovy_add_deg=15.0):
#     ph = env.physics
#     m = ph.model

#     cam_id = -1
#     for i in range(m.ncam):
#         nm = (m.id2name(i, 'camera') or "")
#         if nm == cam_name:
#             cam_id = i; break
#     if cam_id < 0:
#         print(f"[zoom] camera '{cam_name}' not found")
#         return cam_name

#     center = np.array(m.stat.center, dtype=float)
#     pos = m.cam_pos[cam_id].copy()
#     delta = pos - center
#     if np.linalg.norm(delta) > 1e-9 and back_factor != 1.0:
#         m.cam_pos[cam_id] = center + delta * float(back_factor)

#     m.cam_fovy[cam_id] = float(m.cam_fovy[cam_id]) + float(fovy_add_deg)
#     m.cam_fovy[cam_id] = np.clip(m.cam_fovy[cam_id], 10.0, 150.0)

#     ph.forward()
#     try:
#         ph._reset_contexts()
#     except AttributeError:
#         pass

#     print(f"[zoom] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.1f}°")
#     return cam_name

# def set_camera_zoom(env,
#                     cam_name: str = "side",
#                     *,
#                     distance_abs: float | None = None,   # absolute distance in model units
#                     distance_factor: float | None = 8.0, # OR factor×model.stat.extent
#                     fovy_deg: float | None = 70.0,       # absolute FOV in degrees
#                     along_current: bool = True):         # move along the camera's current direction
#     """
#     Set a named (fixed) camera to a *non-compounding*, zoomed-out view.
#     - If distance_abs is given, use it.
#     - Else if distance_factor is given, use distance = distance_factor * model.stat.extent.
#     - If fovy_deg is given, set absolute FOV (clamped to [10, 150] deg).
#     - 'along_current=True' keeps the camera pointing the same way, just farther.
#     """
#     ph = env.physics
#     m = ph.model

#     # find camera id
#     cam_id = -1
#     for i in range(m.ncam):
#         if (m.id2name(i, 'camera') or "") == cam_name:
#             cam_id = i; break
#     if cam_id < 0:
#         print(f"[camera] '{cam_name}' not found"); 
#         return cam_name  # fall back to name

#     center = np.array(m.stat.center, dtype=float)
#     pos    = m.cam_pos[cam_id].copy()

#     # pick direction to back away
#     v = pos - center
#     if (not along_current) or (np.linalg.norm(v) < 1e-9):
#         # stable default direction if current vector is degenerate
#         v = np.array([-1.0, 0.0, 1.0], dtype=float)
#     v /= (np.linalg.norm(v) + 1e-12)

#     # choose distance
#     if distance_abs is not None:
#         dist = float(distance_abs)
#     else:
#         if distance_factor is None:
#             distance_factor = 8.0   # sensible default
#         dist = float(m.stat.extent) * float(distance_factor)

#     # set camera position & FOV *absolutely* (no compounding)
#     m.cam_pos[cam_id] = center + v * dist
#     if fovy_deg is not None:
#         m.cam_fovy[cam_id] = float(np.clip(fovy_deg, 10.0, 150.0))

#     # apply and refresh dm_control render contexts
#     ph.forward()
#     try: ph._reset_contexts()
#     except AttributeError: pass

#     print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.2f}")
#     return cam_name


# def set_all_cameras_zoom(env,
#                          distance_factor: float = 8.0,
#                          fovy_deg: float = 70.0,
#                          skip_tracking: bool = True):
#     """
#     Apply the same zoom to *every* camera (useful if you switch cams).
#     If skip_tracking=True, we skip cameras whose names suggest tracking (e.g. 'track1/2/3').
#     """
#     ph = env.physics
#     m = ph.model
#     names = [(m.id2name(i, 'camera') or "") for i in range(m.ncam)]
#     for nm in names:
#         low = nm.lower()
#         if skip_tracking and ("track" in low):
#             continue
#         set_camera_zoom(env, nm, distance_factor=distance_factor, fovy_deg=fovy_deg)




# # -------------------- render using env.physics (exactly like the notebook loop) ---
# def render_env_with_controls(env, t_ms, act_controls, video_out,
#                              cam_name="side", back_factor=3.0, fovy_scale=0.8,
#                              slowmo=8.0, H=800, W=1280,
#                              boost_targets=("femur_T2_left","femur_T2_right"),
#                              gear_mult=1.0, forcerange_mult=10.0):
#     """
#     Feeds actuator controls into env.physics each step and renders frames via env.physics.render.
#     Slowmo>1 increases video duration (more frames per unit sim time).
#     """
#     ph = env.physics
#     m  = ph.model

#     # choose/modify camera on LIVE model (so it actually shows up)
#     cam_id = set_env_camera(env, cam_name=cam_name, back_factor=back_factor, fovy_scale=fovy_scale)

#     # boost actuator strength on the env model
#     boost_actuator_strength_on_env(env, targets=list(boost_targets) if boost_targets else None,
#                                    gear_mult=gear_mult, forcerange_mult=forcerange_mult)

#     # map actuator names → ids
#     name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
#     # sim timing
#     dt = float(ph.timestep())                     # seconds
#     dt_ms = 1000.0 * dt
#     T = int(math.ceil((t_ms[-1] - t_ms[0]) / dt_ms)) + 1
#     times_sim = t_ms[0] + np.arange(T) * dt_ms

#     # controls resampled to sim steps
#     U = np.zeros((m.nu, T), dtype=float)
#     for nm, series in act_controls.items():
#         aid = name2id.get(nm)
#         if aid is None: continue
#         U[aid, :] = np.interp(times_sim, t_ms, np.asarray(series, float))
#         lo, hi = m.actuator_ctrlrange[aid]
#         if lo < hi:
#             U[aid, :] = np.clip(U[aid, :], lo, hi)

#     # slow motion: write more frames per sim time
#     render_hz = 30
#     base_steps_per_frame = max(1, int(round((1.0 / render_hz) / dt)))
#     steps_per_frame = max(1, int(round(base_steps_per_frame / max(1.0, slowmo))))
#     out_fps = render_hz

#     # writer
#     video_out.parent.mkdir(parents=True, exist_ok=True)
#     writer = imageio.get_writer(str(video_out), fps=out_fps)

#     # reset and preview
#     try:
#         env.reset()
#     except Exception:
#         ph.reset()
#     # one preview frame
#     frame0 = ph.render(height=H, width=W, camera_id=cam_name)
#     imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

#     # simulate & render
#     frame_count = 0
#     for k in range(T):
#         ph.data.ctrl[:] = U[:, k]
#         ph.step()  # advance physics (like your notebook loop)
#         if k % steps_per_frame == 0:
#             frame = ph.render(height=H, width=W, camera_id=cam_name)
#             writer.append_data(frame)
#             frame_count += 1

#     writer.close()
#     est_secs = frame_count / out_fps
#     print(f"[video] wrote {video_out}  frames={frame_count}  fps={out_fps}  slowmo×{slowmo}  (~{est_secs:.1f}s)")

# # -------------------- run everything --------------------
# # 1) mapping + controls from your NEURON run folder
# mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
# t_ms, act_controls = build_actuator_controls(
#     RESULT_DIR, mapping_by_act,
#     tau_rise=1.0, tau_decay=6.0, scale=1.0,
#     from_voltages=True, v_thresh=0.0
# )

# # 2) stretch signals so you can see behavior
# t_long, act_controls_long = extend_signals(t_ms, act_controls, total_ms=120.0)

# # 3) build env the same way your notebook does (env.physics present)

# env = build_env_from_xml(WORLD_XML)

# # pick which camera to render with
# CAM = "track1"   # or "back", "hero", etc.

# # apply absolute zoom so it stays fixed & wide
# # Option A: set one camera
# set_camera_zoom(env, CAM, distance_factor=10.0, fovy_deg=75.0)

# # (optional) Option B: make *all* fixed cams wide in case you switch later
# # set_all_cameras_zoom(env, distance_factor=10.0, fovy_deg=75.0)

# # IMPORTANT: do not re-scale cameras again in the renderer.
# render_env_with_controls(
#     env, t_long, act_controls_long, VIDEO_OUT,
#     cam_name=CAM,
#     back_factor=1.0,      # keep 1.0 so we don't compound the zoom
#     fovy_scale=1.0,       # keep 1.0 so we don't compound the zoom
#     slowmo=8.0,
#     boost_targets=("tibia_T2_left","femur_T2_left","tibia_T2_right", "femur_T2_right"),
#     # boost_targets=("femur_T2_left","femur_T2_right"),
#     gear_mult=1, forcerange_mult=10.0
# )



In [ ]:
# ===== Inspect neuro_fly repo & suggest `pip install -e` target =====
from pathlib import Path
import os, sys, re, textwrap

# <-- SET THIS to your local clone path
NEURO_FLY_ROOT = Path("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/build_fruitfly")

# Optional: auto-run pip install -e on the best candidate
RUN_INSTALL = False

# ---------------- helpers ----------------
SKIP_DIRS = {".git", ".hg", ".svn", "__pycache__", ".ipynb_checkpoints", "build", "dist", "*.egg-info", ".mypy_cache", ".ruff_cache", ".pytest_cache"}

def should_skip_dir(name: str) -> bool:
    if name in SKIP_DIRS: return True
    for pat in SKIP_DIRS:
        if "*" in pat:
            rx = re.compile("^" + pat.replace(".", r"\.").replace("*", ".*") + "$")
            if rx.match(name): return True
    return False

def print_tree(root: Path, max_depth=6, max_entries=20000):
    root = root.resolve()
    print(f"\n[tree] Listing under: {root}")
    count = 0
    for dirpath, dirnames, filenames in os.walk(root):
        # prune
        dirnames[:] = [d for d in dirnames if not should_skip_dir(d)]
        depth = Path(dirpath).relative_to(root).parts
        if len(depth) > max_depth:
            continue
        # print dir
        rel = Path(dirpath).relative_to(root)
        indent = "  " * len(rel.parts)
        print(f"{indent}{rel if rel != Path('.') else Path('.')}"+"/")
        # print files
        for f in sorted(filenames):
            if f.endswith((".pyc", ".pyo")): 
                continue
            print(f"{indent}  {f}")
            count += 1
            if count >= max_entries:
                print("\n[tree] Truncated after", max_entries, "entries.")
                return

def find_candidates(root: Path):
    root = root.resolve()
    pyprojs = list(root.rglob("pyproject.toml"))
    setups   = list(root.rglob("setup.py"))
    setupcfg = list(root.rglob("setup.cfg"))
    # filter out hidden/skip dirs
    def ok(p: Path):
        parts = set(p.parts)
        return not any(should_skip_dir(x) for x in parts)
    pyprojs = [p for p in pyprojs if ok(p.parent)]
    setups  = [p for p in setups  if ok(p.parent)]
    setupcfg= [p for p in setupcfg if ok(p.parent)]
    return pyprojs, setups, setupcfg

def parse_project_name(pyproject_path: Path):
    try:
        txt = pyproject_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return None
    # very light-weight parse for [project] name = "..."
    m = re.search(r"^\s*\[project\][\s\S]*?^\s*name\s*=\s*['\"]([^'\"]+)['\"]", txt, flags=re.M)
    if m: return m.group(1)
    # poetry style
    m = re.search(r"^\s*\[tool\.poetry\][\s\S]*?^\s*name\s*=\s*['\"]([^'\"]+)['\"]", txt, flags=re.M)
    if m: return m.group(1)
    return None

def find_importable_packages(root: Path):
    """
    Look for top-level packages like 'neuro_adapter' (dir containing __init__.py),
    and src-layouts like root/src/neuro_adapter.
    """
    pkgs = []
    for p in root.rglob("__init__.py"):
        if any(should_skip_dir(x) for x in p.parts): 
            continue
        pkg_dir = p.parent
        # try to deduce import path (relative to root or root/src)
        try:
            rel = pkg_dir.relative_to(root)
            if rel.parts and rel.parts[0] == "src":
                rel = Path(*rel.parts[1:])
        except ValueError:
            # not under root directly
            continue
        if rel.parts and all(x not in SKIP_DIRS for x in rel.parts):
            pkgs.append((str(rel), pkg_dir))
    # Deduplicate by import name
    seen = set(); out=[]
    for name, path in pkgs:
        if name not in seen:
            seen.add(name); out.append((name, path))
    return sorted(out)

# ---------------- run ----------------
if not NEURO_FLY_ROOT.exists():
    raise FileNotFoundError(f"Path not found: {NEURO_FLY_ROOT}")

print_tree(NEURO_FLY_ROOT, max_depth=6)

pyprojs, setups, setupcfg = find_candidates(NEURO_FLY_ROOT)
print("\n[candidates] pyproject.toml:")
for p in pyprojs:
    nm = parse_project_name(p)
    print("  ", p, (" (project="+nm+")" if nm else ""))

print("[candidates] setup.py:")
for p in setups:
    print("  ", p)

print("[candidates] setup.cfg:")
for p in setupcfg:
    print("  ", p)

pkgs = find_importable_packages(NEURO_FLY_ROOT)
print("\n[packages] importable package dirs found (import_name → path):")
for name, path in pkgs:
    print(f"  {name:<24} -> {path}")

# Heuristic suggestion for `pip install -e`
suggest = None
if pyprojs:
    # prefer the nearest pyproject at repo root
    pyprojs_sorted = sorted(pyprojs, key=lambda p: len(p.parent.relative_to(NEURO_FLY_ROOT).parts))
    suggest = pyprojs_sorted[0].parent
elif setups:
    setups_sorted = sorted(setups, key=lambda p: len(p.parent.relative_to(NEURO_FLY_ROOT).parts))
    suggest = setups_sorted[0].paren_


In [ ]:
import mujoco
from pathlib import Path

def list_model_cameras(mjcf_path: Path):
    mjcf_path = Path(mjcf_path)
    m = mujoco.MjModel.from_xml_path(str(mjcf_path))
    print(f"Total cameras: {m.ncam}")
    for i in range(m.ncam):
        name = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_CAMERA, i)
        # Some parameters live in model fields:
        # pos/orient are in m.cam_pos / m.cam_quat; fovy in m.cam_fovy; mode in m.cam_mode
        pos = m.cam_pos[i]
        fovy = m.cam_fovy[i]
        mode = int(m.cam_mode[i])  # 0: fixed, 1: track, 2: trackcom, 3: targetbody, 4: targetbodycom (depends on MuJoCo version)
        print(f"[#{i:02d}] name='{name}'  mode={mode}  pos=({pos[0]:.3f},{pos[1]:.3f},{pos[2]:.3f})  fovy={fovy:.2f}")

# --- usage ---
list_model_cameras("/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/assets/floor.xml")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- root of ALL your hemilineage SWCs ----
SWC_ROOT = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc")

# ---- activation table path ----
ACT_TABLE = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A/activation_table.csv")

def find_swc_for_gid(root: Path, gid: int) -> Path | None:
    """Search recursively under `root` for an SWC whose name contains this gid.
    Prefer an exact '{gid}.swc' filename if present.
    """
    gid_str = str(gid)

    # 1) exact filename match anywhere
    exact = list(root.rglob(f"{gid_str}.swc"))
    if exact:
        return exact[0]

    # 2) any file containing the gid in its name
    candidates = list(root.rglob(f"*{gid_str}*.swc"))
    if candidates:
        candidates.sort(key=lambda p: (len(p.name), len(str(p))))
        return candidates[0]

    return None

def load_swc_points(swc_path: Path) -> np.ndarray:
    """Minimal SWC loader — returns Nx3 array of xyz points."""
    data = []
    with open(swc_path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            x, y, z = map(float, parts[2:5])
            data.append((x, y, z))
    return np.array(data) if data else np.zeros((0, 3))

def plot_hemi_morphology_from_activation(
    act_csv: str | Path = ACT_TABLE,
    swc_root: Path = SWC_ROOT,
    save_pdf: str | Path = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/hemi_09A_hemilineage_morphology.pdf",
):
    # ---- load activation table and select hemilineage neurons ----
    df = pd.read_csv(act_csv)
    gids = df.loc[df["in_hemilineage"] == True, "neuron_id"].unique()
    print(f"[INFO] Hemilineage neurons: {len(gids)}")

    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    # ---- plot morphology for each hemilineage neuron ----
    missing = 0
    for gid in gids:
        swc_file = find_swc_for_gid(swc_root, int(gid))
        if swc_file is None:
            missing += 1
            continue

        pts = load_swc_points(swc_file)
        if pts.size == 0:
            continue

        ax.plot(pts[:, 0], pts[:, 1], pts[:, 2],
                linewidth=0.6, alpha=0.6)

    if missing:
        print(f"[WARN] No SWC found for {missing} hemilineage neuron(s).")

    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")
    ax.set_zlabel("Z (µm)")
    ax.set_title("Hemilineage 09A morphology (all in_hemilineage == True)")

    # no legend, no AIS 'x' markers
    plt.tight_layout()
    plt.savefig(save_pdf, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[INFO] Saved PDF to:\n  {save_pdf}")


In [ ]:
plot_hemi_morphology_from_activation()


## =============Flywire API Attempt=============

In [ ]:
# !pip install caveclient
# !pip install seaborn

In [ ]:
# import caveclient
# import pandas as pd
# import seaborn as sns
# from matplotlib import pyplot as plt

# client = caveclient.CAVEclient()
# client.auth.setup_token(make_new=True)
# "621d6716b512da5af200f4d03fd371b5"
# my_token = "621d6716b512da5af200f4d03fd371b5"

In [ ]:
# --- FlyWire (CAVE) enrichment for MN folders + mapping CSV ------------------
# pip install --upgrade caveclient pandas

from pathlib import Path
import json, re, os, sys
import pandas as pd

# === CONFIG ===
MN_ROOT   = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN")
CSV_PATH  = Path("/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/mn_to_actuator_mapping.csv")
DATASET   = "flywire_fafb_production"     # CAVE project name
DRY_RUN   = False                         # if True: don't write labels.json / CSV
CACHE_MAP = MN_ROOT / "_flywire_root_map.json"  # persists { "<instance>/<id>": root_id }



from caveclient import CAVEclient

DATASET = "flywire_fafb_production"
MY_TOKEN = "621d6716b512da5af200f4d03fd371b5"  # <- your 32-char token

# 1) Register token with the global auth client (persists for future sessions)
_global = CAVEclient(server_address="https://global.daf-apis.com")
_global.auth.token = MY_TOKEN
# _global.auth.save_token(token=MY_TOKEN)

# 2) Now open the dataset client using that token
client = CAVEclient(DATASET, auth_token=MY_TOKEN)
print("[auth] CAVEclient ready for", DATASET)


# === AUTH / CLIENT ===
try:
    from caveclient import CAVEclient
    from caveclient.base import AuthException
except Exception as e:
    raise RuntimeError("caveclient is not installed. Run: pip install --upgrade caveclient") from e

def get_cave_client(datastack: str) -> "CAVEclient":
    """
    Returns an authenticated CAVEclient for the given datastack.
    If no token is set up, opens the Globus browser flow once.
    """
    try:
        # If you already have a cached token, this just works.
        return CAVEclient(datastack)
    except AuthException:
        # One-time browser login to cache a token under global server:
        print("[auth] No token found. Opening browser to get a new token (one-time)…")
        global_client = CAVEclient(server_address="https://global.daf-apis.com")
        global_client.auth.get_new_token()   # follow browser flow
        return CAVEclient(datastack)

client = get_cave_client(DATASET)

# === CACHE HELPERS ===
def _load_cache():
    try:
        return json.loads(CACHE_MAP.read_text())
    except Exception:
        return {}

def _save_cache(d):
    if not DRY_RUN:
        CACHE_MAP.write_text(json.dumps(d, indent=2))

# === TABLE / QUERY HELPERS ===
def _tables():
    try:
        return client.materialize.get_tables()
    except Exception:
        return []

def _peek_cols(table):
    try:
        df = client.materialize.query_table(table=table, limit=2)
        return list(df.columns)
    except Exception:
        return []

def _first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def find_root_by_instance(instance: str):
    """
    Given a neuPrint-like instance string (e.g., MNwm34_PDMNp_L),
    try to find a FlyWire root id by scanning materialized tables
    with label-ish text columns. Returns (root_id, table, matched_col)
    or (None, None, None) if not found.
    """
    tables = _tables()
    if not tables:
        return None, None, None

    # Prefer neuron/annotation/info/label tables to speed things up
    pref = [t for t in tables if re.search(r"(neuron|annot|label|info)", t, re.I)]
    order = pref + [t for t in tables if t not in pref]

    for t in order:
        cols = _peek_cols(t)
        if not cols:
            continue
        root_col = _first_existing(cols, ["pt_root_id","root_id","target_id","id"])
        if not root_col:
            continue
        text_cols = [c for c in ("instance","label","name","cell_type","type","tags","annotation") if c in cols]
        if not text_cols:
            continue

        # Case-insensitive exact first
        for tc in text_cols:
            try:
                df = client.materialize.query_table(table=t, filter_equal_dict={tc: instance})
                if df is not None and not df.empty:
                    return int(df.iloc[0][root_col]), t, tc
            except Exception:
                pass

        # LIKE fallback (contains/ILIKE behavior)
        for tc in text_cols:
            try:
                df = client.materialize.query_table(table=t, filter_like_dict={tc: instance})
                if df is not None and not df.empty:
                    ex = df[df[tc].astype(str).str.lower() == instance.lower()]
                    row = ex.iloc[0] if not ex.empty else df.iloc[0]
                    return int(row[root_col]), t, tc
            except Exception:
                pass

    return None, None, None

def get_labels_from_tables(root_id: int):
    """
    Collect label-ish fields for this root from all tables where it appears.
    Returns a sorted list of strings (labels).
    """
    labels = set()
    tables = _tables()
    for t in tables:
        cols = _peek_cols(t)
        if not cols:
            continue
        root_col = _first_existing(cols, ["pt_root_id","root_id","target_id","id"])
        if not root_col:
            continue
        try:
            df = client.materialize.query_table(table=t, filter_equal_dict={root_col: int(root_id)})
        except Exception:
            continue
        if df is None or df.empty:
            continue

        # pull any text-ish columns that look like labels
        for c in cols:
            cl = c.lower()
            if any(k in cl for k in ("label","name","instance","tag","type","cell_type","annotation","class","group","subclassabbr")):
                vals = df[c].dropna().astype(str).tolist()
                for v in vals:
                    v = v.strip()
                    if v and v != "nan":
                        labels.add(v)
    return sorted(labels)

def guess_soma_from_tables(root_id: int):
    """
    Try to find soma columns in any table for this root.
    Returns [x,y,z] (nm) or None.
    """
    tables = _tables()
    for t in tables:
        cols = _peek_cols(t)
        if not cols:
            continue
        root_col = _first_existing(cols, ["pt_root_id","root_id","target_id","id"])
        if not root_col:
            continue
        try:
            df = client.materialize.query_table(table=t, filter_equal_dict={root_col: int(root_id)})
        except Exception:
            continue
        if df is None or df.empty:
            continue

        cand_xyz = [
            ("soma_x","soma_y","soma_z"),
            ("x","y","z"),
            ("pt_x","pt_y","pt_z"),
        ]
        for xs,ys,zs in cand_xyz:
            if xs in df.columns and ys in df.columns and zs in df.columns:
                row = df.iloc[0]
                try:
                    return [float(row[xs]), float(row[ys]), float(row[zs])]
                except Exception:
                    pass
    return None

# === ROI/NERVOUS SYSTEM HEURISTICS → (group, thorax, side) ===================
NERVE_TO_GROUP = {
    "CvN":"head",
    "ADMN":"wing", "PDMN":"wing",
    "ProLN":"leg", "MesoLN":"leg", "MetaLN":"leg",
    "DProN":"leg", "VProN":"leg", "MesoAN":"wing", "ProAN":"leg",
    "DMetaN":"leg",
    "AbN1":"abdomen","AbN2":"abdomen","AbN3":"abdomen","AbN4":"abdomen","AbNT":"abdomen",
}

def derive_group_thorax_side(labels):
    """
    Inspect a set/list of label strings from FlyWire materialized tables
    and infer (group, thorax, side) to fix mis-tagging like MNwm34.
    """
    labset = set(map(str, labels))

    # Leg neuropil pattern like 'LegNp(T2)(L)'
    for L in labset:
        m = re.search(r"LegNp\(T(?P<T>\d)\)\((?P<S>L|R)\)", L)
        if m:
            thorax = f"T{m.group('T')}"
            side   = "left" if m.group('S') == "L" else "right"
            return "leg", thorax, side

    # Nerve tokens e.g. 'PDMN(L)', 'ProLN(R)', etc.
    for L in labset:
        m = re.search(r"(?P<N>AbN[1-4]|AbNT|CvN|ADMN|PDMN|ProLN|MesoLN|MetaLN|DProN|VProN|MesoAN|ProAN|DMetaN)\((?P<S>L|R)\)", L)
        if m:
            nerve = m.group('N')
            side  = "left" if m.group('S') == "L" else "right"
            group = NERVE_TO_GROUP.get(nerve, "")
            thorax = (
                "T2" if nerve in ("ADMN","PDMN","MesoAN","MesoLN")
                else "T1" if nerve in ("DProN","VProN","ProAN","ProLN")
                else "T3" if nerve in ("DMetaN","MetaLN")
                else "Ab" if nerve.startswith("AbN") or nerve == "AbNT"
                else ""
            )
            return group, thorax, side

    # Tectulum hints (UTct/LTct/WTct) → often wing territory; infer thorax & side if present
    for L in labset:
        if re.search(r"(UTct|LTct|WTct)", L):
            t = re.search(r"T(\d)", L)
            thorax = f"T{t.group(1)}" if t else "T2"
            s = re.search(r"\((L|R)\)", L)
            side = "left" if (s and s.group(1) == "L") else ("right" if (s and s.group(1) == "R") else "")
            return "wing", thorax, side

    if any("CvN" in L for L in labset):
        return "head", "", ""
    if any(L.startswith("AbN") or L.startswith("AbNT") for L in labset):
        return "abdomen", "Ab", ""
    return "", "", ""

# === FILE I/O HELPERS ========================================================
def write_labels_json(path: Path, payload: dict):
    if not DRY_RUN:
        (path / "labels.json").write_text(json.dumps(payload, indent=2))

def update_csv(csv_path: Path, updates: dict):
    """
    updates: { (instance, mn_id:int) : {"group":..., "thorax":..., "side":..., "actuator_name":(optional)} }
    """
    if not csv_path.exists():
        print(f"[warn] CSV missing: {csv_path}")
        return

    df = pd.read_csv(csv_path)
    # Ensure columns exist
    for c in ("mn_type","mn_id","group","thorax","side","actuator_name"):
        if c not in df.columns:
            df[c] = ""

    df["mn_type"] = df["mn_type"].astype(str)
    df["mn_id"]   = pd.to_numeric(df["mn_id"], errors="coerce").astype("Int64")

    filled = 0
    for i in range(len(df)):
        row = df.iloc[i]
        key = (row["mn_type"].strip(), int(row["mn_id"]) if pd.notna(row["mn_id"]) else None)
        u = updates.get(key)
        if not u:
            continue

        changed = False
        # Fill group/thorax/side if blank
        for col in ("group","thorax","side"):
            old_val = str(row.get(col, "")).strip()
            new_val = str(u.get(col, "")).strip()
            if old_val == "" and new_val != "":
                df.at[i, col] = new_val
                changed = True
        # Optionally propose actuator_name defaults if blank and group is non-leg
        act_old = str(row.get("actuator_name","")).strip()
        if act_old == "":
            g = str(df.at[i, "group"]).strip()
            s = str(df.at[i, "side"]).strip()
            if g == "wing":
                df.at[i, "actuator_name"] = "wing_pitch_left" if s == "left" else ("wing_pitch_right" if s == "right" else "")
                changed = True
            elif g == "head":
                df.at[i, "actuator_name"] = "head"; changed = True
            elif g == "abdomen":
                df.at[i, "actuator_name"] = "abdomen"; changed = True

        if changed:
            filled += 1

    if not DRY_RUN:
        # one-time backup
        bak = csv_path.with_suffix(".bak.csv")
        if not bak.exists():
            bak.write_text(csv_path.read_text())
        df.to_csv(csv_path, index=False)

    print(f"[csv] filled/updated {filled} row(s) in {csv_path.name}")

# === MAIN LOOP ===============================================================
cache = _load_cache()
updates = {}

type_dirs = [p for p in MN_ROOT.iterdir() if p.is_dir()]
for type_dir in sorted(type_dirs, key=lambda p: p.name):
    instance = type_dir.name  # folder is neuPrint "instance" (e.g., MNwm34_PDMNp_L)
    id_dirs = [q for q in type_dir.iterdir() if q.is_dir() and q.name.isdigit()]
    for id_dir in sorted(id_dirs, key=lambda q: int(q.name)):
        key = f"{instance}/{id_dir.name}"
        root_id = cache.get(key)

        if not root_id:
            rid, t, c = find_root_by_instance(instance)
            if not rid:
                print(f"[warn] no FlyWire root for {key}")
                continue
            cache[key] = int(rid)
            root_id = rid
            print(f"[root] {key} → {root_id} (via {t}.{c})")

        # Pull all label-ish fields & try to infer group/thorax/side from ROI-ish labels
        labels = get_labels_from_tables(int(root_id))
        soma   = guess_soma_from_tables(int(root_id))
        group, thorax, side = derive_group_thorax_side(labels)

        payload = {
            "flywire": {
                "root_id": int(root_id),
                "labels": labels,
                "soma_nm": soma,
                "derived": {"group": group, "thorax": thorax, "side": side}
            },
            "instance": instance,
            "mn_id": int(id_dir.name)
        }
        write_labels_json(id_dir, payload)
        updates[(instance, int(id_dir.name))] = {"group": group, "thorax": thorax, "side": side}

_save_cache(cache)
update_csv(CSV_PATH, updates)
print("Done.")


### ===================== OLD Smarter Auto-Mapping for MN → MuJoCo =====================

In [ ]:


# # build_smart_mapping_csv(MN_ROOT, client, out_csv="mn_to_actuator_mapping.csv")

# import os, re, csv, json, shutil
# from pathlib import Path
# import pandas as pd

# # ----------------------------- CONFIG / DICTS ----------------------------------

# # Exit-nerve → high-level subsystem + thoracic segment hint
# NERVE_TO_SUBSYSTEM = {
#     # LEG (T1/T2/T3)
#     "ProLN": ("leg",  "T1"),  # prothoracic leg nerve
#     "MesoLN":("leg",  "T2"),  # mesothoracic leg nerve
#     "MetaLN":("leg",  "T3"),  # metathoracic leg nerve

#     # WING/FLIGHT (T2)
#     "ADMN": ("wing", "T2"),   # anterior dorsal mesothoracic nerve
#     "PDMN": ("wing", "T2"),   # posterior dorsal mesothoracic nerve
#     "MesoAN": ("wing", "T2"), # mesothoracic accessory nerve

#     # HEAD/NECK / ANTENNA
#     "CvN":   ("head", ""),    # cervical nerve (head/neck/antenna)
#     "DProN": ("head", "T1"),
#     "VProN": ("head", "T1"),

#     # ABDOMEN
#     "AbN1":  ("abdomen", "Ab"), "AbN2": ("abdomen", "Ab"),
#     "AbN3":  ("abdomen", "Ab"), "AbN4": ("abdomen", "Ab"),
#     "AbNT":  ("abdomen", "Ab"),
# }

# # “Coarse” actuator suggestions for subsystems (you can refine these)
# # Keys are (subsystem, side) where side in {'L','R',''}.
# # For wings/head/abdomen we can give confident suggestions immediately.
# ACTUATOR_SUGGEST = {
#     ("wing", "L"): [{"joint":"wing","dof":"pitch","actuator_name":"wing_pitch_left",  "sign": ""}],
#     ("wing", "R"): [{"joint":"wing","dof":"pitch","actuator_name":"wing_pitch_right", "sign": ""}],
#     ("head", "L"): [
#         {"joint":"head","dof":"main","actuator_name":"head",         "sign": ""},
#         {"joint":"antenna","dof":"abduct","actuator_name":"antenna_abduct_left","sign": " +1"},
#     ],
#     ("head", "R"): [
#         {"joint":"head","dof":"main","actuator_name":"head",         "sign": ""},
#         {"joint":"antenna","dof":"abduct","actuator_name":"antenna_abduct_right","sign": " +1"},
#     ],
#     ("abdomen",""): [{"joint":"abdomen","dof":"main","actuator_name":"abdomen","sign": ""}],
# }

# # Optional: name-based hints to infer leg joint if the instance name contains these tokens
# LEG_NAME_HINTS = [
#     # (regex pattern, joint, dof, note)
#     (r'coxa',   "coxa",  "main",  "name hints coxa"),
#     (r'femur',  "femur", "main",  "name hints femur"),
#     (r'tibia',  "tibia", "main",  "name hints tibia"),
#     (r'tarsus', "tarsus","main",  "name hints tarsus"),
#     # Add real muscle family hints if you have them (e.g., "Depressor", "Levator", etc.)
# ]

# # Gains/bias defaults (controller-specific; adjust to taste)
# DEFAULT_GAIN = 1.0
# DEFAULT_BIAS = 0.0

# # --------------------------- NeuPrint helpers ----------------------------------

# def _pick_type_col(df: pd.DataFrame) -> str:
#     for c in ("type","systematicType","instance","name","cellType"):
#         if c in df.columns:
#             return c
#     return None

# def get_full_type_from_neuprint(body_id, client):
#     """
#     Returns full label string like 'MNwm34_PDMNp_L' if present.
#     Tries several columns in priority order.
#     """
#     from neuprint import NeuronCriteria, fetch_neurons
#     df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
#     if df is None or df.empty:
#         return None
#     for col in ("type","instance","systematicType","name","cellType"):
#         if col in df.columns:
#             val = df.iloc[0].get(col, None)
#             if pd.notna(val):
#                 val = str(val).strip()
#                 if val:
#                     return val
#     # fallback: return whatever column exists
#     tcol = _pick_type_col(df)
#     if tcol and pd.notna(df.iloc[0].get(tcol, None)):
#         return str(df.iloc[0][tcol]).strip()
#     return None

# # ----------------------------- Parsing helpers ---------------------------------

# def _strip_trailing_lowercase(s: str) -> str:
#     return re.sub(r'[a-z]+$', '', s)

# def parse_full_mn_name(full_type: str):
#     """
#     Expect shapes like: MNwm34_PDMNp_L  → tokens = [MNwm34, PDMNp, L]
#     Returns: dict(type_short, nerve_token, nerve_base, side)
#     """
#     toks = str(full_type).strip().split('_')
#     side = toks[-1] if toks and toks[-1] in ("L","R","ND") else ""
#     # nerve token is usually the middle (if side exists) or last token
#     if len(toks) >= 2:
#         nerve_token = toks[-2] if side else toks[-1]
#     else:
#         nerve_token = ""
#     nerve_base = _strip_trailing_lowercase(nerve_token) if nerve_token else ""
#     type_short = toks[0] if toks else full_type
#     return {
#         "type_short": type_short,
#         "nerve_token": nerve_token,
#         "nerve_base": nerve_base,
#         "side_token": side,                        # 'L'/'R'/'ND'/''
#         "side": {"L":"left","R":"right"}.get(side, "")
#     }

# # ----------------------------- Core auto-mapper --------------------------------

# def _smart_defaults_for_subsystem(subsystem: str, side_abbrev: str):
#     """
#     Returns a list of actuator row dicts or [] if none.
#     side_abbrev is 'L'/'R'/''.
#     """
#     key = (subsystem, side_abbrev)
#     if key in ACTUATOR_SUGGEST:
#         return ACTUATOR_SUGGEST[key]
#     # abdomen is not side-specific here
#     if subsystem == "abdomen":
#         return ACTUATOR_SUGGEST.get(("abdomen",""), [])
#     return []

# def _leg_guess_from_name(type_full: str):
#     for patt, joint, dof, note in LEG_NAME_HINTS:
#         if re.search(patt, type_full, flags=re.IGNORECASE):
#             return {"joint": joint, "dof": dof, "note": note}
#     return None

# def build_smart_mapping_csv(
#     mn_root,
#     client,
#     out_csv="mn_to_actuator_mapping.csv",
#     overwrite=False
# ):
#     """
#     Walks MN directory, fetches full names (if needed), parses nerve/side,
#     assigns subsystem & actuator suggestions (wings/head/abdomen), and
#     leaves legs blank unless name clearly hints a joint.
#     Writes CSV with confidence & notes.
#     """
#     mn_root = Path(mn_root).expanduser().resolve()

#     rows = []
#     for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         type_name = type_dir.name  # may already be full name
#         for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir() and p.name.isdigit()):
#             bid = int(id_dir.name)

#             # Confirm / fetch full type
#             full_type = type_name
#             if "MN" not in full_type:  # very loose guard; tighten if needed
#                 np_label = get_full_type_from_neuprint(bid, client)
#                 if np_label:
#                     full_type = np_label

#             info  = parse_full_mn_name(full_type)
#             nerve_base = info["nerve_base"]
#             side_abbr  = info["side_token"]
#             side       = info["side"]
#             subsystem, thorax = NERVE_TO_SUBSYSTEM.get(nerve_base, ("", ""))

#             # Decide auto-mapping strategy
#             suggestions = []
#             confidence  = "low"
#             note        = ""

#             if subsystem in ("wing","head","abdomen"):
#                 suggestions = _smart_defaults_for_subsystem(subsystem, side_abbr)
#                 if suggestions:
#                     confidence = "strong"
#                     note = f"auto by subsystem={subsystem}, nerve={nerve_base}, side={side_abbr or 'ND'}"
#                 else:
#                     confidence = "medium"
#                     note = f"subsystem {subsystem} known but no hardcoded actuator"

#             elif subsystem == "leg":
#                 # Always fill segment + side.
#                 # Try to guess joint from name; otherwise leave blank for review.
#                 joint_guess = _leg_guess_from_name(full_type)
#                 if joint_guess:
#                     suggestions = [{
#                         "joint": joint_guess["joint"], "dof": joint_guess["dof"],
#                         "actuator_name": "", "sign": ""
#                     }]
#                     confidence = "medium"
#                     note = f"leg auto (name hint: {joint_guess['note']}); choose actuator for {thorax} {side}"
#                 else:
#                     suggestions = [{
#                         "joint":"", "dof":"", "actuator_name":"", "sign":""
#                     }]
#                     confidence = "low"
#                     note = f"leg auto (nerve {nerve_base}); pick joint/DoF for {thorax} {side}"

#             else:
#                 # Unknown/rare nerve → leave blank but keep hints
#                 suggestions = [{
#                     "joint":"", "dof":"", "actuator_name":"", "sign":""
#                 }]
#                 confidence = "low"
#                 note = f"unmapped nerve {nerve_base or 'unknown'}; fill manually"

#             # Emit one or more rows per MN (if multiple suggestions)
#             for sug in suggestions:
#                 rows.append({
#                     "mn_type": full_type,
#                     "mn_id":   bid,
#                     "nerve":   nerve_base,
#                     "subsystem": subsystem,
#                     "thorax": thorax,
#                     "side":   side,          # 'left'/'right'/''
#                     "joint":  sug.get("joint",""),
#                     "dof":    sug.get("dof",""),
#                     "actuator_name": sug.get("actuator_name",""),
#                     "sign":   sug.get("sign",""),
#                     "action": "",            # optional: 'add','scale','bias' etc.
#                     "gain":   DEFAULT_GAIN,
#                     "bias":   DEFAULT_BIAS,
#                     "confidence": confidence,
#                     "notes":  note
#                 })

#     df = pd.DataFrame(rows, columns=[
#         "mn_type","mn_id","nerve","subsystem","thorax","side",
#         "joint","dof","actuator_name","sign","action","gain","bias",
#         "confidence","notes"
#     ])

#     out_csv = Path(out_csv)
#     if out_csv.exists() and not overwrite:
#         # Merge: keep existing rows, update rows that match (mn_type, mn_id)
#         old = pd.read_csv(out_csv)
#         key = ["mn_type","mn_id"]
#         merged = (
#             df.set_index(key)
#               .combine_first(old.set_index(key))   # prefer existing filled cells
#               .reset_index()
#               [df.columns]  # reorder
#         )
#         merged.to_csv(out_csv, index=False)
#         print(f"[smart-map] Updated {out_csv} (kept your existing edits). "
#               f"Total rows: {len(merged)}")
#     else:
#         df.to_csv(out_csv, index=False)
#         print(f"[smart-map] Wrote {len(df)} rows → {out_csv}")

#     # Quick summary
#     print("  breakdown by subsystem:\n",
#           df.groupby("subsystem")["mn_id"].count().to_string())

# # ------------------------------- How to run ------------------------------------
# MN_ROOT = "/Users/juanlopez2016/Desktop/Digifly/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# build_smart_mapping_csv(MN_ROOT, client, out_csv="mn_to_actuator_mapping.csv", overwrite=False)


### --- OLD Smart fill for MN → MuJoCo actuator mapping (runs in one cell) ---

In [ ]:

# import re, csv, os
# from pathlib import Path
# import pandas as pd
# from xml.etree import ElementTree as ET

# # ========= 1) NeuPrint-style MN name parsing =========
# # Map the nerve base token → coarse segment hint and group
# NERVE_BASES = {
#     "CvN":"head",
#     "DProN":"T1",  "VProN":"T1",  "ProAN":"T1",  "ProLN":"T1",
#     "ADMN":"T2",   "PDMN":"T2",   "MesoAN":"T2", "MesoLN":"T2",
#     "DMetaN":"T3", "MetaLN":"T3",
#     "AbN1":"Ab", "AbN2":"Ab", "AbN3":"Ab", "AbN4":"Ab", "AbNT":"Ab",
# }
# NERVE_GROUP = {
#     "ProLN":"leg", "MesoLN":"leg", "MetaLN":"leg",
#     "ADMN":"wing", "PDMN":"wing",
#     "CvN":"head", "MesoAN":"wing_accessory", "ProAN":"pro_accessory",
#     "DProN":"pro_dorsal", "VProN":"pro_ventral", "DMetaN":"meta_dorsal",
#     "AbN1":"abdomen", "AbN2":"abdomen", "AbN3":"abdomen", "AbN4":"abdomen", "AbNT":"abdomen",
# }

# def parse_full_mn_name(full_type: str):
#     """
#     Expected pattern like: MNwm34_PDMNp_L
#     Returns dict with: side ('L'/'R'/''), nerve_token, nerve_base, thorax ('T1/T2/T3' or ''), group, segment_hint
#     """
#     tokens = str(full_type).strip().split('_')
#     side = tokens[-1] if tokens and tokens[-1] in ("L","R","ND") else ""
#     nerve_token = tokens[-2] if (len(tokens) >= 2 and side) else (tokens[-1] if len(tokens) >= 2 else "")
#     nerve_base = re.sub(r'[a-z]+$', '', nerve_token) if nerve_token else ""
#     thorax = NERVE_BASES.get(nerve_base, "")
#     group  = NERVE_GROUP.get(nerve_base, "")
#     return {
#         "side": side,                                 # 'L'/'R'/''/'ND'
#         "nerve_token": nerve_token,                   # e.g. 'PDMNp'
#         "nerve_base": nerve_base,                     # e.g. 'PDMN'
#         "thorax": thorax if thorax in ("T1","T2","T3") else "",  # keep only thoracic segments
#         "segment_hint": thorax,                       # keeps 'Ab' for abdomen & '' for head
#         "group": group
#     }

# # ========= 2) Discover available actuators from MuJoCo XML (optional but recommended) =========
# def collect_actuators_from_xml(xml_path: str | None):
#     """
#     Returns set of actuator 'name' attributes from the MuJoCo model.
#     If xml_path is None or unreadable, returns empty set (we'll use name heuristics only).
#     """
#     avail = set()
#     if not xml_path: 
#         return avail
#     try:
#         xml_path = str(Path(xml_path).expanduser().resolve())
#         root = ET.parse(xml_path).getroot()
#         for act in root.findall(".//actuator/*"):  # general, motor, position, etc.
#             nm = act.get("name")
#             if nm:
#                 avail.add(nm)
#     except Exception as e:
#         print(f"[smart-fill] Could not parse XML ({xml_path}): {e}")
#     return avail

# # ========= 3) Helpers to propose actuator names =========
# def side_to_word(side_letter: str) -> str:
#     return {"L":"left", "R":"right"}.get(side_letter, "")

# def leg_priority_list(segment: str, side_word: str):
#     """
#     Preference order for legs (customize this to taste).
#     We’ll pick the first that exists in the model’s actuator set.
#     """
#     base = [
#         f"tibia_{segment}_{side_word}",
#         f"tarsus_{segment}_{side_word}",
#         f"tarsus2_{segment}_{side_word}",
#         f"femur_{segment}_{side_word}",
#         f"coxa_{segment}_{side_word}",
#         f"coxa_abduct_{segment}_{side_word}",
#         f"femur_twist_{segment}_{side_word}",
#         f"coxa_twist_{segment}_{side_word}",
#     ]
#     # Filter out names with double underscores if side_word is empty
#     return [nm for nm in base if "__" not in nm]

# def first_existing(candidates, available):
#     if not available:
#         return candidates[0] if candidates else ""
#     for c in candidates:
#         if c in available:
#             return c
#     return candidates[0] if candidates else ""

# def split_joint_dof_from_actuator_name(act_name: str):
#     """
#     Best-effort to fill joint/dof from the actuator name.
#     Example: 'tibia_T2_left' -> joint='tibia', dof='main'
#              'wing_pitch_left' -> joint='wing', dof='pitch'
#              'head_twist' -> joint='head', dof='twist'
#     """
#     if not act_name:
#         return "", ""
#     parts = act_name.split('_')
#     # Wing/head style: wing_pitch_left, head_abduct, head_twist, etc.
#     if parts[0] in ("wing", "head"):
#         joint = parts[0]
#         dof = parts[1] if len(parts) > 1 and parts[1] not in ("left","right","T1","T2","T3") else "main"
#         return joint, dof
#     # Leg style: joint_segment_side
#     joint = parts[0]
#     dof = "main"
#     if "abduct" in parts:
#         dof = "abduct"
#     elif "twist" in parts:
#         dof = "twist"
#     return joint, dof

# def propose_actuator_for_row(mn_type: str, available: set[str]):
#     """
#     Returns actuator_name (string) or "" if we can’t infer a safe default.
#     """
#     info = parse_full_mn_name(mn_type)
#     side_word = side_to_word(info["side"])
#     segment = info["thorax"]  # 'T1'/'T2'/'T3' or ''
#     group = info["group"]

#     # HEAD
#     if group == "head" or info["segment_hint"] == "head":
#         # Prefer 'head'; fall back to head_abduct/head_twist if needed
#         head_candidates = ["head", "head_abduct", "head_twist"]
#         return first_existing(head_candidates, available)

#     # WING
#     if group == "wing":
#         # Prefer pitch on that side; then roll, yaw; then any wing actuator on that side
#         if side_word:
#             wing_candidates = [
#                 f"wing_pitch_{side_word}",
#                 f"wing_roll_{side_word}",
#                 f"wing_yaw_{side_word}",
#             ]
#         else:
#             wing_candidates = ["wing_pitch_left", "wing_pitch_right"]
#         return first_existing(wing_candidates, available)

#     # ABDOMEN
#     if info["segment_hint"] == "Ab" or group == "abdomen":
#         return first_existing(["abdomen", "abdomen_abduct"], available)

#     # LEG
#     if group == "leg" and segment in ("T1","T2","T3") and side_word:
#         candidates = leg_priority_list(segment, side_word)
#         return first_existing(candidates, available)

#     # Accessory/ambiguous → no guess
#     return ""

# # ========= 4) Main: smart-fill your CSV =========
# def smart_fill_mapping_csv(
#     csv_path="mn_to_actuator_mapping.csv",
#     xml_path=None,
#     overwrite=False,
#     default_gain=1.0,
#     default_bias=0.0
# ):
#     """
#     - Loads CSV.
#     - For each row with empty actuator_name, proposes one from (mn_type, NeuPrint tokens).
#     - If overwrite=False (default), keeps any pre-filled actuator_name.
#     - Fills joint/dof from actuator name when possible.
#     - Writes CSV back in place and prints a summary.
#     """
#     csv_path = str(Path(csv_path).expanduser().resolve())
#     if not os.path.exists(csv_path):
#         raise FileNotFoundError(csv_path)

#     df = pd.read_csv(csv_path)
#     required_cols = {"mn_type","mn_id","actuator_name","joint","dof","gain","bias"}
#     missing = required_cols - set(df.columns)
#     if missing:
#         raise ValueError(f"CSV missing required columns: {missing}")

#     available = collect_actuators_from_xml(xml_path)
#     if available:
#         print(f"[smart-fill] Found {len(available)} actuators in model (via XML).")
#     else:
#         print("[smart-fill] No XML supplied or parse failed; using name heuristics only.")

#     n_rows = len(df)
#     n_pre_filled = (df["actuator_name"].astype(str).str.len() > 0).sum()
#     n_changed = 0
#     n_new = 0

#     for i, row in df.iterrows():
#         cur_act = str(row.get("actuator_name","")).strip()
#         if cur_act and not overwrite:
#             # keep user's choice
#             continue

#         proposal = propose_actuator_for_row(str(row["mn_type"]), available)
#         if not proposal:
#             # leave blank if we can’t infer with confidence
#             continue

#         # Set fields
#         if cur_act and overwrite and cur_act != proposal:
#             n_changed += 1
#         if not cur_act:
#             n_new += 1

#         df.at[i, "actuator_name"] = proposal
#         joint, dof = split_joint_dof_from_actuator_name(proposal)
#         if not str(row.get("joint","")).strip():
#             df.at[i, "joint"] = joint
#         if not str(row.get("dof","")).strip():
#             df.at[i, "dof"] = dof

#         # Keep existing gain/bias if present, else set defaults
#         try:
#             if pd.isna(row.get("gain")):
#                 df.at[i, "gain"] = float(default_gain)
#         except Exception:
#             df.at[i, "gain"] = float(default_gain)
#         try:
#             if pd.isna(row.get("bias")):
#                 df.at[i, "bias"] = float(default_bias)
#         except Exception:
#             df.at[i, "bias"] = float(default_bias)

#     df.to_csv(csv_path, index=False)
#     n_after = (df["actuator_name"].astype(str).str.len() > 0).sum()

#     print("\n[smart-fill] Summary")
#     print(f"  rows total        : {n_rows}")
#     print(f"  pre-filled kept   : {n_pre_filled - n_changed if not overwrite else 0}")
#     print(f"  new fills added   : {n_new}")
#     print(f"  changed (overwrite): {n_changed}")
#     print(f"  now filled        : {n_after}")
#     print(f"  still blank       : {n_rows - n_after}")

# # ========= 5) Run it =========
# # Set XML path if you want it to check actual actuator names in your model:
# XML_PATH = "/Users/juanlopez2016/Desktop/Digifly/flybody-main/flybody/fruitfly/build_fruitfly/fruitfly.xml"  # ← set to your fruitfly.xml, or None

# smart_fill_mapping_csv(
#     csv_path="mn_to_actuator_mapping.csv",
#     xml_path=XML_PATH,            # put None if you don’t want XML validation
#     overwrite=False,              # keep anything you already chose
#     default_gain=1.0,
#     default_bias=0.0
# )


### ==== neuPrint schema + columns inspector ====

In [ ]:

# import os, json, inspect
# import pandas as pd

# # If you already have a `client` object, we’ll use it.
# # Otherwise we’ll try to build one from your env vars.
# def _ensure_client():
#     try:
#         return client  # your existing one
#     except NameError:
#         pass
#     from neuprint import Client
#     url = os.environ.get("NEUPRINT_SERVER", "https://neuprint.janelia.org")
#     ds  = os.environ.get("NEUPRINT_DATASET")
#     tok = os.environ.get("NEUPRINT_TOKEN")
#     if not ds or not tok:
#         raise RuntimeError("Set NEUPRINT_DATASET and NEUPRINT_TOKEN env vars, or create `client` beforehand.")
#     return Client(url, dataset=ds, token=tok)

# cli = _ensure_client()

# # --- 1) Show what NeuronCriteria accepts (so you know valid query kwargs)
# from neuprint import NeuronCriteria
# print("\n=== NeuronCriteria help (valid query fields) ===")
# print(inspect.getdoc(NeuronCriteria) or "(no docstring)")
# print("\nSignature:", inspect.signature(NeuronCriteria))

# # --- 2) List the actual property keys present on :Neuron nodes in THIS dataset
# def list_neuron_property_keys(client):
#     cy = """
#     MATCH (n:Neuron)
#     WITH keys(n) AS ks
#     UNWIND ks AS k
#     RETURN DISTINCT k ORDER BY k
#     """
#     df = client.fetch_custom(cy, format="pandas")
#     return sorted(df.iloc[:,0].astype(str).tolist())

# props = list_neuron_property_keys(cli)
# print("\n=== Distinct :Neuron property keys in this dataset ===")
# for p in props:
#     print(" -", p)

# # --- 3) Peek at a small sample of neurons to see which columns are actually populated
# from neuprint import fetch_neurons
# sample_df, _ = fetch_neurons(NeuronCriteria(status="Traced"), client=cli)
# # Reduce to a small sample to keep output readable
# sample_df = sample_df.head(10).copy()
# print("\n=== Sample fetch_neurons(...) columns ===")
# print(list(sample_df.columns))

# # For each "interesting" column we often use, show non-null counts in the sample
# interesting = [
#     "type","instance","name","cellType","class","subclass","subclassabbr","superClass",
#     "side","neuromere","nerve","hemilineage",
#     "somaLocation","somaRadius","vfbId","group","status"
# ]
# present = [c for c in interesting if c in sample_df.columns]
# if present:
#     print("\nNon-null counts in sample:")
#     print(sample_df[present].notna().sum().sort_values(ascending=False))
# else:
#     print("\n(‘interesting’ fields not found in sample; rely on the property list above.)")

# # --- 4) Dump ALL properties for a specific bodyId so you can see the ground truth
# def dump_neuron_meta(body_id, client):
#     from neuprint import fetch_neurons, NeuronCriteria
#     df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
#     if df is None or df.empty:
#         print(f"[body {body_id}] no rows.")
#         return
#     row = df.iloc[0].to_dict()
#     print(f"\n=== Metadata for bodyId {body_id} ===")
#     # pretty print with stable ordering: standard fields first, then the rest
#     first_keys = [
#         "bodyId","type","instance","name","cellType","class","subclass","subclassabbr","superClass",
#         "side","neuromere","nerve","hemilineage","group",
#         "somaLocation","somaRadius","status","vfbId"
#     ]
#     ordered = {k: row[k] for k in first_keys if k in row}
#     for k in row:
#         if k not in ordered:
#             ordered[k] = row[k]
#     print(json.dumps(ordered, indent=2, default=str))

# # Example usage:
# # dump_neuron_meta(10068, cli)   # <-- change to a body you care about

# # --- 5) Utility to see which of several alias names actually exist in your dataset
# def which_exist(columns, client):
#     df, _ = fetch_neurons(NeuronCriteria(status="Traced"), client=client)
#     cols = set(df.columns)
#     return {c: (c in cols) for c in columns}

# print("\n=== Column availability check ===")
# checks = ["type","instance","name","cellType","class","subclass","subClass",
#           "superClass","side","neuromere","nerve","hemilineage","group",
#           "somaLocation","somaRadius","vfbId","vfblId"]
# print(which_exist(checks, cli))
